# Structural aliasing in Chronos-Bolt - Bayesian analysis (streaming, analysis only)

**This notebook reads the fits; it does not produce them.** Every posterior of the Deliverable 3
design is already sampled and checkpointed under `CKPT_DIR`, so nothing here calls `pm.sample()`.
If a checkpoint a cell needs is missing, that cell fails with its name rather than starting a
multi-hour fit.

## Why it replaces the previous version

The previous notebook exhausted a 12 GB runtime at the first checkpoint load, before any
comparison started. Two separate things caused it, and both are gone:

1. **`log_likelihood` was materialised.** One float per posterior draw, per chain, per
   observation. For Model A that is 4 chains x 2000 draws x ~1.4e5 observations x 8 bytes,
   about 9 GB. The "chunked" loader still built the whole array in RAM (`np.empty(shape,
   "float32")`) before chunking anything.
2. **The chunked LOO path never ran.** It depended on `arviz_stats.loo.loo_helper`, which does
   not exist on the resolved ArviZ version, so it silently fell back to
   `az.loo(idata, pointwise=True)` - whose internal `log_weights` is a *second* array the same
   size as `log_likelihood`, on top of the copy already resident.

PSIS-LOO is computed independently per observation, so `log_likelihood` never has to be resident
at all. Part 4 now reads it one observation-slice at a time straight off the checkpoint and keeps
only the three per-observation numbers LOO needs. The slice budget is fixed (`LOO_TARGET_BYTES`,
256 MiB), so peak memory no longer depends on how large the fit is.

Measured on a 0.86 GiB checkpoint, same `elpd_loo` to the last digit:

| path | peak RSS |
|---|---|
| `az.from_netcdf` + `az.loo` | 2.73 GiB |
| streaming | 0.61 GiB |

The monolithic peak is about 3.2x the file, which is what put Model A's ~9 GB checkpoint at
~29 GB and over the limit.

## What is unchanged

Every model, prior, threshold and decision rule is the one in `sections/four.tex` and its
appendix. The PSIS smoothing, the LOO comparison and the reliability and 2*dse gates are ArviZ's:
`_gpdfit`, `_gpinv` and `_psislw` are verbatim ports of `arviz.stats.stats` (ported rather than
imported because the two ArviZ major versions expose them differently), and the comparison table
is produced by `az.compare` itself on a reconstructed `ELPDData`. Both were checked against the
monolithic path on real checkpoints: `elpd_loo`, `se`, `p_loo`, every `elpd_i`, every `pareto_k`,
`rank`, `elpd_diff`, `dse` and the stacking `weight` all agree to 0.0.

Three further changes, all memory only:

* **Posterior-only loads.** `report`, `prob`, the plots and the convergence gate read `posterior`
  and `sample_stats`; `log_likelihood` is never opened for them.
* **Posterior-predictive checks replicate in draw-blocks.** The full replicated array has the
  same shape as `log_likelihood`; only the per-draw stratum means are accumulated, so every draw
  still contributes exactly once and the intervals are the same.
* **Per-fit LOO is checkpointed.** `elpd_i` and `pareto_k` are a couple of MB per fit, so a
  session that dies resumes at the fit it died on instead of restarting the group.

## Running it

Run top to bottom. Part 4 stages each checkpoint to local disk before streaming it, because the
slices are strided reads and Drive's FUSE mount is slow at those; set `STAGE_LOCALLY = False` if
local disk is short. Expect roughly ten minutes of PSIS per Model A variant and far less for the
others.


## 0.1 - Repository

Locate the checkout containing this notebook. A fresh hosted runtime clones the repository first,
so the next cell can install the exact environment recorded in the committed `uv.lock`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
MARKER = Path("chronos") / "bayesian" / "probe_lib.py"

def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target

REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))
print("repository:", REPO)
print("modules   :", BAYES_DIR)

## 0.2 - Locked environment

Install the environment exported from the committed `uv.lock`, with the Python-version markers in
that lock. Colab pre-imports several packages (numpy included) into the kernel process before any
user cell runs, so installing a different numpy/scipy afterwards can leave a stale compiled
extension resident in memory next to freshly written Python source files - the usual symptom is an
`ImportError` several frames deep inside `scipy` or `arviz` (e.g. `cannot import name '_slice' from
'numpy._core.umath'`), which is a numpy/scipy version-skew problem, not anything wrong with those
libraries themselves. The cell below installs the locked environment and then **restarts the Colab
runtime automatically** (at most twice) so the new versions load into a clean process, verifying the
fix in a throwaway subprocess each time before declaring success.

**When it restarts, Colab shows a red "session crashed" banner - that's expected, not a real
crash.** Just choose *Runtime > Run all* again; this cell detects it has already installed and
verified the environment and skips straight past the reinstall.

In [ ]:
import importlib.util, json, os, shutil, subprocess, sys, sysconfig, tempfile
from pathlib import Path

# Colab pre-loads several packages (including numpy) into the kernel process before any user
# cell runs. Installing a different numpy/scipy afterwards leaves that already-imported, already
# -compiled extension resident in memory while newer pure-Python files land on disk, so a later
# `import arviz` (or anything else that reaches scipy.special) can fail with something like
# "cannot import name '_slice' from 'numpy._core.umath'". This is not an ArviZ problem: any
# scipy-heavy import (PyMC included) can trip the same mismatch. The only reliable fix is a full
# process restart after installing, so this cell does that automatically, at most twice, and
# verifies the fix in a clean subprocess before giving up and asking for help.

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_env_restart_state.json"
MAX_RESTARTS = 2

def _on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False

def _uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")

def _load_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}

def _save_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")

def _clean_import_healthy() -> bool:
    """Import the numpy/scipy/arviz chain in a brand-new subprocess, so the check reflects what's
    actually on disk instead of whatever this (possibly already-poisoned) process has cached."""
    probe = subprocess.run(
        [sys.executable, "-c", "import numpy, scipy.special, scipy.stats, arviz"],
        capture_output=True, text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2000:])
    return probe.returncode == 0

def _restart_runtime(reason: str) -> None:
    print(f"\n{'=' * 78}\nRESTARTING THE COLAB RUNTIME NOW ({reason}).\n"
          "Colab will show a 'session crashed' message — that is expected, this is a deliberate,"
          " automatic restart, not a real crash.\n"
          "Once it reconnects, just choose Runtime > Run all again; this cell will detect that the"
          " environment is already installed and skip straight past the install.\n" + "=" * 78)
    sys.stdout.flush()
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)

state = _load_state()
UV = _uv_executable()

if state["restarts"] == 0:
    # First run in this process: install the locked environment.
    with tempfile.TemporaryDirectory() as temporary:
        requirements = Path(temporary) / "requirements.locked.txt"
        subprocess.check_call([
            UV, "export", "--frozen", "--no-dev", "--no-emit-project", "--no-hashes",
            "--output-file", str(requirements),
        ], cwd=REPO)
        subprocess.check_call([
            UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)
        ])
    # `idata.to_netcdf` / `az.from_netcdf` need a netCDF4-format backend (h5netcdf or
    # netCDF4) installed. uv.lock only pins h5netcdf for python_full_version < "3.12" - on
    # 3.12+ (Colab is on 3.13) the locked export omits it entirely, so every checkpointed
    # model fit would fail the first time save_idata() runs. This install is unconstrained
    # (no version pin), so on python<3.12 it is a harmless no-op against the already-locked
    # h5netcdf==1.8.1, and on 3.12+ it fills the real gap.
    subprocess.check_call([UV, "pip", "install", "--python", sys.executable, "h5netcdf", "h5py"])

    # torchvision is NOT a dependency of this project (absent from pyproject.toml and
    # uv.lock entirely) - it is only present because Colab pre-installs it, paired with
    # Colab's ORIGINAL torch build. Installing the locked torch==2.12.1 over that swaps
    # torch out but leaves the old torchvision behind, so its compiled custom ops (e.g.
    # "torchvision::nms") no longer match torch's ABI, and `import chronos` -> `import
    # transformers` crashes trying to load it (RuntimeError: operator torchvision::nms
    # does not exist). `transformers` skips that code path cleanly when torchvision is
    # genuinely absent, so - since nothing here does vision work - the fix is to remove
    # the now-mismatched package rather than chase a matching version.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
                   check=False, capture_output=True)

    if _on_colab():
        _save_state({"restarts": 1})
        _restart_runtime("locked environment just installed")
    else:
        print("Locked environment installed. If imports fail below, restart the kernel and "
              "resume at this cell (Colab does this automatically; outside Colab you'll need to "
              "do it by hand).")
elif state["restarts"] == 1:
    # We're in the fresh post-restart process. Verify the fix worked before trusting it.
    if _clean_import_healthy():
        print("Locked environment already installed and verified clean in this runtime — "
              "skipping reinstall.")
        _save_state({"restarts": "verified"})
    elif _on_colab():
        print("Still seeing a broken numpy/scipy/arviz chain after one restart — forcing a clean "
              "reinstall of numpy and scipy specifically, then restarting once more.")
        subprocess.check_call([
            UV, "pip", "install", "--python", sys.executable,
            "--reinstall-package", "numpy", "--reinstall-package", "scipy",
        ])
        _save_state({"restarts": 2})
        _restart_runtime("forced a clean numpy/scipy reinstall")
    else:
        print("Locked environment installed, but numpy/scipy/arviz still don't import cleanly. "
              "Restart the kernel by hand and re-run from the top.")
else:
    # Second restart already used (or we're re-running after a manual restart). Don't loop forever
    # — report status and let the next cell's imports speak for themselves.
    healthy = _clean_import_healthy()
    if healthy:
        print("Locked environment installed and verified clean after 2 restarts.")
        _save_state({"restarts": "verified"})
    else:
        print("numpy/scipy/arviz STILL don't import cleanly after 2 automatic restarts.\n"
              "This points at something more fundamental than a stale-process cache — most likely "
              "the locked numpy/scipy versions in uv.lock genuinely conflict with what this Colab "
              "image ships. Try: Runtime > Disconnect and delete runtime (a full fresh VM, not "
              "just a restart), then Runtime > Run all once from a clean slate. If it still fails, "
              "the uv.lock pins for numpy/scipy need to be loosened to versions Colab's own image "
              "is compatible with.")

def _real_module(name: str) -> bool:
    try:
        spec = importlib.util.find_spec(name)
    except (ImportError, ValueError):
        return False
    return spec is not None and spec.origin is not None

HAVE_TORCH = _real_module("torch") and _real_module("chronos")
print({"locked environment installed": True, "Part 2 available": HAVE_TORCH})

## 0.3 - Imports, seed and immutable run namespace

The run namespace includes the Deliverable 3 design version. Every artifact is recorded in an
analysis manifest with its SHA-256; a file without a matching entry is not a checkpoint.

In [ ]:
from __future__ import annotations

import json, random
from importlib import metadata as importlib_metadata

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import arviz as az
import pymc as pm
from scipy import stats

# uv.lock resolves arviz==0.23.4 for python_full_version < '3.12' and arviz==1.1.0 for >=
# '3.12' (Colab's 3.13 gets 1.1.0). arviz 1.x's compare() dropped the `ic` parameter (LOO is
# now the only supported IC), but chronos/bayesian/bayesian_checks.py (in the cloned repo, not
# this notebook) still calls `az_module.compare(loo_objects, ic="loo")`. Patching az.compare in
# this one place - rather than editing that shared helper - keeps every call site in this
# notebook working unmodified against either arviz major version.
import inspect as _inspect
if "ic" not in _inspect.signature(az.compare).parameters:
    _az_compare_impl = az.compare
    def _az_compare_ic_compat(compare_dict, ic=None, **kwargs):
        return _az_compare_impl(compare_dict, **kwargs)
    az.compare = _az_compare_ic_compat
    print("arviz.compare(): patched for the ic= -> (removed) API change in arviz", az.__version__)

# arviz>=1.0 also reworked plot_forest(): the `hdi_prob` keyword was renamed (it moved into
# az.hdi() as `prob`) and the return type changed from an array of matplotlib Axes to a
# PlotCollection-based object, so `axes[0].axvline(...)` style code downstream would break too.
# Rather than branch every call site on the arviz version, this notebook builds its one forest
# plot (Part 4.1) directly on az.hdi() - a stats function whose contract is far more stable than
# the plotting layer - and returns a plain `[Axes]` so existing `axes[0].<...>` calls keep working.
_HDI_PARAMS = set(_inspect.signature(az.hdi).parameters)

def plot_forest_compat(idata, var_name: str, prob: float = 0.95, figsize=(8, 3.4)):
    """Per-level credible-interval forest plot for a chain/draw + one-other-dim variable."""
    post = idata.posterior[var_name]
    other_dims = [d for d in post.dims if d not in ("chain", "draw")]
    dim = other_dims[0] if other_dims else None

    hdi_kw = {"prob": prob} if "prob" in _HDI_PARAMS else {"hdi_prob": prob}
    hdi_da = az.hdi(idata, var_names=[var_name], **hdi_kw)[var_name]
    bound_dim = "hdi" if "hdi" in hdi_da.dims else "ci_bound"

    if dim is not None:
        levels = [str(v) for v in hdi_da[dim].values]
        bounds = hdi_da.transpose(dim, bound_dim).values
        point = post.mean(dim=("chain", "draw")).sel({dim: hdi_da[dim].values}).values
    else:
        levels = [var_name]
        bounds = hdi_da.transpose(bound_dim).values[None, :]
        point = np.atleast_1d(post.mean(dim=("chain", "draw")).values)

    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(levels))
    ax.hlines(y, bounds[:, 0], bounds[:, 1], color="steelblue", lw=2.2)
    ax.plot(point, y, "o", color="steelblue", ms=5, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(levels)
    ax.invert_yaxis()
    ax.set_xlabel(var_name)
    return [ax]

import checkpointing as cp
import bayesian_checks as bc
import model_loader as ml

PLOT_STYLE = next((name for name in ("arviz-whitegrid", "seaborn-v0_8-whitegrid")
                   if name in plt.style.available), "default")
plt.style.use(PLOT_STYLE)
print("plot style  :", PLOT_STYLE)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

RUN_MODE = os.environ.get("PATCHALIASING_RUN_MODE", "FULL").strip().upper()
if RUN_MODE not in {"FULL", "SMOKE"}:
    raise ValueError("PATCHALIASING_RUN_MODE must be FULL or SMOKE")
IS_SMOKE = RUN_MODE == "SMOKE"

RESUME = True
DRAWS, TUNE, CHAINS = ((100, 100, 2) if IS_SMOKE else (2000, 2000, 4))
TARGET_ACCEPT = 0.9
NUTS_BACKEND = "pymc"       # "nutpie" is allowed; it targets the same posterior

if NUTS_BACKEND != "pymc":
    if importlib.util.find_spec(NUTS_BACKEND) is None:
        raise ImportError(f"{NUTS_BACKEND} is absent from the locked environment")
print("NUTS backend:", NUTS_BACKEND)

USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/patchAliasing"
RUN_ID = "d3_loc_smoke_v1" if IS_SMOKE else "d3_localisation_k3_v1"

def _run_root() -> Path:
    try:
        on_colab = importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        on_colab = False
    if not on_colab or not USE_DRIVE:
        root = BAYES_DIR / "_run"
        root.mkdir(parents=True, exist_ok=True)
        return root
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    root = Path(DRIVE_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    return root

_root = _run_root()
CKPT_DIR = _root / "full" / RUN_ID
DATA_DIR = CKPT_DIR / "data"
FIG_DIR = CKPT_DIR / "figures"
for directory in (CKPT_DIR, DATA_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def _version(package: str):
    try:
        return importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        return None

# Provenance of the analysis logic itself. This used to read bayesian_analysis.ipynb, which no
# longer exists: the collection, the fits and the analysis are now one notebook. It reads the
# repository's copy, not the executing file, so what it records is the state of the source the
# clone provides; a copy edited elsewhere and run from there is not fingerprinted by it.
NOTEBOOK_NAME = "localisation_analysis.ipynb"
_nb_path = BAYES_DIR / NOTEBOOK_NAME
if _nb_path.is_file():
    notebook_logic = [
        (item["cell_type"], "".join(item.get("source", "")))
        for item in json.loads(_nb_path.read_text(encoding="utf-8"))["cells"]
    ]
else:
    print(f"note: {NOTEBOOK_NAME} is not in {BAYES_DIR}; the notebook-logic fingerprint records "
          f"its absence rather than a hash.")
    notebook_logic = [("absent", NOTEBOOK_NAME)]
source_sha256 = {
    name: cp.sha256_file(BAYES_DIR / name)
    for name in ("collect.py", "probe_lib.py", "model_loader.py", "checkpointing.py",
                 "bayesian_checks.py")
}
source_sha256[f"{NOTEBOOK_NAME}:logic"] = cp.fingerprint(notebook_logic)

# Session override for the injected tone ratio. probe_lib fixes it at the deliverable's value of
# 1.0 and is not edited here, so a clone from the repository always runs that; setting
# PATCHALIASING_TONE_SNR rebinds it for this session only, which is how the previous convention of
# 4.0 can be run alongside without touching the file another person would get from git.
#
# It is applied here, before the manifest is built, and recorded in ANALYSIS_SPEC. That matters:
# the override leaves probe_lib's file hash untouched, so source_sha256 cannot see it, and two runs
# at different ratios sharing a RUN_ID would otherwise merge tables that are not comparable. With
# the ratio in the spec, reusing a RUN_ID across ratios fails the fingerprint check instead.
import probe_lib as _pl_spec

# The localisation estimator has two knobs and both change what h MEANS, so both are recorded in
# the manifest beside the tone ratio. Part 2 fixes them by measurement; these are its answer.
FHAT_TOPK = int(os.environ.get("PATCHALIASING_FHAT_TOPK", _pl_spec.FHAT_TOPK))
FHAT_TOL_HZ = float(os.environ.get("PATCHALIASING_FHAT_TOL", _pl_spec.FHAT_TOL_HZ))
_pl_spec.FHAT_TOPK, _pl_spec.FHAT_TOL_HZ = FHAT_TOPK, FHAT_TOL_HZ
print(f"localisation : top-{FHAT_TOPK} peaks, tolerance {FHAT_TOL_HZ} Hz")

TONE_SNR_DEFAULT = float(_pl_spec.TONE_SNR)
TONE_SNR = float(os.environ.get("PATCHALIASING_TONE_SNR", TONE_SNR_DEFAULT))
if TONE_SNR <= 0:
    raise ValueError("PATCHALIASING_TONE_SNR must be positive")
_pl_spec.TONE_SNR = TONE_SNR          # probe_lib.build_context reads this at call time
if TONE_SNR != TONE_SNR_DEFAULT:
    print(f"tone SNR    : {TONE_SNR} for this session, overriding the deliverable's "
          f"{TONE_SNR_DEFAULT}; recorded in the analysis manifest")
else:
    print(f"tone SNR    : {TONE_SNR}")

ANALYSIS_SPEC = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "tone_snr": TONE_SNR,
    "fhat_topk": FHAT_TOPK,
    "fhat_tol_hz": FHAT_TOL_HZ,
    "deliverable": "coursework/deliverable3",
    "seed": SEED,
    "sampling": {"draws": DRAWS, "tune": TUNE, "chains": CHAINS,
                 "target_accept": TARGET_ACCEPT, "backend": NUTS_BACKEND},
    "checkpoint_repo": ml.SWEEP_REPO,
    "checkpoint_revision": ml.SWEEP_REVISION,
    "source_sha256": source_sha256,
    "uv_lock_sha256": cp.sha256_file(REPO / "uv.lock"),
    "packages": {name: _version(name) for name in
                 ("pymc", "arviz", "pyarrow", "torch", "chronos-forecasting")},
}
ANALYSIS_FINGERPRINT = cp.fingerprint(ANALYSIS_SPEC)
ANALYSIS_MANIFEST_PATH = CKPT_DIR / "analysis_manifest.json"

if ANALYSIS_MANIFEST_PATH.is_file():
    ANALYSIS_MANIFEST = json.loads(ANALYSIS_MANIFEST_PATH.read_text(encoding="utf-8"))
    if ANALYSIS_MANIFEST.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        # Diagnose what actually moved before deciding whether this is safe to auto-resolve.
        # source_sha256 is compared key-by-key (separately from every other top-level field)
        # because "bayesian_analysis.ipynb:logic" - this notebook's own cell content - is the
        # one field expected to change across ordinary Part 4 debugging edits; nothing else
        # (packages, sampling, checkpoint_repo/revision, uv_lock_sha256, or the hash of any of
        # the 5 external .py files) should ever move without a deliberate, reviewed change.
        on_disk_spec = ANALYSIS_MANIFEST.get("analysis_spec", {})
        diff_keys = sorted(k for k in set(on_disk_spec) | set(ANALYSIS_SPEC)
                            if k != "source_sha256" and on_disk_spec.get(k) != ANALYSIS_SPEC.get(k))
        on_disk_sha = on_disk_spec.get("source_sha256", {})
        this_sha = ANALYSIS_SPEC["source_sha256"]
        sha_diff_keys = sorted(k for k in set(on_disk_sha) | set(this_sha)
                               if on_disk_sha.get(k) != this_sha.get(k))

        if not diff_keys and sha_diff_keys == [f"{NOTEBOOK_NAME}:logic"]:
            # Safe, narrow auto-heal: the ONLY thing that changed is this notebook's own cell
            # content - nothing that could affect a computed result (no package, no sampling
            # setting, no external source file moved). Update the on-disk manifest's
            # fingerprint/spec in place, keeping every already-recorded artifact, instead of
            # hard-failing and forcing a RUN_ID bump that would orphan every checkpoint already
            # computed under this RUN_ID.
            print("  notebook logic changed since this RUN_ID's manifest was last written "
                  f"(source_sha256['{NOTEBOOK_NAME}:logic'] differs) - nothing under "
                  "packages / sampling / checkpoint_repo / checkpoint_revision / uv_lock_sha256 "
                  "/ the 5 external source files moved, so this is auto-resolved in place. "
                  "Existing checkpoints are unaffected.")
            ANALYSIS_MANIFEST["analysis_fingerprint"] = ANALYSIS_FINGERPRINT
            ANALYSIS_MANIFEST["analysis_spec"] = ANALYSIS_SPEC
            cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)
        else:
            print("analysis manifest mismatch - fields that differ from the on-disk manifest:")
            for k in diff_keys:
                print(f"    {k}: on-disk={on_disk_spec.get(k)!r}  this-session={ANALYSIS_SPEC.get(k)!r}")
            for k in sha_diff_keys:
                print(f"    source_sha256[{k}]: on-disk={on_disk_sha.get(k)!r}  "
                      f"this-session={this_sha.get(k)!r}")
            raise ValueError("analysis manifest mismatch; increment RUN_ID instead of mixing artifacts")
else:
    ANALYSIS_MANIFEST = {
        "schema_version": 1,
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "analysis_spec": ANALYSIS_SPEC,
        "artifacts": {},
    }
    cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

def banner(part: str) -> None:
    print("=" * 78)
    print(part)
    print("=" * 78)

def ckpt(name: str) -> Path:
    return CKPT_DIR / name

def _record(name: str) -> None:
    """Record one checkpoint in the manifest, merging with whatever is on disk right now.

    Multiple Colab sessions can run different Part 4 models in parallel against the same Drive
    CKPT_DIR (B, C, D1 do not depend on A or on each other - only on Part 0-3, which is already
    cached). Each session keeps its own in-memory ANALYSIS_MANIFEST from whenever ITS cell 6 ran,
    so a naive "overwrite the file with my snapshot" (the original version of this function) would
    let one session's save silently erase another session's already-recorded artifacts - the file
    stays on disk, but have() sees a file with no manifest entry and raises. Re-reading the current
    file immediately before merging in this one entry closes that race for any two saves that
    aren't in the exact same instant, which is the realistic case here (saves are minutes to hours
    apart). This changes only how the manifest file is written; it does not change what gets
    computed, saved, or any analysis conclusion.
    """
    path = ckpt(name)
    entry = {"sha256": cp.sha256_file(path), "bytes": path.stat().st_size}
    if ANALYSIS_MANIFEST_PATH.is_file():
        on_disk = json.loads(ANALYSIS_MANIFEST_PATH.read_text(encoding="utf-8"))
        if on_disk.get("analysis_fingerprint") == ANALYSIS_FINGERPRINT:
            ANALYSIS_MANIFEST["artifacts"] = {**on_disk.get("artifacts", {}),
                                              **ANALYSIS_MANIFEST["artifacts"]}
    ANALYSIS_MANIFEST["artifacts"][name] = entry
    cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

def have(name: str) -> bool:
    if not RESUME:
        return False
    path = ckpt(name)
    entry = ANALYSIS_MANIFEST["artifacts"].get(name)
    if path.exists() != (entry is not None):
        raise ValueError(f"untracked or missing analysis artifact: {name}")
    if not path.exists():
        return False
    if cp.sha256_file(path) != entry["sha256"]:
        raise ValueError(f"analysis artifact hash mismatch: {name}")
    return True

def save_idata(idata, name: str):
    cp.atomic_netcdf(ckpt(name), idata)
    _record(name)
    print("  checkpoint ->", name)
    return idata

def load_idata(name: str):
    if not have(name):
        raise FileNotFoundError(name)
    print("  checkpoint <-", name)
    return az.from_netcdf(ckpt(name))

def save_df(frame: pd.DataFrame, name: str):
    cp.atomic_parquet(ckpt(name), frame)
    _record(name)
    print("  checkpoint ->", name)
    return frame

def load_df(name: str) -> pd.DataFrame:
    if not have(name):
        raise FileNotFoundError(name)
    print("  checkpoint <-", name)
    return pd.read_parquet(ckpt(name))

def save_json(value, name: str):
    cp.atomic_json(ckpt(name), value)
    _record(name)
    print("  checkpoint ->", name)
    return value

def load_json(name: str):
    if not have(name):
        raise FileNotFoundError(name)
    print("  checkpoint <-", name)
    return json.loads(ckpt(name).read_text(encoding="utf-8"))

def cached_fit(name: str, build_and_sample):
    return load_idata(name) if have(name) else save_idata(build_and_sample(), name)

print("run mode   :", RUN_MODE, "(NON-REPORTABLE)" if IS_SMOKE else "(REPORTABLE DESIGN)")
print("checkpoints:", CKPT_DIR)
print("tables     :", DATA_DIR)
print("figures    :", FIG_DIR)
print(f"draws={DRAWS} tune={TUNE} chains={CHAINS} target_accept={TARGET_ACCEPT}")

In [ ]:
# =====================================================================================
#  have() without re-hashing multi-gigabyte checkpoints on every call
# =====================================================================================
# have() verifies a checkpoint's sha256 against the manifest. That is the right check, but
# it reads the whole file, and a Model A checkpoint is ~9 GB on a Drive mount: the analysis
# below calls have() several times per fit, so the verification alone costs more I/O than
# the analysis. The digest is therefore cached on (size, mtime_ns): the first call still
# hashes the file in full and still fails on a mismatch, and any rewrite changes both size
# and mtime, so a stale entry cannot survive a modified file. Nothing about which artifacts
# are accepted changes - only how often an unchanged file is re-read to prove it.
_HASH_CACHE_PATH = CKPT_DIR / "_hash_cache.json"
try:
    _HASH_CACHE = json.loads(_HASH_CACHE_PATH.read_text(encoding="utf-8"))
except Exception:
    _HASH_CACHE = {}

def _cached_sha256(path):
    stat = path.stat()
    key = str(path)
    entry = _HASH_CACHE.get(key)
    if entry and entry.get("size") == stat.st_size and entry.get("mtime_ns") == stat.st_mtime_ns:
        return entry["sha256"]
    digest = cp.sha256_file(path)
    _HASH_CACHE[key] = {"size": stat.st_size, "mtime_ns": stat.st_mtime_ns, "sha256": digest}
    try:
        _HASH_CACHE_PATH.write_text(json.dumps(_HASH_CACHE), encoding="utf-8")
    except Exception:
        pass
    return digest

def have(name: str) -> bool:
    if not RESUME:
        return False
    path = ckpt(name)
    entry = ANALYSIS_MANIFEST["artifacts"].get(name)
    if path.exists() != (entry is not None):
        raise ValueError(f"untracked or missing analysis artifact: {name}")
    if not path.exists():
        return False
    if _cached_sha256(path) != entry["sha256"]:
        raise ValueError(f"analysis artifact hash mismatch: {name}")
    return True

print("have(): digest cache active at", _HASH_CACHE_PATH.name)


## 0.3b - Model selection

Controls which parts of this run actually execute. Recovery (Part 3.3) always runs regardless
of these flags - it's cheap once cached, and Part 4 assumes `RECOVERY_OK` is already
established either way.

Use `FIT_MODEL_*` to split Part 4's expensive fits across parallel Colab sessions that share the
same Drive checkpoint folder: set only the flags for the models THIS session should fit, run it,
and start a separate session (same Drive account, same `RUN_ID`) for the rest. A model already
completed in *any* session always reloads instantly from its checkpoint instead of refitting,
regardless of these flags - checkpointing is unconditional, these flags only decide whether a
model's section is attempted at all in this particular run.

`RUN_PART_5` gates Part 5 (PPC, LOO summaries, verdicts), which needs every model available in
this session. Leave it `False` in a partial/parallel session; set it `True` only for the final
consolidation run, once every model has been fitted somewhere and every `FIT_MODEL_*` flag below
is `True`.

In [ ]:

FIT_MODEL_A = True
FIT_MODEL_B = False
FIT_MODEL_C = False
FIT_MODEL_D1 = False
FIT_MODEL_D2 = False

# Which scope carries the reported estimand, fixed here rather than after the comparison.
# "unconditioned" keeps every arm and reads gamma as a statement about the design as collected;
# "conditioned" keeps arms where the tone is measurable at all and reads gamma as a statement
# about the model given a measurable target. See the scope cell in Part 4.1.
A_PRIMARY_SCOPE = "unconditioned"

RUN_PART_5 = True

# Every posterior of the main design is already sampled and checkpointed, so the only fits this
# notebook can still need are the prior-sensitivity ladder (05_sensitivity_0..3.nc), which is
# absent from CKPT_DIR. ALLOW_SAMPLING gates them:
#   True  - fit the ladder variants that are missing, then run Part 5 on everything
#   False - skip them; the sensitivity gate then fails and A/M1 are NOT REPORTABLE, while
#           every other claim is decided normally
# Nothing else is ever sampled: the guard installed after the model factories refuses any
# other call, so a missing checkpoint fails with its name instead of starting a fit in a
# runtime sized for reading.
ALLOW_SAMPLING = True

# Which ladder variants this session fits, by index into the variant list built in Part 5.3
# (0, 1, 2 are the StudentT prior scales of PRIOR_LADDER; 3 is the Normal-likelihood check).
# None means all of them. The variants are independent and the manifest merges concurrent
# writes, so four sessions can take one index each and finish in the time of the slowest.
SENSITIVITY_SUBSET = None

print("model selection:",
      ", ".join(name for name, flag in (
          ("A", FIT_MODEL_A), ("B", FIT_MODEL_B), ("C", FIT_MODEL_C),
          ("D1", FIT_MODEL_D1), ("D2", FIT_MODEL_D2),
      ) if flag) or "(none)")
print("Part 5           :", "will run" if RUN_PART_5 else "skipped in this session")
print("sampling         :", "sensitivity ladder only" if ALLOW_SAMPLING else "disabled")
print("ladder subset    :", "all variants" if SENSITIVITY_SUBSET is None else SENSITIVITY_SUBSET)


## 0.4 - Frozen Deliverable 3 design

`probe_lib.DELIVERABLE3_MODELS` is the immutable 15-configuration registry. The environment
variable `PATCHALIASING_MODELS` may choose a collection session subset, but cannot alter the
planned design, union grid or final coverage gate.

In [ ]:
import probe_lib as pl

FULL_MODELS = list(pl.DELIVERABLE3_MODELS)
SESSION_MODELS = list(pl.SESSION_MODELS)
assert len(FULL_MODELS) == 15
assert all(pl.CTX % S == 0 for _, S in FULL_MODELS)

print(f"fs={pl.FS} Hz  context={pl.CTX}  horizon={pl.PRED}  band={pl.BAND} Hz")
print(f"generators={pl.GENERATORS}  full models={len(FULL_MODELS)}  "
      f"session models={len(SESSION_MODELS)}")

rows = []
for P, S in FULL_MODELS:
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    usable = [f for f, delta in offsets.items() if np.isfinite(delta)]
    used_delta = [offsets[f] for f in usable]
    rows.append({
        "model": pl.model_tag(P, S), "P": P, "S": S, "overlap": (P - S) / P,
        "context_closes": pl.CTX % S == 0,
        "n_lock": len(offsets), "n_usable": len(usable),
        "stride_only": sum(pl.lock_family(f, P, S) == "stride" for f in offsets),
        "delta_min_hz": min(used_delta), "delta_max_hz": max(used_delta),
    })
design = pd.DataFrame(rows)
display(design)

gaps = pl.design_gaps(SESSION_MODELS)
if gaps:
    print("\nSession limitations (not empirical null results):")
    for gap in gaps:
        print(" -", gap)
print("\nS in {4, 5, 15, 28} is excluded exactly as documented in Deliverable 3; "
      "no excluded checkpoint can enter the final manifest.")

---
# Part 1, Priors

The deliverable's Bayesian section names a prior for every parameter. This part does two things
with them, **before any observation exists**:

1. writes them down in one table, so the reported analysis and the document cannot drift apart;
2. checks what they *imply*. A prior is not weakly informative because it is labelled weakly
   informative, it is weakly informative if the data it predicts are plausible and if it does not
   quietly assert the conclusion. We therefore push each prior through its own likelihood and look
   at the resulting distribution of the quantities we care about: the recovery ratio $e^{\bar\beta}$,
   the codelength ratio $e^{\theta_{lock}}$, the shape of a collapse comb.

Nothing here needs Chronos: the *design* (which geometries, which lock sites, which phases) is
fixed in advance by `probe_lib`, and only the responses are unknown. That is precisely what makes a
genuine prior predictive check possible.

In [ ]:
banner("PART 1, PRIORS")

# Prior scales are preregistered here, including the sensitivity ladder used in Part 5.
PRIOR_SCALE = 0.5                       # the deliverable's primary choice
BASELINE_SCALE = 1.5                    # beta_bar only: a log-odds intercept needs more room than
                                        # an effect, since the baseline hit rate is nowhere near
                                        # 0.5 and log-odds of 0.9 is already 2.2
PRIOR_LADDER = [0.25, 0.5, 1.0]         # sceptical / primary / wide, for the sensitivity refits
NU = 4                                  # Student-t degrees of freedom, as written in the .tex

# Decision thresholds, preregistered so they cannot be chosen after seeing the posterior.
ATTENUATION_20 = np.log(0.8)            # "odds fall by at least 20%"  ->  gamma < log 0.8
ROPE_LOG = np.log(1.1)                  # practical equivalence: a +/-10% effect is no effect
ROPE_SLOPE = 0.1                        # H3a / H3b movement: |kappa_F - 1| < 0.1

PRIOR_SPEC = pd.DataFrame([
    # model, parameter, prior, role, where it is written in the deliverable
    ("A/C", "gamma",         f"StudentT(nu=4, 0, {PRIOR_SCALE})",    "population phase-lock effect, on the log-odds scale", "Eq. (9)"),
    ("A/C", "beta_bar",      f"StudentT(nu=4, 0, {BASELINE_SCALE})", "baseline log-odds of a correct reconstruction at a control", "Eq. (9)"),
    ("A",   "delta_O",       f"StudentT(nu=4, 0, {PRIOR_SCALE})",    "slope in the centred overlap Otilde; M1", "Eq. (9)"),
    ("A",   "delta_P",       f"StudentT(nu=4, 0, {PRIOR_SCALE})",    "slope in the centred log patch size", "Eq. (9)"),
    ("A/C", "tau",           f"HalfStudentT(nu=4, {PRIOR_SCALE})",   "spread of configuration effects", "Eq. (9)"),
    ("A/C", "sigma_harm",    f"HalfStudentT(nu=4, {PRIOR_SCALE})",   "spread across lock harmonics", "Eq. (9)"),
    ("A/C", "sigma_bg",      f"HalfStudentT(nu=4, {PRIOR_SCALE})",   "spread across background realisations", "Eq. (9)"),
    # A Bernoulli likelihood carries no residual scale: the old "sigma" row of the contrast model
    # has no counterpart here, and tab:bayesPriors no longer prints one.
    ("C",   "sigma_phase",   "HalfNormal(0.25)",                   "spread of the per-phase offsets (8 phase bins)", "Eq. (11)"),
    ("B",   "alpha_0",       "Normal(log mean(y), 1)",             "baseline log codelength", "Eq. (10)"),
    ("B",   "theta_lock",    "Normal(0, 0.5)",                     "log codelength expansion at a lock", "Eq. (10)"),
    ("B",   "k (shape)",     "Gamma(2, 0.1)",                      "Gamma dispersion", "Eq. (10)"),
    ("D1",  "alpha_g",       "Normal(0, 1)",                       "off-grid level of log z_g", "Eq. (12)"),
    ("D1",  "theta_S",       "Normal(0, 1)",                       "dip on the stride grid; H3 predicts < 0", "Eq. (12)"),
    ("D1",  "theta_P",       "Normal(0, 1)",                       "dip on the patch grid; H3 predicts < 0", "Eq. (12)"),
    ("D1",  "sigma",         "HalfNormal(1)",                      "residual scale", "Eq. (12)"),
    ("D2",  "kappa_S",       "Normal(0, 1)",                       "measured / stride-predicted spacing; H3a predicts 1", "Eq. (13)"),
    ("D2",  "kappa_P",       "Normal(0, 1)",                       "measured / patch-predicted spacing; H3b predicts 1", "Eq. (13)"),
    ("D2",  "sigma_extra",   "HalfNormal(5) Hz",                   "scatter beyond the sweep resolution", "Eq. (13)"),
], columns=["model", "parameter", "prior", "role", "deliverable"])
display(PRIOR_SPEC)

## 1.1, The design skeleton

Which geometries, lock sites, phases, backgrounds and generators the experiment will visit is
fixed by the design, not by the model. Building that skeleton now lets Part 1 simulate responses
from the prior on exactly the rows Part 2 will later fill in, a real prior predictive check on the
real design, not on a stylised one.

In [ ]:
def design_skeleton(n_phase=10, n_bg=6, generators=pl.GENERATORS) -> pd.DataFrame:
    """Compact support skeleton for prior prediction; Part 2 uses 100 backgrounds per generator."""
    rows = []
    for P, S in FULL_MODELS:
        offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
        for f_lock in [f for f, delta in offsets.items() if np.isfinite(delta)]:
            for generator in generators:
                for bg_id in range(n_bg):
                    for phase_idx, phase in enumerate(pl.phases_Sf(f_lock, n_phase)):
                        rows.append({
                            "model": pl.model_tag(P, S), "P": P, "S": S,
                            "overlap": (P - S) / P, "f_lock": f_lock,
                            "family": pl.lock_family(f_lock, P, S), "generator": generator,
                            "bg_id": bg_id, "phase_idx": phase_idx, "phase": float(phase),
                        })
    return pd.DataFrame(rows)

SKELETON = design_skeleton(n_phase=10, n_bg=6)
FULL_CONTRAST_ROWS = sum(
    len(pl.phases_Sf(f, 10)) * 100 * len(pl.GENERATORS)
    for P, S in FULL_MODELS
    for f in pl.f_lock(P, S)
    if np.isfinite(pl.control_offset(P, S, f))
)
print(f"prior-predictive support skeleton: {len(SKELETON):,} rows")
print(f"exact full contrast design: {FULL_CONTRAST_ROWS:,} rows (100 backgrounds/generator)")
display(SKELETON.groupby("model").agg(rows=("f_lock", "size"), locks=("f_lock", "nunique")))

## 1.2, Prior predictive for Models A and C (H1 behavioural, H2)

The contrast $d$ lives on a log-ratio scale, so $\bar\beta$ is read as $e^{\bar\beta}$: the
multiplicative change in forecast amplitude recovery at a lock relative to its controls. The
question a prior predictive answers is whether $\text{Student-}t_4(0, 0.5)$ describes *ignorance*
about that ratio or a belief about it.

Three things are checked:

* the prior is symmetric, it gives attenuation and amplification the same mass, so finding
  attenuation cannot be an artefact of the prior;
* it reaches far enough, a scale of $0.5$ puts $e^{\pm 0.5}\approx 1.65$ at one scale unit and the
  heavy $t_4$ tails admit far larger effects, so a real strong effect will not be shrunk away;
* the simulated contrasts $d$ are on the same order as contrasts that could physically occur
  (recovery ratios are bounded below by 0 and rarely exceed a few).

In [ ]:
def simulate_contrast_prior(skeleton: pd.DataFrame, scale: float,
                            baseline_scale: float = BASELINE_SCALE, n_draw: int = 2000,
                            rng=None) -> dict:
    """Draw parameters from the Model A prior and push them through the Bernoulli likelihood.

    Returns the population-level effect draws and the hit probabilities they imply on the real
    design, so the prior can be judged on the data it predicts rather than on its own parameters.
    """
    rng = rng or np.random.default_rng(SEED)
    cfg_codes, cfg_index = pd.factorize(skeleton["model"])
    O = skeleton.groupby("model")["overlap"].first().reindex(cfg_index).to_numpy()
    O_t = (O - O.mean()) / 0.5
    P = skeleton.groupby("model")["P"].first().reindex(cfg_index).to_numpy()
    logP_t = np.log(P) - np.mean(np.log(P))

    # priors, exactly as tabulated above
    draw = lambda s: stats.t.rvs(NU, scale=s, size=n_draw, random_state=rng.integers(1 << 31))
    gamma = draw(scale)
    beta_bar = draw(baseline_scale)
    delta_O, delta_P = draw(scale), draw(scale)
    half = lambda: np.abs(draw(scale))
    tau, sig_h, sig_b = half(), half(), half()

    # one simulated dataset per draw would be huge; simulate a random design row per draw instead,
    # which gives the marginal prior predictive of a single observation
    i = rng.integers(0, len(skeleton), n_draw)
    beta_c = (beta_bar + delta_O * O_t[cfg_codes[i]] + delta_P * logP_t[cfg_codes[i]]
              + tau * rng.normal(size=n_draw))
    eta_ctrl = beta_c + sig_h * rng.normal(size=n_draw) + sig_b * rng.normal(size=n_draw)
    p_ctrl = 1.0 / (1.0 + np.exp(-eta_ctrl))
    p_lock = 1.0 / (1.0 + np.exp(-(eta_ctrl + gamma)))
    return dict(gamma=gamma, beta_bar=beta_bar, delta_O=delta_O, delta_P=delta_P,
                p_ctrl=p_ctrl, p_lock=p_lock, odds_ratio=np.exp(gamma))


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for scale in PRIOR_LADDER:
    sim = simulate_contrast_prior(SKELETON, scale)
    lbl = f"scale {scale}" + ("  (primary)" if scale == PRIOR_SCALE else "")
    axes[0].hist(sim["gamma"], bins=80, range=(-3, 3), histtype="step", lw=1.6, label=lbl, density=True)
    axes[1].hist(np.clip(sim["odds_ratio"], 0, 4), bins=80, histtype="step", lw=1.6, label=lbl, density=True)
    axes[2].hist(sim["p_lock"], bins=80, range=(0, 1), histtype="step", lw=1.6, label=lbl, density=True)

axes[0].axvline(ATTENUATION_20, color="crimson", ls="--", lw=1, label="log 0.8 (odds fall 20%)")
axes[0].set_title(r"prior on $\gamma$ (log odds ratio)"); axes[0].set_xlabel(r"$\gamma$")
axes[1].axvline(1.0, color="k", lw=0.8)
axes[1].set_title(r"implied odds ratio $e^{\gamma}$"); axes[1].set_xlabel("odds ratio")
axes[2].set_title("prior predictive hit probability at a candidate"); axes[2].set_xlabel(r"$\Pr(h=1)$")
for a in axes: a.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG_DIR / "P1_prior_predictive_contrast.png", dpi=140,
                                bbox_inches="tight"); plt.show()

sim = simulate_contrast_prior(SKELETON, PRIOR_SCALE, n_draw=20000)
print(f"under the primary prior (scale {PRIOR_SCALE}, baseline scale {BASELINE_SCALE}):")
print(f"  P(gamma < 0)                    = {np.mean(sim['gamma'] < 0):.3f}   <- 0.5 means symmetric: "
      f"the prior does not favour the hypothesis")
print(f"  P(odds fall >=20%) a priori     = {np.mean(sim['gamma'] < ATTENUATION_20):.3f}")
print(f"  P(odds fall >=50%) a priori     = {np.mean(sim['gamma'] < np.log(0.5)):.3f}   <- reachable, "
      f"so a real strong effect will not be shrunk away")
print(f"  central 95% of the odds ratio   = [{np.exp(np.quantile(sim['gamma'], .025)):.2f}, "
      f"{np.exp(np.quantile(sim['gamma'], .975)):.2f}]")
print(f"  prior predictive hit rate       = [{np.quantile(sim['p_ctrl'], .025):.2f}, "
      f"{np.quantile(sim['p_ctrl'], .975):.2f}] at a control   <- must cover the observed rate, "
      f"or the baseline prior is fighting the data")


### The H2 phase term

Model C cuts the phase circle into eight equal slices and gives each its own offset. The estimand
$\sigma_\phi$ is the spread of those offsets: how much the lock deficit moves as the signal slides
through its cycle. Its prior, $\text{Half-}\mathcal N(0,0.25)$, puts mass on both sides of the
equivalence boundary $\log 1.1$, so the H2 verdict is decided by the data rather than by the prior.

In [ ]:
sigma_phase_prior = np.abs(RNG.normal(0, 0.25, 40000))     # Half-Normal(0.25)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(sigma_phase_prior, bins=90, density=True, color="steelblue", alpha=.8)
ax[0].axvline(ROPE_LOG, color="crimson", ls="--", label=f"ROPE edge log 1.1 = {ROPE_LOG:.3f}")
ax[0].set_title(r"prior on the phase spread $\sigma_\phi$"); ax[0].legend(fontsize=8)
ax[0].set_xlabel(r"$\sigma_\phi$")

# what a prior-plausible set of per-phase offsets looks like
for _ in range(25):
    ax[1].plot(np.arange(N_PHASE_BINS := 8),
               RNG.choice(sigma_phase_prior) * RNG.normal(size=8),
               marker="o", ms=3, alpha=.35, lw=1, color="steelblue")
ax[1].axhline(0, color="k", lw=.8)
ax[1].set_xlabel("phase bin"); ax[1].set_ylabel("offset")
ax[1].set_title("prior-plausible per-phase offsets")
fig.tight_layout(); fig.savefig(FIG_DIR / "P1_prior_predictive_phase.png", dpi=140,
                                bbox_inches="tight"); plt.show()

print(f"P(sigma_phase < ROPE) a priori = {np.mean(sigma_phase_prior < ROPE_LOG):.3f}   "
      f"<- mass on both sides, so the H2 verdict comes from the data, not this prior")

## 1.3, Prior predictive for Model B (H1 representational)

The response is a prequential codelength in bits: strictly positive, right-skewed, and with a
variance that grows with its mean. That is what the Gamma likelihood with a log link is for, and it
is why $\theta_{lock}$ is read multiplicatively, $e^{\theta_{lock}}$ is the factor by which
describing the labels costs more at a locked frequency.

In [ ]:
n = 20000
# alpha_0 ~ Normal(log(y_bar), 1) is evaluated in units of the observed baseline y_bar.
alpha_offset = RNG.normal(0.0, 1.0, n)
theta_p = RNG.normal(0.0, 0.5, n)
k_p = RNG.gamma(2.0, 1 / 0.1, n)
mu_locked_relative = np.exp(alpha_offset + theta_p)
y_locked_relative = RNG.gamma(k_p, mu_locked_relative / k_p)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(np.exp(theta_p), bins=90, range=(0, 4), density=True, color="darkorange", alpha=.85)
ax[0].axvline(1, color="k", lw=.8)
ax[0].axvline(1.2, color="crimson", ls="--", label="+20% codelength")
ax[0].set_title(r"implied codelength ratio $e^{\theta_{lock}}$")
ax[0].legend(fontsize=8)
ax[1].hist(np.clip(y_locked_relative, 0, 20), bins=90, density=True,
           color="darkorange", alpha=.85)
ax[1].set_title(r"prior predictive locked codelength $L/\bar L$")
fig.tight_layout()
fig.savefig(FIG_DIR / "P1_prior_predictive_codelength.png", dpi=140, bbox_inches="tight")
plt.show()

print(f"P(theta_lock > 0) a priori = {np.mean(theta_p > 0):.3f}")
print(f"P(codelength expands by >=20%) = {np.mean(theta_p > np.log(1.2)):.3f}")
print("The absolute bit scale is supplied by each observed table through log(mean(L)); "
      "the prior predictive therefore does not invent a 40-bit baseline.")

## 1.4, Prior predictive for Models D1 and D2 (H3)

D1 asks whether the token dispersion is lower on a predicted grid than off it: $\theta_S$ is that
difference, on a log scale, and H3 predicts $\theta_S<0$. Its $\mathcal N(0,1)$ prior is symmetric,
so a dip is not assumed.

D2 asks whether the measured comb spacing follows the predicted one. It is a plain regression of
measured spacing on $f_s/S$ and $f_s/P$, and **H3 predicts a slope of exactly 1 on $f_s/S$ and 0 on
$f_s/P$**. The prior on both slopes is centred on $0$, *no* relationship, so the hypothesis has to
be earned from the data.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))

theta_S = RNG.normal(0, 1, 20000)
ax[0].hist(theta_S, bins=80, density=True, color="steelblue", alpha=.85)
ax[0].axvline(0, color="k", lw=.9)
ax[0].set_title(r"D1 prior on $\theta_S$ (H3 predicts $<0$)")

kappa_S = RNG.normal(0, 1, 20000)
ax[1].hist(kappa_S, bins=80, density=True, color="seagreen", alpha=.85)
ax[1].axvline(1.0, color="crimson", ls="--", label=r"$\kappa_F=1$")
ax[1].axvspan(1 - ROPE_SLOPE, 1 + ROPE_SLOPE, color="crimson", alpha=.12)
ax[1].axvline(0.0, color="k", lw=.8)
ax[1].set_title(r"D2 prior on $\kappa_F$")
ax[1].legend(fontsize=8)

predicted = RNG.choice([pl.FS / S for _, S in FULL_MODELS]
                       + [pl.FS / P for P, _ in FULL_MODELS], size=20000)
sigma_extra = np.abs(RNG.normal(0, 5, 20000))
f1_prior = RNG.normal(kappa_S * predicted, np.sqrt(sigma_extra ** 2 + 1.0 ** 2))
ax[2].hist(np.clip(f1_prior, -100, 250), bins=100, density=True, color="mediumpurple", alpha=.8)
ax[2].axvspan(*pl.BAND, color="0.5", alpha=.08, label="analysed band")
ax[2].set_title(r"D2 prior predictive $\hat f_1$ [Hz]")
ax[2].legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "P1_prior_predictive_H3.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"P(kappa_F in ROPE around 1) a priori = "
      f"{np.mean(np.abs(kappa_S - 1) < ROPE_SLOPE):.3f}")
print(f"P(prior-predictive f1 < 0) = {np.mean(f1_prior < 0):.3f}; this exposes the deliberately "
      "broad Normal D2 prior fixed in Deliverable 3 rather than hiding it.")

## 1.5, Checkpoint

The prior specification and its predictive summaries are stored so the reported analysis can be
traced back to priors that were fixed before the data existed.

In [ ]:
prior_summary = {
    "prior_scale": PRIOR_SCALE, "prior_ladder": PRIOR_LADDER, "nu": NU,
    "attenuation_20": float(ATTENUATION_20), "rope_log": float(ROPE_LOG),
    "rope_slope": ROPE_SLOPE,
    "p_attenuation20_prior": float(np.mean(sim["gamma"] < ATTENUATION_20)),
    "baseline_scale": BASELINE_SCALE,
    "p_sigma_phase_in_rope_prior": float(np.mean(sigma_phase_prior < ROPE_LOG)),
    "p_kappaS_in_rope_prior": float(np.mean(np.abs(kappa_S - 1) < ROPE_SLOPE)),
    "support_skeleton_rows": int(len(SKELETON)),
    "full_contrast_rows": int(FULL_CONTRAST_ROWS), "seed": SEED,
}
save_json(prior_summary, "01_prior_spec.json")
save_df(PRIOR_SPEC, "01_prior_spec.parquet")
print(json.dumps(prior_summary, indent=2))

---
# Part 2, The instrument: what the indicator can and cannot see

The response of Eq. (8) is not a direct reading of Chronos. It is a reading of Chronos *through an
estimator*, and an estimator has a ceiling: on an arm where the tone is not among the strongest
components of the true continuation, no forecast, however perfect, can score a hit. Before any GPU
time is spent, this part measures that ceiling and the chance floor beneath it, and fixes the two
knobs that set them.

**Two reference forecasters bound the scale.** The *truth* forecaster returns the true continuation
verbatim: it cannot be beaten, so its hit rate is the ceiling. The *blind* forecaster returns the
background with no tone at all: it knows nothing, so its hit rate is the chance floor. A usable
estimator is one where these are far apart, because $\gamma$ is estimated inside that gap.

**Two knobs.** $\Delta$, the tolerance of Eq. (8), and $K$, how many spectral peaks count as "the
frequency the forecast rebuilt". $K=1$ asks whether the tone is *the* strongest line, which on a
corpus whose own energy sits low in the band it usually is not; the claim under test is that the
tone is among the strongest components, so $K$ is part of the specification and not a relaxation of
it. The pair is chosen by maximising ceiling minus floor, and both are written into the manifest,
because both change what $h$ means.

Nothing here loads a model.


In [ ]:
banner("PART 2, THE INSTRUMENT")

import collect
CFG = collect.Config.smoke_cfg() if IS_SMOKE else collect.Config()
CFG.batch_size = 32 if IS_SMOKE else 64
CFG.fhat_topk, CFG.fhat_tol_hz = FHAT_TOPK, FHAT_TOL_HZ
assert list(pl.DELIVERABLE3_MODELS) == FULL_MODELS
assert CFG.smoke == IS_SMOKE

SWEEP_FILE = "02_estimator_sweep.parquet"
SWEEP_DELTAS = [1.0, 2.0]
SWEEP_KS = [1, 2, 3, 5]
SWEEP_N_BG = 2 if IS_SMOKE else 8


def estimator_sweep(deltas=SWEEP_DELTAS, ks=SWEEP_KS, n_bg=SWEEP_N_BG) -> pd.DataFrame:
    """Ceiling and floor of the localisation indicator over Delta x K, on the real corpora.

    The truth forecaster returns the true continuation, so it defines the ceiling; the blind
    forecaster returns the background alone, so it defines the chance floor. Both are built from
    the same windows, so the gap between them is not confounded by which arms were drawn. The
    peaks are computed once at max(ks) and re-read at each smaller k, since the estimator ranks
    them by height: the top 3 of a row are its top 5 truncated.
    """
    sites = sorted({f for P, S in FULL_MODELS for f in pl.f_lock(P, S)})
    truth_rows, blind_rows, carried = [], [], []
    for generator in CFG.generators:
        for bg in pl.background_pool(generator, n_bg, pl.CTX + pl.PRED):
            for f in sites:
                phase = float(pl.phases_Sf(f, 1)[0])
                full = pl.build_context(bg, f, phase, pl.CTX + pl.PRED)
                truth_rows.append(full[pl.CTX:])          # the true future, tone included
                blind_rows.append(bg[pl.CTX:])            # the same window with no tone at all
                carried.append(f)
    carried = np.asarray(carried, float)
    k_max = max(ks)
    peaks = {"truth (ceiling)": pl.dominant_freqs(np.stack(truth_rows), k=k_max),
             "blind (floor)": pl.dominant_freqs(np.stack(blind_rows), k=k_max)}

    rows = []
    for k in ks:
        for delta in deltas:
            scores = {name: float(pl.localisation_hit(p[:, :k], carried, tol=delta).mean())
                      for name, p in peaks.items()}
            rows.append({"k": k, "delta_hz": delta,
                         "ceiling": scores["truth (ceiling)"], "floor": scores["blind (floor)"],
                         "gap": scores["truth (ceiling)"] - scores["blind (floor)"],
                         "n_windows": len(carried)})
    return pd.DataFrame(rows).sort_values("gap", ascending=False).reset_index(drop=True)


sweep = load_df(SWEEP_FILE) if have(SWEEP_FILE) else save_df(estimator_sweep(), SWEEP_FILE)
display(sweep.round(4))
best = sweep.iloc[0]
print(f"widest gap: k={int(best['k'])}, Delta={best['delta_hz']} Hz  ->  "
      f"ceiling {best['ceiling']:.3f}, floor {best['floor']:.3f}, gap {best['gap']:.3f}")
print(f"in use    : k={FHAT_TOPK}, Delta={FHAT_TOL_HZ} Hz")
if (int(best["k"]), float(best["delta_hz"])) != (FHAT_TOPK, FHAT_TOL_HZ):
    print("!" * 78)
    print("The sweep does not agree with the configured estimator. Either set FHAT_TOPK and")
    print("FHAT_TOL_HZ in Part 0.3 to the widest-gap pair and use a fresh RUN_ID, or record why")
    print("the configured pair is preferred. Both values define h and both are in the manifest.")
    print("!" * 78)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for delta, style in zip(SWEEP_DELTAS, ("-o", "--s")):
    g = sweep[sweep["delta_hz"] == delta].sort_values("k")
    axes[0].plot(g["k"], g["ceiling"], style, lw=1.6, color="#1565c0",
                 label=f"ceiling, $\\Delta$={delta} Hz")
    axes[0].plot(g["k"], g["floor"], style, lw=1.4, color="#c62828",
                 label=f"floor, $\\Delta$={delta} Hz")
    axes[1].plot(g["k"], g["gap"], style, lw=1.8, label=f"$\\Delta$={delta} Hz")
axes[0].set_xlabel("$K$, peaks retained"); axes[0].set_ylabel("hit rate")
axes[0].set_ylim(-0.02, 1.02); axes[0].set_title("what the instrument can and cannot see")
axes[0].legend(fontsize=7)
axes[1].axvline(FHAT_TOPK, color="grey", ls=":", lw=1, label=f"in use, $K$={FHAT_TOPK}")
axes[1].set_xlabel("$K$, peaks retained"); axes[1].set_ylabel("ceiling $-$ floor")
axes[1].set_title("the room $\\gamma$ is estimated in"); axes[1].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "P2_estimator_sweep.png", dpi=140, bbox_inches="tight"); plt.show()


---
# Part 3, Observation: Chronos produces the data

This is the only part that loads a model, and the only part that wants a GPU. It runs
`collect.py`, which visits the 15 geometries and writes five tidy tables. Everything after this
point treats those tables as plain data.

**What is measured, and why each measurement exists**

| table | measurement | why |
|---|---|---|
| `contrasts` | forecast amplitude recovery $R$ at each lock $f_k$ and at both controls $f_k\pm 0.25 f_s/S$, sharing background and phase | the *behavioural* endpoint of Eq. (8); the phase index is kept so H2 is testable |
| `mdl_cells` | prequential codelength $L(D)$ of a probe separating $f_c-1$ Hz from $f_c+1$ Hz, per probe stage | the *representational* endpoint of Eq. (10) |
| `mdl_bandtasks` | the seven hierarchical band tasks, with shuffled-label and random-init controls | descriptive cross-check of overall decodability |
| `collapse` | across-patch token dispersion $z_g(f)$ on the union grid, in three signal modes | the location endpoint of Eq. (12) |
| `sites` | detected collapse sites, their fundamental $f_1$ and spacing $\hat\Delta$ | the movement endpoint of Eq. (13) |

**Signals.** Following the deliverable's Data section, the tone rides on a unit-variance
**TSMixup** or **KernelSynth** background at SNR 4, the two synthetic corpora the project
generates. The collapse sweep additionally records the **pure sinusoid** mode, because only there
is the degeneracy exact ($z=0$ when consecutive patches coincide); the background modes show the
same comb as a deep dip and are the realistic-signal cross-check.

**Resumption.** `collect.py` shards per geometry under `data/raw/`. A geometry whose shards exist
is skipped *and its model is never loaded*, so an interrupted run costs only the geometry it died
on.

In [ ]:
banner("PART 2 - OBSERVATION (CHRONOS)")

import collect

# CFG was built and checked in Part 2; the estimator knobs it carries are already the manifest's.
assert CFG.fhat_topk == FHAT_TOPK and CFG.fhat_tol_hz == FHAT_TOL_HZ

print("reportable design:", {k: v for k, v in vars(CFG).items() if k != "generators"})
print("generators:", CFG.generators)
print("session:", [pl.model_tag(P, S) for P, S in SESSION_MODELS])

## 2.1 - Collection and MCMC cost before execution

In [ ]:
# ---- what Part 2 costs, before it starts ---------------------------------------------------
# On a CPU runtime the collection is the long pole, so this cell counts the forward passes each
# geometry needs and turns them into an estimate. Nothing here loads a model unless CALIBRATE is
# on, and the counts are exact: they are read from the same probe_lib functions the collectors
# loop over.
BUDGET_HOURS = 70.0     # what you have to spend, for the session split printed at the end
CALIBRATE    = True     # time one batch on the smallest geometry to get real seconds per pass
SEC_FORECAST = 0.035    # fallback, seconds per forecast on one CPU core-set; overwritten if
SEC_CAPTURE  = 0.020    # CALIBRATE succeeds. A forecast decodes the horizon, a capture does not.


def _counts(P, S, cfg):
    """Exact forward-pass counts for one geometry, split into forecasts and state captures."""
    offsets = {f: pl.control_offset(P, S, f) for f in pl.f_lock(P, S)}
    sites = [f for f, d in offsets.items() if np.isfinite(d)]
    n_gen = len(cfg.generators)

    # contrasts: every triplet arm, at every phase, on every background
    fore = sum(len(pl.phases_Sf(fk, cfg.n_phase_contrast)) for fk in sites) * cfg.n_bg * 3 * n_gen

    # mdl cells: two classes per centre, capped at mdl_n_per_class contexts each
    centers = {round(f, 6) for fk in sites
               for f in (fk, fk - offsets[fk], fk + offsets[fk])}
    cap = len(centers) * 2 * cfg.mdl_n_per_class

    # band tasks: the descriptive sweep, plus the untrained clone on a 4x coarser grid
    if cfg.band_tasks:
        fl = np.arange(pl.BAND[0], pl.BAND[1] + 1e-9, cfg.bt_step)
        cap += sum(len(pl.phases_Sf(f, cfg.bt_n_phase)) for f in fl)
        if cfg.bt_random_init:
            co = np.arange(pl.BAND[0], pl.BAND[1] + 1e-9, cfg.bt_step * 4)
            cap += sum(len(pl.phases_Sf(f, max(2, cfg.bt_n_phase // 2))) for f in co)

    # collapse: the union grid, once per replicate per signal mode
    cap += len(pl.union_grid(FULL_MODELS, cfg.collapse_step)) * cfg.collapse_reps * \
        len(cfg.collapse_modes)
    return fore, cap


if CALIBRATE and HAVE_TORCH:
    try:
        import time
        _p = pl.Probe(8, 8, device=CFG.device, batch_size=CFG.batch_size)
        try:
            _n = min(32, CFG.batch_size)
            _ctx = np.stack([pl.build_context(None, 40.0, 0.0, pl.CTX + pl.PRED)] * _n)
            _t = time.perf_counter()
            _p.recovery(_ctx[:, :pl.CTX], _ctx[:, pl.CTX:], np.full(_n, 40.0))
            SEC_FORECAST = (time.perf_counter() - _t) / _n
            _t = time.perf_counter()
            _p.capture_reg(_ctx[:, :pl.CTX])
            SEC_CAPTURE = (time.perf_counter() - _t) / _n
        finally:
            _p.close()
        print(f"calibrated on p8-s8, device={_p.device}: {SEC_FORECAST*1000:.1f} ms per forecast, "
              f"{SEC_CAPTURE*1000:.1f} ms per capture")
    except Exception as _e:
        print(f"calibration skipped ({type(_e).__name__}: {_e}); using the defaults above")
else:
    print("calibration off or torch unavailable; using the defaults above")

rows = []
for _P, _S in FULL_MODELS:
    f, c = _counts(_P, _S, CFG)
    rows.append(dict(model=pl.model_tag(_P, _S), forecasts=f, captures=c,
                     hours=(f * SEC_FORECAST + c * SEC_CAPTURE) / 3600))
COST = pd.DataFrame(rows).sort_values("hours")
COST["cumulative_h"] = COST["hours"].cumsum()
display(COST.round({"hours": 2, "cumulative_h": 2}))

_tot = COST["hours"].sum()
print(f"\ntotal: {COST.forecasts.sum():,} forecasts + {COST.captures.sum():,} captures "
      f"= {_tot:.1f} h at the rates above")
print(f"budget: {BUDGET_HOURS:.0f} h  ->  "
      + ("fits, with %.0f h to spare" % (BUDGET_HOURS - _tot) if _tot <= BUDGET_HOURS
         else "SHORT by %.0f h; see the note below" % (_tot - BUDGET_HOURS)))

# A session plan. Shards are written per table per geometry and skipped on the next run, so a
# session that ends mid-geometry loses only the table in flight.
SESSION_H = 3.0
_grp, _acc, _plan = [], 0.0, []
for _, r in COST.iterrows():
    if _acc + r.hours > SESSION_H and _grp:
        _plan.append((_grp, _acc)); _grp, _acc = [], 0.0
    _grp.append(r.model); _acc += r.hours
if _grp:
    _plan.append((_grp, _acc))
print(f"\nsuggested split into ~{SESSION_H:g} h sessions:")
for _i, (_g, _h) in enumerate(_plan, 1):
    print(f"  session {_i} ({_h:.1f} h):  models={_g}")
print("\nrun one session with, for example:")
print("    DATA = collect.collect_all(DATA_DIR, models=[(8, 8), (16, 8)], cfg=CFG, planned_models=FULL_MODELS, allow_partial=True)")
if not HAVE_TORCH:
    print("\ntorch is not available here, so Part 2 cannot run in this runtime at all.")

print("\nThe estimate above covers Chronos forward passes only. It excludes parameter recovery, "
      "the empirical posterior fits, four full sensitivity refits, LOO and posterior-predictive "
      "simulation. Those stages use 4 chains and the preregistered 2000 tune + 2000 retained "
      "draws; do not interpret this collection estimate as total wall time.")

In [ ]:
# Session subsets are allowed only as resumable collection work. Inference remains locked until
# the manifest contains complete, validated shards for all 15 planned geometries.
if HAVE_TORCH:
    DATA = collect.collect_all(
        DATA_DIR,
        models=SESSION_MODELS,
        cfg=CFG,
        planned_models=FULL_MODELS,
        allow_partial=set(SESSION_MODELS) != set(FULL_MODELS),
    )
    collection_manifest = json.loads((DATA_DIR / collect.MANIFEST_NAME).read_text(encoding="utf-8"))
    if collection_manifest["status"] != "complete":
        completed = sorted({key.split("__", 1)[1] for key in collection_manifest["shards"]})
        raise RuntimeError(
            "This collection session is safely checkpointed but the 15-model design is partial. "
            f"Completed shard tags: {completed}. Run the remaining sessions; do not continue to MCMC.")
else:
    print("Torch/Chronos unavailable: validating a completed Drive collection.")

DATA = collect.load_collection(DATA_DIR, cfg=CFG, planned_models=FULL_MODELS, require_complete=True)
collection_manifest_path = DATA_DIR / collect.MANIFEST_NAME
ANALYSIS_MANIFEST["collection_manifest_sha256"] = cp.sha256_file(collection_manifest_path)
cp.atomic_json(ANALYSIS_MANIFEST_PATH, ANALYSIS_MANIFEST)

contrasts = DATA["contrasts"]
mdl_cells = DATA["mdl_cells"]
collapse = DATA["collapse"]
sites = DATA["sites"]
bandtasks = DATA.get("mdl_bandtasks")
print("rows:", {name: len(frame) for name, frame in DATA.items()})

## 2.2 - Generated-signal archive and quality gate

In [ ]:
sig_dir = DATA_DIR / "signals"
sig_index = pd.read_parquet(sig_dir / "signals_index.parquet")
required_signal_columns = {
    "generator", "seed", "mean", "std", "sha256", "recipe_sha256",
    "pool_max_abs_corr", "pool_effective_rank", "pool_quality_ok",
}
if not required_signal_columns.issubset(sig_index.columns):
    raise ValueError("signal archive predates the Deliverable 3 quality/provenance schema")
if not sig_index["pool_quality_ok"].all():
    raise ValueError("a generated-signal pool failed its quality gate")
for row in sig_index.itertuples():
    if cp.sha256_file(sig_dir / row.file) != row.sha256:
        raise ValueError(f"signal hash mismatch: {row.file}")

display(sig_index.groupby("generator").agg(
    draws=("file", "size"), max_abs_mean=("mean", lambda x: float(np.max(np.abs(x)))),
    mean_std=("std", "mean"), max_abs_corr=("pool_max_abs_corr", "first"),
    effective_rank=("pool_effective_rank", "first"),
))

fig, axes = plt.subplots(2, 2, figsize=(13, 5), sharex=True)
for row_index, generator in enumerate(CFG.generators):
    background = np.load(sig_dir / f"background_{generator}_bg0.npy", allow_pickle=False)
    time = np.arange(len(background)) / pl.FS
    axes[row_index][0].plot(time, background, lw=.8, color="0.4")
    axes[row_index][0].axvline(pl.CTX / pl.FS, color="k", ls=":", lw=1)
    axes[row_index][0].set_ylabel(generator)
    model_input = pl.build_context(background, 64.0, 0.0, len(background))
    axes[row_index][1].plot(time, model_input, lw=.8, color="#c62828")
axes[0][0].set_title("centred, unit-variance archived background")
axes[0][1].set_title("background + 64 Hz tone at SNR 4")
fig.tight_layout()
fig.savefig(FIG_DIR / "P2_signals.png", dpi=140, bbox_inches="tight")
plt.show()

## 2.3 - Does the collected design support the inference?

In [ ]:
design_check = collect.check_design(DATA, expected_models=FULL_MODELS, cfg=CFG)
display(pd.DataFrame(design_check["d2_identification"]).T)

# Every arm enters A and C. The indicator response of Eq. (8) has no degenerate value: an arm
# whose forecast rebuilds nothing measurable returns h = 0, which is a reading and not a
# division by a noise floor, so the recovery floor the contrast model needed to keep a log-ratio
# finite has no counterpart here and nothing is filtered out.
arms = contrasts.reset_index(drop=True)
if arms.empty:
    raise ValueError("no localisation arms were collected")
if arms["h"].nunique() < 2:
    raise ValueError(f"the localisation indicator is constant at {arms['h'].mean():.0f}; "
                     f"gamma is not identified on this collection")

balance = arms.groupby(["model", "P", "S"]).agg(
    rows=("h", "size"), hits=("h", "sum"), lock_rows=("is_lock", "sum"),
    locks=("f_lock", "nunique"), phases=("phase_idx", "nunique"),
    generators=("generator", "nunique"), backgrounds=("bg_id", "nunique"),
).reset_index()
balance["hit_rate"] = (balance["hits"] / balance["rows"]).round(3)
display(balance)
# The instrument's own ceiling, per geometry. h_truth is the same estimator run on the TRUE
# continuation, so an arm with h_truth = 0 is one where the tone is not among the K strongest
# components of the window that actually follows: no forecast can score there, and the row
# carries no evidence about the model. The ceiling is NOT uniform over the design - it depends on
# where the arm sits relative to the corpus's own energy, and candidate frequencies are c*fs/S and
# k*fs/P - so it varies with the design axes and confounds delta_O and delta_P. gamma survives it,
# being a within-triplet contrast on arms that share a background and a phase; M1 does not.
ceiling = arms.groupby(["model", "P", "S"]).agg(
    ceiling=("h_truth", "mean"), sites=("f_lock", "nunique"),
    sites_surviving=("f_lock", lambda s: arms.loc[s.index][arms.loc[s.index, "h_truth"] == 1]
                     ["f_lock"].nunique()),
).reset_index().round(3)
display(ceiling)
if ceiling["ceiling"].std() > 0.05:
    print(f"NOTE  the ceiling varies across geometries (sd {ceiling['ceiling'].std():.3f}); "
          f"M1 is read against the conditioned fit of Part 5, not the unconditioned one")

# Two scopes, both fitted, compared by LOO in Part 5 rather than chosen here. Dropping is per ARM,
# never per triplet: the one-row-per-arm layout supports it, and a triplet-level rule would
# discard every arm of a triplet whenever one of the three was unmeasurable.
MEASURABLE_IDX = np.flatnonzero(arms["h_truth"].to_numpy(bool))
arms_measurable = arms.iloc[MEASURABLE_IDX].reset_index(drop=True)
print(f"\nscope A  unconditioned : {len(arms):,} arms, all of them")
print(f"scope A' conditioned   : {len(arms_measurable):,} arms where the tone is measurable "
      f"({len(arms_measurable) / max(len(arms), 1):.1%} of the design)")
if arms_measurable["h"].nunique() < 2 or arms_measurable["is_lock"].nunique() < 2:
    raise ValueError("the conditioned scope is degenerate; gamma is not identified on it")

by_arm = arms.groupby("is_lock")["h"].mean()
print(f"\narms entering A/C: {len(arms):,}   overall hit rate {arms['h'].mean():.3f}   "
      f"instrument ceiling {arms['h_truth'].mean():.3f}")
print(f"  raw hit rate at controls {by_arm.get(0, float('nan')):.3f}, "
      f"at candidates {by_arm.get(1, float('nan')):.3f}   "
      f"(descriptive only; the estimand is gamma, which is adjusted for every grouping)")

## 2.4 - Raw observations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))

# (a) hit rate against cycles-per-patch, candidates against their controls. H1 predicts the
# candidate curve below the control curve; this is the raw picture behind gamma, unadjusted.
for is_lock, style in ((0, dict(marker="o", ls="-", color="#1565c0")),
                       (1, dict(marker="s", ls="--", color="#c62828"))):
    g = arms[arms["is_lock"] == is_lock]
    prof = g.groupby(g["cpp"].round(3), as_index=False)["h"].mean()
    axes[0].plot(prof["cpp"], prof["h"], lw=1.4, ms=4, alpha=.85,
                 label="candidates" if is_lock else "controls", **style)
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_xlabel("cycles per patch  cpp = $f_k P / f_s$")
axes[0].set_ylabel("$\\Pr(h=1)$   (lower = the frequency is rebuilt less often)")
axes[0].set_title("Eq. (8) localisation rate, all arms"); axes[0].legend(fontsize=7)

# (b) the collapse profile of one geometry, with both competing grids drawn on it
sub = collapse[(collapse["model"] == "p16-s8") & (collapse["mode"] == "tsmixup")]
if len(sub) == 0:
    sub = collapse[collapse["model"] == collapse["model"].iloc[0]]
prof = sub.groupby("f", as_index=False)["z"].mean().sort_values("f")
P0, S0 = int(sub["P"].iloc[0]), int(sub["S"].iloc[0])
axes[1].plot(prof["f"], prof["z"], color="#c62828", lw=1.3)
for j, f in enumerate(pl.stride_locks(S0)):
    axes[1].axvline(f, color="#6a1b9a", ls="--", lw=1, alpha=.75,
                    label=f"predicted stride lock $c f_s/S$ (S={S0})" if j == 0 else None)
for j, f in enumerate(pl.patch_nulls(P0)):
    axes[1].axvline(f, color="#1565c0", ls=":", lw=1, alpha=.6,
                    label=f"predicted patch null $k f_s/P$ (P={P0})" if j == 0 else None)
axes[1].set_xlabel("frequency [Hz]"); axes[1].set_ylabel("across-patch token dispersion $z$")
axes[1].set_title(f"Eq. (12) collapse profile, p{P0}-s{S0}"); axes[1].legend(fontsize=7)

fig.tight_layout(); fig.savefig(FIG_DIR / "P2_raw_data.png", dpi=140, bbox_inches="tight"); plt.show()

display(sites[sites.rep == -1][["model", "P", "S", "mode", "n_sites", "f1", "delta_hat"]]
        .assign(fs_over_S=lambda d: (pl.FS / d["S"]).round(2),
                fs_over_P=lambda d: (pl.FS / d["P"]).round(2)).round(2))

---
# Part 3, The likelihood

Part 1 fixed the priors; Part 2 produced the observations. This part supplies the third ingredient
and, more importantly, **checks that it works before it is trusted**.

Three things happen here:

1. **Ãƒâ€šSection 3.1, the likelihood, written out.** The Student-$t_4$ density that Eq. (9) prescribes is
   implemented directly in NumPy and checked against SciPy. Seeing it as an explicit function makes
   concrete what the sampler is otherwise doing invisibly, and shows *why* $t_4$ rather than a
   Gaussian: its tails make a handful of extreme contrasts cost far less, so the population effect
   is not dragged around by them.
2. **Ãƒâ€šSection 3.2, the five models.** Each is a small factory function returning a PyMC model.
3. **Ãƒâ€šSection 3.3, parameter recovery.** The likelihood is used to *simulate* contrasts with a known
   effect on the real design matrix, and the model is refitted. If a known $\log 0.5$ cannot be
   recovered from this design, no posterior fitted on the real data means anything.

## 3.1, The Bernoulli likelihood with a logit link

For arm $i$ with linear predictor $\eta_i$, the response of Eq. (8) is an indicator and the
likelihood is

$$\log p(h_i \mid \eta_i) = h_i\,\eta_i - \log\!\left(1 + e^{\eta_i}\right),
\qquad p_i = \operatorname{logit}^{-1}(\eta_i) = \frac{1}{1+e^{-\eta_i}}.$$

There is no scale parameter: a Bernoulli's variance is fixed by its mean at $p(1-p)$, so the
contrast model's residual $\sigma$ has no counterpart here and `tab:bayesPriors` no longer prints
one. The link is what makes $\gamma$ multiplicative. Adding $\gamma$ to $\eta$ multiplies the
odds $p/(1-p)$ by $e^{\gamma}$ whatever the baseline is, which is why the estimand can be quoted
as one number across geometries whose baseline hit rates differ.

Two consequences worth keeping in view. The cost of a residual is bounded below but not above:
an arm the model is confident about and gets wrong costs $-\log p_i$, which diverges as
$p_i\to0$, so a *systematically* mispredicted stratum is expensive while a single surprise is
not. And the information per row is at most one bit and falls away as $p$ approaches either
extreme; near a hit rate of $0.9$ each row carries roughly a third of what it carries at $0.5$,
which is why the design needs many arms per group rather than a few precise ones.


In [ ]:
banner("PART 3, LIKELIHOOD")

def bernoulli_logit_logpmf(h, eta):
    """Log pmf of Bernoulli(logit^-1(eta)), written out rather than imported.

    Evaluated as h*eta - log(1+e^eta) via np.logaddexp, which is the numerically stable form: the
    naive route through p = 1/(1+exp(-eta)) underflows to 0 or 1 around |eta| > 37 and then takes
    log(0). The linear predictor here carries a baseline near 1.9 plus three group offsets, so
    values that far out are not reachable in practice, but the fit is not asked to rely on that.
    """
    eta = np.asarray(eta, float)
    return np.asarray(h, float) * eta - np.logaddexp(0.0, eta)

# cross-check against SciPy: the two must agree to machine precision
_eta = np.linspace(-8, 8, 501)
_p = 1.0 / (1.0 + np.exp(-_eta))
for _h in (0.0, 1.0):
    assert np.allclose(bernoulli_logit_logpmf(_h, _eta), stats.bernoulli.logpmf(_h, _p))
print("bernoulli_logit_logpmf matches scipy.stats.bernoulli.logpmf to machine precision")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(_eta, _p, lw=1.8, label=r"$\operatorname{logit}^{-1}(\eta)$")
ax[0].axhline(0.5, color="k", lw=.7); ax[0].axvline(0, color="k", lw=.7)
ax[0].axhline(0.40, color="crimson", ls=":", lw=1,
              label="measured ceiling: a PERFECT forecaster scores 0.40 at SNR 1.25")
ax[0].set_xlabel(r"linear predictor $\eta$"); ax[0].set_ylabel(r"$\Pr(h=1)$")
ax[0].set_title("the link"); ax[0].legend(fontsize=8)

ax[1].plot(_eta, -bernoulli_logit_logpmf(1.0, _eta), lw=1.8, label="cost of a miss ($h=1$)")
ax[1].plot(_eta, -bernoulli_logit_logpmf(0.0, _eta), lw=1.4, ls="--", label="cost of a hit ($h=0$)")
ax[1].set_ylim(0, 8); ax[1].set_xlabel(r"$\eta$")
ax[1].set_title("cost of a residual  $-\log p$   (confident and wrong is what is expensive)")
ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "P3_likelihood.png", dpi=140, bbox_inches="tight"); plt.show()


### The likelihood is what makes the effect identifiable

A quick profile makes the point that Part 1's prior predictive could not: holding the nuisance
parameters at sensible values and sweeping $\bar\beta$, the observed contrasts pick out a narrow
range. The prior was flat over a factor of ~3; the likelihood is not.

In [ ]:
# The profile below is the Bernoulli counterpart of the contrast model's: it holds every grouping
# at zero and sweeps gamma alone, so it shows where the DATA place the lock effect before any
# hierarchy is fitted, against the prior that will be combined with it.
lock = arms["is_lock"].to_numpy(float)
hits = arms["h"].to_numpy(float)
base = float(np.clip(hits[lock == 0].mean(), 1e-6, 1 - 1e-6))     # controls set the baseline
b0 = np.log(base / (1 - base))

grid_g = np.linspace(-2.5, 2.5, 400)
eta = b0 + np.outer(grid_g, lock)                                  # [grid, obs]
ll = (hits * eta - np.logaddexp(0.0, eta)).sum(axis=1)
prior_ll = stats.t.logpdf(grid_g, NU, scale=PRIOR_SCALE)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(grid_g, ll - ll.max(), label="log-likelihood profile (data)", lw=1.8)
ax.plot(grid_g, prior_ll - prior_ll.max(), label="log prior", lw=1.4, ls="--")
ax.axvline(grid_g[ll.argmax()], color="crimson", lw=1,
           label=f"profile maximum = {grid_g[ll.argmax()]:+.3f}  "
                 f"(odds ratio {np.exp(grid_g[ll.argmax()]):.3f})")
ax.axvline(ATTENUATION_20, color="grey", lw=1, ls=":", label="support threshold, $\\log 0.8$")
ax.set_xlabel(r"$\gamma$"); ax.set_ylabel("relative log density"); ax.set_ylim(-30, 1)
ax.legend(fontsize=8); ax.set_title("the data, not the prior, locate the effect")
fig.tight_layout(); fig.savefig(FIG_DIR / "P3_profile.png", dpi=140, bbox_inches="tight"); plt.show()


## 3.2 - The five Deliverable 3 model families

In [ ]:
def _codes(series):
    """Integer codes plus the ordered level names, for PyMC `coords`."""
    codes, levels = pd.factorize(series)
    return np.asarray(codes), list(map(str, levels))


def _overlap_scaled(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred, scaled patch overlap O = (P-S)/P; one unit = 0.5 of overlap."""
    O = df.groupby("model")["overlap"].first().reindex(cfg_levels).to_numpy(float)
    return (O - O.mean()) / 0.5


def _logP_centred(df: pd.DataFrame, cfg_levels: list[str]) -> np.ndarray:
    """Centred log patch size.

    Deliverable 3, H1: "The overlap enters as a ratio and the patch size as log P, because the
    patch grid has spacing fs/P: equal steps in log P are then equal ratios of spacing." On a raw-P
    scale one coefficient would make 8->16 and 16->24 the same change, which the geometry does not.
    """
    Pv = df.groupby("model")["P"].first().reindex(cfg_levels).to_numpy(float)
    lp = np.log(Pv)
    return lp - lp.mean()


# --------------------------------------------------------------------------------------- #
def model_A_contrast(df: pd.DataFrame, scale: float = PRIOR_SCALE,
                     baseline_scale: float = BASELINE_SCALE, nu: int = NU,
                     link: str = "logit", config_level: str = "both") -> pm.Model:
    """Model A, H1 behavioural, and the configuration level that carries M1.

        h_i ~ Bernoulli(p_i)
        logit p_i = beta[config_i] + gamma * IsLock_i + u_harm[i] + u_bg[i]
        beta_c ~ Normal(beta_bar + delta_O * Otilde_c + delta_P * logPtilde_c, tau)

    The response is the localisation indicator of Eq. (8): h = 1[|f_hat - f| <= 1 Hz], one row per
    ARM rather than one per triplet, with IsLock marking the candidate against its two controls.
    A candidate can be lost two ways that an amplitude cannot separate - the forecast puts too
    little energy at the right frequency, or puts it somewhere else - and structural aliasing
    predicts the second, so the arm is scored by WHERE its reconstruction lands.

    The estimand is `gamma`, reported as e^gamma: the multiplicative change in the odds of
    rebuilding the right frequency at a candidate relative to its controls. e^gamma < 1 is what H1
    predicts. `beta_bar` is not an estimand but the baseline it is read against, the log-odds at a
    control of the average configuration; it gets a wider prior than the effects because a hit rate
    far from 0.5 is already several units of log-odds away from zero.

    `config_level` selects which covariates the configuration level carries, which is how M1 is
    decided, and the four fits below are compared by LOO:

        "both"     beta_bar + delta_O * Otilde + delta_P * logPtilde   (the reported model)
        "overlap"  beta_bar + delta_O * Otilde
        "patch"    beta_bar + delta_P * logPtilde
        "none"     beta_bar

    `link="probit"` swaps the link for the Part 5 robustness check. It replaces the Gaussian
    likelihood the contrast model used for that purpose, which a Bernoulli response has no
    counterpart for; the question it answers is the same, whether the verdict survives a change
    in the shape of the link at fixed data and priors.
    """
    cfg_c, cfg_l = _codes(df["model"])
    harm_c, harm_l = _codes(df["f_lock"].round(3).astype(str))
    bg_c, bg_l = _codes(df["generator"] + "#" + df["bg_id"].astype(str))
    O_t = _overlap_scaled(df, cfg_l)
    lP_t = _logP_centred(df, cfg_l)
    is_lock = df["is_lock"].to_numpy(float)
    y = df["h"].to_numpy(int)

    coords = {"config": cfg_l, "harmonic": harm_l, "background": bg_l, "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        #, population level: the estimand of H1 -------------------------------------
        gamma = pm.StudentT("gamma", nu=nu, mu=0.0, sigma=scale)
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=baseline_scale)
        cfg_mean = beta_bar
        if config_level in ("both", "overlap"):
            delta_O = pm.StudentT("delta_O", nu=nu, mu=0.0, sigma=scale)   # M1
            cfg_mean = cfg_mean + delta_O * O_t
        if config_level in ("both", "patch"):
            delta_P = pm.StudentT("delta_P", nu=nu, mu=0.0, sigma=scale)
            cfg_mean = cfg_mean + delta_P * lP_t

        #, hierarchy: configuration, lock harmonic, background realisation ----------
        # Centred group effects, with the harmonic and background offsets constrained to sum to
        # zero. Two decisions, for two separate pathologies.
        #
        # CENTRED is the form Eq. (eq:appA:model) of the appendix already prints, and it is the
        # right set of sampling coordinates for this data: with thousands of observations per
        # harmonic and hundreds per background the likelihood pins each offset, and the
        # non-centred substitution u = sigma_u * z then trades z against sigma_u along a ridge.
        #
        # SUM-TO-ZERO fixes what centring did not. In eta = beta[c] + gamma*IsLock + u_harm + u_bg
        # nothing pins the group means: add eps to every u_harm and subtract eps from every beta
        # and the likelihood is unchanged, so beta_bar and the two group means are identified only
        # jointly, by the priors, and the sampler drifts along that flat direction. A ridge, not a
        # funnel, which is why the symptom on the contrast model was R-hat far above 1 at
        # single-digit ESS with ZERO divergences. Constraining the two grouping factors leaves
        # beta as the only term carrying the level. `gamma` is untouched by any of this: it is a
        # within-triplet contrast, and the three arms of a triplet share their harmonic and their
        # background, so no group mean can absorb it.
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Normal("beta", mu=cfg_mean, sigma=tau, dims="config")

        sigma_h = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_h = pm.ZeroSumNormal("u_harm", sigma=sigma_h, dims="harmonic")
        sigma_b = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_b = pm.ZeroSumNormal("u_bg", sigma=sigma_b, dims="background")

        eta = beta[cfg_c] + gamma * is_lock + u_h[harm_c] + u_b[bg_c]
        if link == "probit":
            p = pm.Deterministic("p", 0.5 * (1.0 + pm.math.erf(eta / np.sqrt(2.0))), dims="obs")
            pm.Bernoulli("h", p=pm.math.clip(p, 1e-9, 1 - 1e-9), observed=y, dims="obs")
        else:
            pm.Bernoulli("h", logit_p=eta, observed=y, dims="obs")

        # reported on the natural scale: the multiplicative change in the odds at a lock
        pm.Deterministic("odds_ratio", pm.math.exp(gamma))
    return m


# --------------------------------------------------------------------------------------- #
def model_B_codelength(df: pd.DataFrame, adjusted: bool = True) -> pm.Model:
    """Model B, H1 representational.  Deliverable Eq. (10).

        y_i ~ Gamma(shape=k, mean=mu_i),   log mu_i = alpha_0 + theta_lock * IsLocked

    `adjusted=True` adds zero-mean stage and geometry offsets. Eq. (10) taken literally pools
    codelengths from probe stages whose scales differ several-fold, which inflates the residual
    without touching the estimand; blocking on stage is the standard remedy and leaves theta_lock
    exactly as specified. The two versions are compared by LOO in Part 4.
    """
    y = np.clip(df["L_bits"].to_numpy(float), 1e-3, None)    # Gamma support is strictly positive
    locked = df["is_locked"].to_numpy(float)
    st_c, st_l = _codes(df["stage"])
    mo_c, mo_l = _codes(df["model"])

    coords = {"stage": st_l, "geometry": mo_l, "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        alpha0 = pm.Normal("alpha_0", mu=float(np.log(np.mean(y))), sigma=1.0)
        theta = pm.Normal("theta_lock", mu=0.0, sigma=0.5)       # the estimand
        k = pm.Gamma("k", alpha=2.0, beta=0.1)                   # Gamma shape (dispersion)

        log_mu = alpha0 + theta * locked
        if adjusted:
            s_st = pm.HalfNormal("sigma_stage", 1.0)
            s_mo = pm.HalfNormal("sigma_geometry", 1.0)
            eff_st = pm.Deterministic("eff_stage", s_st * pm.Normal("z_stage", 0, 1, dims="stage"),
                                      dims="stage")
            eff_mo = pm.Deterministic("eff_geometry",
                                      s_mo * pm.Normal("z_geometry", 0, 1, dims="geometry"),
                                      dims="geometry")
            log_mu = log_mu + eff_st[st_c] + eff_mo[mo_c]

        mu = pm.math.exp(log_mu)
        pm.Gamma("y", alpha=k, beta=k / mu, observed=y, dims="obs")
        pm.Deterministic("codelength_ratio", pm.math.exp(theta))
    return m


# --------------------------------------------------------------------------------------- #
N_PHASE_BINS = 8       # how many equal slices of the phase circle get their own offset

def model_C_phase(df: pd.DataFrame, with_phase: bool = True, scale: float = PRIOR_SCALE,
                  baseline_scale: float = BASELINE_SCALE, nu: int = NU) -> pm.Model:
    """Model C, H2.  Does the localisation loss depend on WHERE in its cycle the signal starts?

        h_i ~ Bernoulli(p_i)
        logit p_i = beta[config] + gamma * IsLock + u_harm + u_bg + u_phase[bin of phi_i]
        u_phase ~ Normal(0, sigma_phase)

    Model C is Model A on the same response and the same rows, without the configuration-level
    covariates, which are not what H2 asks about. The phase circle is cut into N_PHASE_BINS equal
    slices and each slice gets its own offset; `sigma_phase` is the spread of those offsets, that
    is how much the odds of a correct reconstruction move as the signal slides through its cycle.
    H2 says the loss is a property of the stride/patch geometry, not of the signal's phase, i.e.
    sigma_phase ~ 0, read against log 1.1 in tab:bayesDecisions.

    Slices rather than a first-harmonic (a cos phi + b sin phi) term: the variance component is
    easier to state and to defend, it is the direct Bayesian counterpart of the phase spread the
    project's frequentist suite already reports, and it is strictly more general, catching ANY
    dependence on phase and not only a sinusoidal one.

    `with_phase=False` builds the nested null with no phase term, for the LOO comparison.
    """
    cfg_c, cfg_l = _codes(df["model"])
    harm_c, harm_l = _codes(df["f_lock"].round(3).astype(str))
    # the background realisation (generator + draw) is a grouping factor here exactly as in Model A:
    # without it, variation between TSMixup and KernelSynth backgrounds would be pushed into the
    # residual and could inflate or mask the phase term this model exists to measure
    bg_c, bg_l = _codes(df["generator"] + "#" + df["bg_id"].astype(str))
    # phase matters only modulo 2*pi: it is the tone's alignment with the patch grid
    phi = np.mod(df["phase"].to_numpy(float), 2 * np.pi)
    bin_c = np.floor(phi / (2 * np.pi / N_PHASE_BINS)).astype(int)
    bin_l = [f"{j * 360 // N_PHASE_BINS}-{(j + 1) * 360 // N_PHASE_BINS} deg" for j in range(N_PHASE_BINS)]
    is_lock = df["is_lock"].to_numpy(float)
    y = df["h"].to_numpy(int)

    coords = {"config": cfg_l, "harmonic": harm_l, "background": bg_l, "phasebin": bin_l,
              "obs": np.arange(len(y))}
    with pm.Model(coords=coords) as m:
        # Centred, and sum-to-zero on the two grouping factors, as in Model A and for the same
        # reasons. beta itself is left unconstrained, since beta_bar is defined as the population
        # mean the configuration effects are drawn around. u_phase below is deliberately neither
        # centred nor constrained: see the note there.
        gamma = pm.StudentT("gamma", nu=nu, mu=0.0, sigma=scale)
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=baseline_scale)
        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Normal("beta", mu=beta_bar, sigma=tau, dims="config")
        sigma_h = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_h = pm.ZeroSumNormal("u_harm", sigma=sigma_h, dims="harmonic")
        sigma_b = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_b = pm.ZeroSumNormal("u_bg", sigma=sigma_b, dims="background")

        eta = beta[cfg_c] + gamma * is_lock + u_h[harm_c] + u_b[bg_c]
        if with_phase:
            # the estimand: the spread of the per-phase offsets
            # The one group left non-centred, and the only one for which the appendix's original
            # argument holds: sigma_phase is the estimand of H2, and H2 predicts it near zero. A
            # group scale near zero is exactly the regime in which the centred form develops the
            # funnel that produces divergences, so the form is chosen per group.
            sigma_phase = pm.HalfNormal("sigma_phase", 0.25)
            u_p = pm.Deterministic("u_phase",
                                   sigma_phase * pm.Normal("z_phase", 0, 1, dims="phasebin"),
                                   dims="phasebin")
            eta = eta + u_p[bin_c]

        pm.Bernoulli("h", logit_p=eta, observed=y, dims="obs")
        pm.Deterministic("odds_ratio", pm.math.exp(gamma))
    return m


# --------------------------------------------------------------------------------------- #
COMB_FLOOR = 1e-2      # z_norm floor before the log; see the note in Ãƒâ€šSection 4.4
GRID_TOL_HZ = 1.0      # how close to a predicted site counts as "on the grid"

def model_D1_sites(df: pd.DataFrame, grid: str) -> pm.Model:
    """Model D1, H3 location.  Deliverable Eq. (12).

        log z_g(f) ~ Normal(alpha_g + theta_S 1[f on stride grid] + theta_P 1[f on patch grid], sigma)

    Each frequency is labelled by which predicted grid it falls on (within GRID_TOL_HZ), and the
    model asks whether the token dispersion is systematically lower there. H3 predicts theta_S < 0.

    `grid` selects the labelling: "both" keeps both indicators (the model H3 states), "stride" only
    c*fs/S, "patch" only k*fs/P, "none" neither. The four are fitted on identical observations and
    compared by LOO,
    which is what identifies WHICH parameter generates the sites, the S<P configurations are the
    ones that separate the two families, since on the P=S diagonal the grids coincide.

    This is deliberately the same shape of model as Eq. (10): a linear predictor on a log scale
    with an indicator variable. It replaced an earlier Gaussian-dip "comb" likelihood with free
    depth and width, which estimated two nuisance quantities nothing downstream used and was far
    harder to justify than the claim it was testing.
    """
    g_c, g_l = _codes(df["model"])
    logz = np.log(np.clip(df["z_norm"].to_numpy(float), COMB_FLOOR, None))

    # membership indicators, built from the same helper that defines the grids everywhere else
    on_stride = np.zeros(len(df))
    on_patch = np.zeros(len(df))
    for gi, _name in enumerate(g_l):
        sel = g_c == gi
        P = int(df.loc[sel, "P"].iloc[0]); S = int(df.loc[sel, "S"].iloc[0])
        f = df.loc[sel, "f"].to_numpy(float)
        on_stride[sel] = (pl.comb_distance(f, pl.FS / S) <= GRID_TOL_HZ).astype(float)
        on_patch[sel] = (pl.comb_distance(f, pl.FS / P) <= GRID_TOL_HZ).astype(float)

    coords = {"geometry": g_l, "obs": np.arange(len(logz))}
    with pm.Model(coords=coords) as m:
        alpha_g = pm.Normal("alpha_g", 0.0, 1.0, dims="geometry")   # off-grid level per geometry
        sigma = pm.HalfNormal("sigma", 1.0)
        mu = alpha_g[g_c]
        if grid in ("stride", "both"):
            theta_S = pm.Normal("theta_S", 0.0, 1.0)
            mu = mu + theta_S * on_stride
        if grid in ("patch", "both"):
            theta_P = pm.Normal("theta_P", 0.0, 1.0)
            mu = mu + theta_P * on_patch
        pm.Normal("logz", mu=mu, sigma=sigma, observed=logz, dims="obs")
    return m


# --------------------------------------------------------------------------------------- #
def model_D2_movement(df: pd.DataFrame, branch: str, resolution_hz: float | None = None,
                      response: str = "f1") -> pm.Model:
    """Models D2, H3a and H3b.  One scaling law per branch, no intercept.

        f1_hat[g] ~ Normal(kappa_F * Delta_F[g], sqrt(sigma_F^2 + df^2))

    Deliverable 3, H3: every detected dip is first assigned to the branch that predicts it and
    ambiguous sites are set aside, then the fundamental of that branch is compared with the spacing
    that branch predicts, Delta_S = fs/S or Delta_P = fs/P. H3a predicts kappa_S = 1 and H3b
    predicts kappa_P = 1: each branch tracks its own parameter one for one.

    This replaced a single regression of one pooled spacing on BOTH predictors, with an intercept
    and the prediction kappa_P = 0. That formulation contradicted H3 as approved in the approved Deliverable 3 formulation,
    which claims that each branch moves with its own parameter; and it treated the detected set as
    one comb, when F_lock is a union and the union of two combs has two interlaced spacings. The
    dip that formulation had to call spurious at p16-s8 is a patch null at fs/P = 32 Hz, i.e. an
    observation H3 predicts.

    **Measurement resolution.** The spacing is read off a frequency sweep, so it carries a floor of
    about one grid step whatever the sampling noise. Stating it is not cosmetic: when the sites land
    on the predicted comb exactly, the residuals vanish, sigma is pushed to zero and the posterior
    develops a funnel NUTS cannot traverse (R-hat above 2, hundreds of divergences). Adding the
    floor in quadrature removes the pathology and is the honest statement: a near-zero sigma is then
    a result, the spacings follow the law to within the sweep resolution, not a sampler failure.

    `response="f1"` uses the branch's fundamental, the lowest detected site of that branch, which is
    what the deliverable specifies. `response="delta_hat"` uses the median gap of the same branch and
    is fitted only as a robustness check: with three or four sites in band one spurious detection
    inserts a short interval and flips the median, to which the fundamental is immune.
    """
    sub = df[df["branch"] == branch]
    y = sub[response].to_numpy(float)                      # measured spacing [Hz]
    x = sub["predicted_spacing"].to_numpy(float)           # fs/S or fs/P [Hz]
    name = {"stride": "kappa_S", "patch": "kappa_P"}[branch]

    step = resolution_hz if resolution_hz is not None else CFG.collapse_step
    with pm.Model(coords={"obs": np.arange(len(y))}) as m:
        kappa = pm.Normal(name, mu=0.0, sigma=1.0)         # H3a / H3b predict 1
        sigma_d = pm.HalfNormal("sigma_extra", 5.0)        # scatter beyond the grid floor [Hz]
        sigma_total = pm.math.sqrt(sigma_d ** 2 + step ** 2)
        pm.Normal("f1_hat", mu=kappa * x, sigma=sigma_total, observed=y, dims="obs")
    return m


def sample(model, name: str, draws=None, tune=None, chains=None,
           idata_kwargs=None, **kw):
    """One place for the sampler settings the deliverable's Inference paragraph prescribes.

    `NUTS_BACKEND` (set in Part 0.3) selects who does the sampling. The model, the priors and the
    number of draws are identical in every case; only the implementation of NUTS changes, so the
    posterior is the same target distribution.

      "pymc"     the default. Pure Python/PyTensor, CPU only.
      "nutpie"   a Rust implementation, several times faster on the same CPU.
    `idata_kwargs` overrides the default {"log_likelihood": True}; it exists so a fit nothing
    computes LOO on can skip building that array (see sample_without_log_likelihood below).
    Every other call is unchanged, and the default is the one it always had.

      "numpyro"  JAX. This is the one that uses a GPU, and the only reason a GPU runtime helps:
                 PyMC's own sampler never touches it. Chains run vectorised rather than in series.
    """
    extra = dict(kw)
    if NUTS_BACKEND != "pymc":
        extra["nuts_sampler"] = NUTS_BACKEND
        if NUTS_BACKEND == "numpyro":
            extra.setdefault("nuts_sampler_kwargs", {"chain_method": "vectorized"})
    target_accept = extra.pop("target_accept", TARGET_ACCEPT)
    with model:
        return pm.sample(draws=draws or DRAWS, tune=tune or TUNE, chains=chains or CHAINS,
                         cores=1, random_seed=SEED, target_accept=target_accept,
                         progressbar=False,
                         idata_kwargs=idata_kwargs or {"log_likelihood": True}, **extra)

print("model factories ready: A (H1 behavioural + M1), B (H1 representational), "
      "C (H2), D1 (H3 location), D2 (H3a stride branch, H3b patch branch)")

In [ ]:

# =====================================================================================
#  Sampling policy
# =====================================================================================
# The cell above defines the project's sample(); this wraps it so that nothing gets fitted
# by accident. Only the prior-sensitivity ladder is missing from CKPT_DIR, and only that is
# allowed through - anything else asking to sample means a checkpoint is absent, which should
# surface as a named failure rather than as a multi-hour fit starting unattended.
_sample_impl = sample

def sample(model, name: str, *args, **kwargs):
    if not ALLOW_SAMPLING:
        raise RuntimeError(
            f"sampling is disabled (ALLOW_SAMPLING = False) and '{name}' has no checkpoint in "
            f"{CKPT_DIR}, and ALLOW_SAMPLING is False. Set it to True to fit it here.")
    return _sample_impl(model, name, *args, **kwargs)


def sample_without_log_likelihood(model, name: str, **kwargs):
    """sample(), with PyMC's post-hoc log_likelihood computation switched off.

    Only for fits nothing computes LOO on. The prior-sensitivity ladder is the whole of that
    category: Part 5.3 reads beta_bar, delta_O and the convergence diagnostics, all of which
    live in posterior and sample_stats, and no comparison is ever run against these fits.

    It matters twice over. log_likelihood is one float per draw, per chain, per observation,
    so for Model A's ~138k rows it is an 8.4 GiB array that PyMC allocates *after* sampling
    finishes, when memory is already at its peak - that transient is what took down a session
    fitting Model C fresh. It is also what makes each existing checkpoint 8.4 GiB on Drive;
    a ladder variant saved this way is a few MB. The posterior is bit-identical either way:
    log_likelihood is computed from the draws, it does not influence them.
    """
    return _sample_impl(model, name, idata_kwargs={"log_likelihood": False}, **kwargs)

print("sampling policy:", "ladder only" if ALLOW_SAMPLING else "disabled (analysis only)")


## 3.3 - Synthetic parameter recovery for every likelihood family

Before empirical fitting, A, B, C, D1 and both D2 branches are exercised on known effects using
the real design axes. Recovery uses four chains and is a fail-closed method gate. These synthetic
numbers are **NON-REPORTABLE** and cannot be cited as evidence about Chronos.

In [ ]:
RECOVERY_VERSION = "d3-bernoulli-k3-v1"
RECOVERY_DRAWS = 100 if IS_SMOKE else 5000
RECOVERY_TUNE = 100 if IS_SMOKE else 1500
RECOVERY_FILE = "03_recovery_bernoulli_k3_v1.parquet"

def _zero_sum(effects: np.ndarray) -> np.ndarray:
    """Centre a draw of group offsets, so the simulator matches the prior it is recovered under.

    `u_harm` and `u_bg` are ZeroSumNormal in Models A and C. An iid draw has a group mean of
    order sigma_u/sqrt(n), which under the constrained model has nowhere to go except into the
    intercept: at 31 harmonics and 6 backgrounds that is a systematic ~0.04 shift in beta_bar,
    small against its interval but a bias in the recovery test rather than in the estimator.
    """
    return effects - effects.mean()


def _recovery_rows(idata, model_name: str, truths: dict[str, float]) -> list[dict]:
    diagnostics = bc.fit_diagnostics(f"recovery:{model_name}", idata, az)
    rows = []
    for parameter, truth in truths.items():
        posterior = idata.posterior[parameter].values.ravel()
        low, high = np.quantile(posterior, [0.025, 0.975])
        rows.append({
            "recovery_version": RECOVERY_VERSION,
            "model": model_name, "parameter": parameter, "truth": truth,
            "median": float(np.median(posterior)), "hdi_low": float(low),
            "hdi_high": float(high), "covered": bool(low <= truth <= high),
            "diagnostics_ok": diagnostics["diagnostics_ok"],
            "max_rhat": diagnostics["max_rhat"],
            "min_ess_bulk": diagnostics["min_ess_bulk"],
            "min_ess_tail": diagnostics["min_ess_tail"],
            "divergences": diagnostics["divergences"],
        })
    return rows

def _sample_recovery(model, label: str, draws=None, tune=None, target_accept=0.97):
    return sample(model, f"recovery:{label}", draws=draws or RECOVERY_DRAWS,
                  tune=tune or RECOVERY_TUNE, chains=CHAINS, target_accept=target_accept)

def simulate_A(df: pd.DataFrame, seed: int = SEED) -> tuple[pd.DataFrame, dict]:
    """Draw h from Model A's own generative process, on the real design axes.

    The baseline log-odds is -0.40, a control hit rate of 0.40. That figure is measured, not
    guessed: running the collection with a PERFECT forecaster - one that returns the true
    continuation verbatim - scores h = 1 on only 40% of arms at TONE_SNR = 1.25. The rest are
    arms where the corpus's own energy at some other frequency exceeds the tone's, so the argmax
    lands elsewhere. That 0.40 is the ceiling of the measurement, and the fraction of it Chronos
    reaches is what Model A is reading. On a flat background the same estimator scores 0.87; the
    gap is the spectral shape of TSMixup and KernelSynth, which put most of their energy low.

    A baseline near 0.5 is where a Bernoulli carries the most information per row, so the design
    is well placed for power even though the ceiling is far from 1. Truth gamma = -0.60 is an odds
    ratio of 0.55 at a candidate, comfortably past the log 0.8 support threshold, so a recovery
    that misses it is a failure of the method and not of the design.
    """
    rng = np.random.default_rng(seed)
    out = df[df["bg_id"] < 3].copy().reset_index(drop=True)
    config_codes, config_levels = _codes(out["model"])
    harmonic_codes, harmonic_levels = _codes(out["f_lock"].round(3).astype(str))
    background_codes, background_levels = _codes(
        out["generator"] + "#" + out["bg_id"].astype(str))
    baseline, gamma, delta_overlap, delta_patch = -0.40, -0.60, 0.25, -0.15
    overlap = _overlap_scaled(out, config_levels)
    log_patch = _logP_centred(out, config_levels)
    eta = (baseline + gamma * out["is_lock"].to_numpy(float)
           + delta_overlap * overlap[config_codes] + delta_patch * log_patch[config_codes]
           + rng.normal(0, .10, len(config_levels))[config_codes]
           + _zero_sum(rng.normal(0, .12, len(harmonic_levels)))[harmonic_codes]
           + _zero_sum(rng.normal(0, .08, len(background_levels)))[background_codes])
    out["h"] = rng.binomial(1, 1.0 / (1.0 + np.exp(-eta))).astype(np.int8)
    return out, {"gamma": gamma, "delta_O": delta_overlap, "delta_P": delta_patch}


def simulate_B(df: pd.DataFrame, seed: int = SEED + 1) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    out = df.copy().reset_index(drop=True)
    stage_codes, stages = _codes(out["stage"])
    geometry_codes, geometries = _codes(out["model"])
    theta = 0.30
    alpha = np.log(max(float(out["L_bits"].mean()), 1.0))
    log_mu = (alpha + theta * out["is_locked"].to_numpy(float)
              + rng.normal(0, .20, len(stages))[stage_codes]
              + rng.normal(0, .12, len(geometries))[geometry_codes])
    shape = 20.0
    mean = np.exp(log_mu)
    out["L_bits"] = rng.gamma(shape, mean / shape)
    return out, {"theta_lock": theta}

def simulate_C(df: pd.DataFrame, seed: int = SEED + 2) -> tuple[pd.DataFrame, dict]:
    """Draw h from Model C's own generative process, all three grouping factors included.

    sigma_phase = 0.08 on the log-odds scale is a deliberately small truth, and a binary response
    carries far less information per row than the contrast did: a per-bin offset is estimable to
    about sqrt(1/(n*p*(1-p))) log-odds, so even at the most favourable baseline, p near 0.4,
    recovering a spread this size needs thousands of rows per bin. If this fit misses, read it as a statement about the power of
    the design for H2 before reading it as a sampler failure.
    """
    rng = np.random.default_rng(seed)
    out = df[df["bg_id"] < 3].copy().reset_index(drop=True)
    config_codes, config_levels = _codes(out["model"])
    harmonic_codes, harmonic_levels = _codes(out["f_lock"].round(3).astype(str))
    background_codes, background_levels = _codes(
        out["generator"] + "#" + out["bg_id"].astype(str))
    bins = np.floor(np.mod(out["phase"], 2 * np.pi) /
                    (2 * np.pi / N_PHASE_BINS)).astype(int)
    sigma_phase = 0.08
    offsets = np.linspace(-1, 1, N_PHASE_BINS)
    offsets = sigma_phase * (offsets - offsets.mean()) / offsets.std()
    eta = (-0.40 - 0.60 * out["is_lock"].to_numpy(float)
           + rng.normal(0, .10, len(config_levels))[config_codes]
           + _zero_sum(rng.normal(0, .12, len(harmonic_levels)))[harmonic_codes]
           + _zero_sum(rng.normal(0, .08, len(background_levels)))[background_codes]
           + offsets[bins])
    out["h"] = rng.binomial(1, 1.0 / (1.0 + np.exp(-eta))).astype(np.int8)
    return out, {"sigma_phase": sigma_phase}


def simulate_D1(df: pd.DataFrame, seed: int = SEED + 3) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    generated_mode = next(mode for mode in CFG.generators if mode in set(df["mode"]))
    out = df[(df["mode"] == generated_mode) & (df["rep"] == 0)].copy().reset_index(drop=True)
    geometry_codes, geometries = _codes(out["model"])
    on_stride = np.zeros(len(out))
    on_patch = np.zeros(len(out))
    for code, _ in enumerate(geometries):
        selected = geometry_codes == code
        P = int(out.loc[selected, "P"].iloc[0])
        S = int(out.loc[selected, "S"].iloc[0])
        frequencies = out.loc[selected, "f"].to_numpy(float)
        on_stride[selected] = pl.comb_distance(frequencies, pl.FS / S) <= GRID_TOL_HZ
        on_patch[selected] = pl.comb_distance(frequencies, pl.FS / P) <= GRID_TOL_HZ
    theta_stride, theta_patch = -0.65, -0.45
    log_z = (rng.normal(0, .10, len(geometries))[geometry_codes]
             + theta_stride * on_stride + theta_patch * on_patch + rng.normal(0, .30, len(out)))
    out["z_norm"] = np.exp(log_z)
    return out, {"theta_S": theta_stride, "theta_P": theta_patch}

def simulate_D2(branch: str, seed: int) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    kappa = 0.95
    rows = []
    for P, S in FULL_MODELS:
        predicted = pl.FS / (S if branch == "stride" else P)
        for mode in CFG.generators:
            for replicate in range(3):
                measured = rng.normal(kappa * predicted,
                                      np.sqrt(0.5 ** 2 + CFG.collapse_step ** 2))
                rows.append({
                    "model": pl.model_tag(P, S), "P": P, "S": S, "mode": mode,
                    "rep": replicate, "branch": branch, "predicted_spacing": predicted,
                    "f1": measured, "delta_hat": measured, "n_sites": 3,
                })
    name = "kappa_S" if branch == "stride" else "kappa_P"
    return pd.DataFrame(rows), {name: kappa}

# Recovery validates the method for the models this session actually fits. Exercising C, D1 and
# D2 while only A is selected would cost more sampling than the fit it is meant to license, and
# Part 5.5 gates every verdict on recovery_ok(<model>) anyway, so a model that is neither fitted
# nor recovered simply never reaches the table.
# A and C are recovered unconditionally, whatever the fit flags say. They are the two Bernoulli
# builders, they have never been sampled by PyMC, and the failure that ended the previous
# iteration was R-hat 1.52 at ESS 7 with ZERO divergences - a sampler stuck on a ridge, not a
# misspecified likelihood. A Bernoulli carries LESS information per row than a Student-t, so the
# new response is not automatically safer. This costs no Chronos and no collection: if the three
# gates fail here they will fail on the real data, and finding out now is free.
RECOVERY_MEMBERS = {"A": True, "B": FIT_MODEL_B, "C": True,
                    "D1": FIT_MODEL_D1, "D2-stride": FIT_MODEL_D2, "D2-patch": FIT_MODEL_D2}
required_recovery_models = {name for name, on in RECOVERY_MEMBERS.items() if on}
if not required_recovery_models:
    raise RuntimeError("no model selected; set at least one FIT_MODEL_* flag in Part 0.3b")

if have(RECOVERY_FILE):
    recovery_tbl = load_df(RECOVERY_FILE)
else:
    recovery_rows = []
    if RECOVERY_MEMBERS["A"]:
        sim_A, truth_A = simulate_A(arms)
        recovery_rows += _recovery_rows(
            _sample_recovery(model_A_contrast(sim_A), "A", target_accept=0.99, draws=2500),
            "A", truth_A)
    if FIT_MODEL_B:
        sim_B, truth_B = simulate_B(mdl_cells)
        recovery_rows += _recovery_rows(
            _sample_recovery(model_B_codelength(sim_B, adjusted=True), "B"), "B", truth_B)
    if RECOVERY_MEMBERS["C"]:
        sim_C, truth_C = simulate_C(arms)
        recovery_rows += _recovery_rows(
            _sample_recovery(model_C_phase(sim_C, with_phase=True), "C", draws=16000, tune=2000,
                             target_accept=0.99),
            "C", truth_C)
    if FIT_MODEL_D1:
        sim_D1, truth_D1 = simulate_D1(collapse)
        recovery_rows += _recovery_rows(
            _sample_recovery(model_D1_sites(sim_D1, "both"), "D1"), "D1", truth_D1)
    if FIT_MODEL_D2:
        for branch, model_name, seed in (
            ("stride", "D2-stride", SEED + 4), ("patch", "D2-patch", SEED + 5)
        ):
            sim_D2, truth_D2 = simulate_D2(branch, seed)
            recovery_rows += _recovery_rows(
                _sample_recovery(model_D2_movement(sim_D2, branch), model_name),
                model_name, truth_D2)
    recovery_tbl = save_df(pd.DataFrame(recovery_rows), RECOVERY_FILE)

if set(recovery_tbl.get("recovery_version", [])) != {RECOVERY_VERSION}:
    raise ValueError("stale/incompatible parameter-recovery checkpoint")
RECOVERY_OK = (bc.recovery_gate(recovery_tbl, required_recovery_models)
               and recovery_tbl["diagnostics_ok"].all())
display(recovery_tbl.round(4))
print("SYNTHETIC METHOD VALIDATION ONLY - NON-REPORTABLE")
if RECOVERY_OK:
    print("parameter recovery gate: PASS for", ", ".join(sorted(required_recovery_models)))
else:
    # Reported per model, not all-or-nothing. The appendix's rule is that a fit failing a
    # diagnostic "is not used to support a claim, although it may still be reported", and Part 5.5
    # enforces precisely that: every verdict carries its own recovery_ok(<model>) term, so a model
    # failing here arrives at the table as NOT REPORTABLE while the others are untouched.
    _bad = sorted(recovery_tbl.loc[~(recovery_tbl["diagnostics_ok"].astype(bool)
                                     & recovery_tbl["covered"].astype(bool)), "model"].unique())
    print(f"parameter recovery gate: FAILED for {_bad}; their hypotheses are marked NOT "
          f"REPORTABLE by recovery_ok() in Part 5.5. Fitting proceeds for the rest.")

---
# Part 4, Posterior inference

Priors are fixed, the data are in, the likelihood is validated. This part fits the five models and
reports, for each, what the deliverable's *Inference* paragraph asks for: the effect on its natural
scale with a 95% credible interval, the posterior probability of the preregistered threshold, and,
where two formulations compete, a leave-one-out comparison rather than a null-hypothesis test.

Every fit is checkpointed individually, so a disconnect costs at most one model.

In [ ]:
import gc, json, math, shutil, time
import numpy as np
import xarray as xr

banner("PART 4 - POSTERIOR INFERENCE (streaming, analysis only)")

# =====================================================================================
#  Why this file exists
# =====================================================================================
# Every fit is already on disk. What exhausted the 12 GB Colab runtime was never the
# sampling: it was that computing LOO required log_likelihood - one float per posterior
# draw, per chain, per observation - to be RESIDENT. For Model A that array is
# 4 chains x 2000 draws x ~1.4e5 observations x 8 bytes ~= 9 GB, and the previous
# "chunked" loader still built the whole thing in RAM (np.empty(shape, float32)) before
# chunking anything, which is why the very first checkpoint load saturated the runtime.
#
# PSIS-LOO is computed independently per observation, so log_likelihood never has to be
# resident at all. Everything below reads it one observation-slice at a time straight off
# the netCDF file and keeps only the three per-observation numbers LOO actually needs
# (elpd_i, pareto_k, lppd_i). Peak RAM for a Model A LOO becomes a few hundred MB instead
# of ~9 GB, and no posterior, prior, decision rule or threshold changes.

LOO_TARGET_BYTES = 256 * 1024**2   # RAM budget for one observation-slice
STAGE_LOCALLY = True               # copy a checkpoint to local disk before streaming it

# -------------------------------------------------------------------------------------
#  Exact ports of arviz.stats.stats._gpdfit / _gpinv / _psislw (arviz 0.23.4).
#  Ported rather than imported because Colab resolves arviz 1.x, where these live in
#  arviz_stats under a different API; the numbers are arviz's own either way. Verified
#  against az.loo(idata, pointwise=True) to a difference of exactly 0.0 on elpd_loo, se,
#  p_loo, every elpd_i and every pareto_k.
# -------------------------------------------------------------------------------------
def _lse(a):
    a_max = np.max(a)
    if not np.isfinite(a_max):
        a_max = 0.0
    return float(np.log(np.sum(np.exp(a - a_max))) + a_max)


def _gpdfit(ary):
    prior_bs, prior_k = 3, 10
    n = len(ary)
    m_est = 30 + int(n**0.5)
    b_ary = 1 - np.sqrt(m_est / (np.arange(1, m_est + 1, dtype=float) - 0.5))
    b_ary /= prior_bs * ary[int(n / 4 + 0.5) - 1]
    b_ary += 1 / ary[-1]
    k_ary = np.log1p(-b_ary[:, None] * ary).mean(axis=1)
    len_scale = n * (np.log(-(b_ary / k_ary)) - k_ary - 1)
    weights = 1 / np.exp(len_scale - len_scale[:, None]).sum(axis=1)
    real_idxs = weights >= 10 * np.finfo(float).eps
    if not np.all(real_idxs):
        weights = weights[real_idxs]
        b_ary = b_ary[real_idxs]
    weights /= weights.sum()
    b_post = np.sum(b_ary * weights)
    k_post = np.log1p(-b_post * ary).mean()
    sigma = -k_post / b_post
    k_post = (n * k_post + prior_k * 0.5) / (n + prior_k)
    return k_post, sigma


def _gpinv(probs, kappa, sigma):
    x = np.full_like(probs, np.nan)
    if sigma <= 0:
        return x
    ok = (probs > 0) & (probs < 1)
    if np.all(ok):
        if np.abs(kappa) < np.finfo(float).eps:
            x = -np.log1p(-probs)
        else:
            x = np.expm1(-kappa * np.log1p(-probs)) / kappa
        x *= sigma
    else:
        if np.abs(kappa) < np.finfo(float).eps:
            x[ok] = -np.log1p(-probs[ok])
        else:
            x[ok] = np.expm1(-kappa * np.log1p(-probs[ok])) / kappa
        x *= sigma
        x[probs == 0] = 0
        x[probs == 1] = np.inf if kappa >= 0 else -sigma / kappa
    return x


def _psislw_1d(log_weights, cutoff_ind, cutoffmin):
    x = np.asarray(log_weights, dtype=float)
    max_x = np.max(x)
    x = x - max_x
    x_sort_ind = np.argsort(x)
    xcutoff = max(x[x_sort_ind[cutoff_ind]], cutoffmin)
    expxcutoff = np.exp(xcutoff)
    (tailinds,) = np.where(x > xcutoff)
    x_tail = x[tailinds]
    tail_len = len(x_tail)
    if tail_len <= 4:
        k = np.inf
    else:
        x_tail_si = np.argsort(x_tail)
        x_tail = np.exp(x_tail) - expxcutoff
        k, sigma = _gpdfit(x_tail[x_tail_si])
        if np.isfinite(k):
            sti = np.arange(0.5, tail_len) / tail_len
            x[tailinds[x_tail_si]] = np.log(_gpinv(sti, k, sigma) + expxcutoff)
            x[x > 0] = 0
    x -= _lse(x)
    return x, k


def _psis_block(block, cutoff_ind, cutoffmin):
    """One observation-slice. `block` is [n_obs, n_samples]; returns elpd_i, k, lppd_i."""
    n_obs, n_samples = block.shape
    log_n = np.log(n_samples)
    elpd_i = np.empty(n_obs)
    pareto_k = np.empty(n_obs)
    lppd_i = np.empty(n_obs)
    for i in range(n_obs):
        ll = block[i]
        lw, k = _psislw_1d(-ll, cutoff_ind, cutoffmin)
        elpd_i[i] = _lse(lw + ll)
        pareto_k[i] = k
        lppd_i[i] = _lse(ll) - log_n
    return elpd_i, pareto_k, lppd_i


# -------------------------------------------------------------------------------------
#  Reading a checkpoint without its log_likelihood
# -------------------------------------------------------------------------------------
_LL = "log_likelihood"


def _group_names(path):
    """Group names in a netCDF4/HDF5 checkpoint, without reading any of their data."""
    try:
        import h5py
        with h5py.File(path, "r") as handle:
            return [k for k in handle.keys() if isinstance(handle[k], h5py.Group)]
    except Exception:
        pass
    try:
        import netCDF4
        with netCDF4.Dataset(path) as root:
            return list(root.groups.keys())
    except Exception:
        return None


def _idata_from_groups(groups):
    """Assemble an InferenceData from a dict of already-loaded xarray Datasets.

    arviz 1.x's from_dict takes group Datasets directly; arviz 0.x's expects raw arrays and
    rejects a Dataset, so its InferenceData(**groups) constructor is used instead. Both are
    tried so the notebook runs under either resolution of the locked environment.
    """
    try:
        return az.InferenceData(**groups)
    except Exception:
        return az.from_dict(groups)


def load_posterior_only(name, extra_groups=()):
    """Reload every group of a checkpoint EXCEPT log_likelihood.

    report(), prob(), the forest/phase plots and the convergence gate read posterior and
    sample_stats only, so log_likelihood - the 9 GB majority of the file - is never opened.
    """
    if not have(name):
        raise FileNotFoundError(name)
    path = ckpt(name)
    names = _group_names(path)
    if names is None:
        names = ["posterior", "sample_stats", "observed_data", "constant_data"]
    wanted = [g for g in names if g != _LL]
    groups = {}
    for g in wanted:
        try:
            groups[g] = xr.open_dataset(path, group=g).load()
        except (OSError, IOError, ValueError, KeyError):
            pass
    idata = _idata_from_groups(groups)
    print(f"  checkpoint <- {name} (posterior only, log_likelihood not opened)")
    return idata


def posterior_or_fit(name, build):
    """The checkpoint's posterior, fitting it first if it is not on disk.

    The analysis reads posteriors and streams log_likelihood off disk, so it never wants a fit
    resident in full. But a run against a fresh RUN_ID has nothing to read yet, and splitting the
    work across two notebooks would mean sampling in one and streaming in the other. This keeps
    both in one place: a missing checkpoint is sampled and written exactly as the fitting notebook
    would write it, and everything downstream then reads it the cheap way.
    """
    if not have(name):
        if not ALLOW_SAMPLING:
            raise FileNotFoundError(
                f"{name} is not in {CKPT_DIR} and ALLOW_SAMPLING is False.")
        print(f"\nfitting {name}")
        save_idata(build(), name)
        gc.collect()
    return load_posterior_only(name)


def _reff(idata):
    """Relative ESS, exactly as az.loo computes it when reff is not supplied."""
    posterior = idata.posterior
    n_chains = len(posterior.chain)
    n_samples = n_chains * len(posterior.draw)
    if n_chains == 1:
        return 1.0
    ess_p = az.ess(posterior, method="mean")
    return float(np.hstack([ess_p[v].values.flatten() for v in ess_p.data_vars]).mean() / n_samples)


def _stage(path):
    """Copy a Drive checkpoint to local disk before streaming it.

    Streaming reads slices along the observation axis, which is the fastest-varying one, so
    every slice is a strided read. Over Drive's FUSE mount that is punishingly slow; the same
    access pattern on Colab's local disk is not. The copy itself is one sequential read.
    """
    if not STAGE_LOCALLY:
        return path, False
    try:
        local_root = Path("/content/_loo_stage")
        if not Path("/content").exists():
            return path, False
        local_root.mkdir(parents=True, exist_ok=True)
        need = path.stat().st_size
        free = shutil.disk_usage(local_root).free
        if free < need * 1.15:
            print(f"    (not staging: needs {need/2**30:.1f} GiB, {free/2**30:.1f} GiB free)")
            return path, False
        target = local_root / path.name
        if not (target.exists() and target.stat().st_size == need):
            t0 = time.time()
            shutil.copyfile(path, target)
            print(f"    staged {need/2**30:.2f} GiB to local disk in {time.time()-t0:.0f}s")
        return target, True
    except Exception as exc:
        print(f"    (staging unavailable: {exc!r})")
        return path, False


def stream_loo(name, idata_post, var_name=None, target_bytes=LOO_TARGET_BYTES):
    """PSIS-LOO for a checkpoint, reading log_likelihood one observation-slice at a time.

    Returns the same quantities az.loo(idata, pointwise=True) returns - elpd_loo, se, p_loo,
    per-observation elpd_i and pareto_k, warning and good_k - without log_likelihood ever
    being resident. The group is opened lazily through xarray, so `.isel(...).values` reads
    only the slice asked for; this works on whichever netCDF backend is installed (the locked
    environment ships h5netcdf, not netCDF4).
    """
    path = ckpt(name)
    read_path, staged = _stage(path)
    reff = _reff(idata_post)

    try:
        with xr.open_dataset(read_path, group=_LL) as group:
            this_var = var_name or list(group.data_vars)[0]
            array = group[this_var]
            obs_dims = [d for d in array.dims if d not in ("chain", "draw")]
            if len(obs_dims) != 1:
                raise NotImplementedError(f"{name}: expected one observation dim, got {array.dims}")
            obs_dim = obs_dims[0]
            n_chain = int(array.sizes["chain"])
            n_draw = int(array.sizes["draw"])
            n_obs = int(array.sizes[obs_dim])
            n_samples = n_chain * n_draw

            cutoff_ind = -int(np.ceil(min(n_samples / 5.0, 3 * (n_samples / reff) ** 0.5))) - 1
            cutoffmin = np.log(np.finfo(float).tiny)

            per_obs = n_samples * 8
            chunk = max(1, min(n_obs, int(target_bytes // per_obs)))
            n_chunks = math.ceil(n_obs / chunk)
            print(f"  LOO <- {name}: {n_obs:,} obs x {n_samples:,} draws, "
                  f"{n_chunks} slices of <= {chunk:,} obs "
                  f"(~{chunk * per_obs / 2**20:.0f} MiB resident)")

            elpd_parts, k_parts, lppd_parts = [], [], []
            t0 = time.time()
            for start in range(0, n_obs, chunk):
                stop = min(start + chunk, n_obs)
                block = array.isel({obs_dim: slice(start, stop)})
                # arviz stacks __sample__ = (chain, draw), so the sample index must run
                # chain-major; transposing to (chain, draw, obs) before the reshape gives that.
                values = np.asarray(
                    block.transpose("chain", "draw", obs_dim).values, dtype=np.float64)
                flat = values.reshape(n_samples, stop - start).T
                del block, values
                e, k, l = _psis_block(flat, cutoff_ind, cutoffmin)
                del flat
                elpd_parts.append(e); k_parts.append(k); lppd_parts.append(l)
                gc.collect()
            print(f"     {time.time() - t0:.0f}s")
    finally:
        if staged:
            try:
                Path(read_path).unlink()
            except OSError:
                pass

    elpd_i = np.concatenate(elpd_parts)
    pareto_k = np.concatenate(k_parts)
    lppd = float(np.concatenate(lppd_parts).sum())
    elpd_loo = float(elpd_i.sum())
    se = float((n_obs * np.var(elpd_i)) ** 0.5)
    p_loo = lppd - elpd_loo
    good_k = float(min(1 - 1 / np.log10(n_samples), 0.7))
    warning = bool(np.any(pareto_k > good_k))
    return dict(elpd_loo=elpd_loo, se=se, p_loo=p_loo, n_samples=int(n_samples),
                n_obs=int(n_obs), elpd_i=elpd_i, pareto_k=pareto_k,
                warning=warning, good_k=good_k)


def _pointwise_names(name):
    stem = name[:-3] if name.endswith(".nc") else name
    return f"loo_pointwise__{stem}.parquet", f"loo_scalars__{stem}.json"


def stream_loo_cached(name, idata_post=None):
    """stream_loo, with its per-observation result checkpointed.

    elpd_i and pareto_k are one float per observation - a couple of MB against a 9 GB
    checkpoint - so persisting them makes each fit's LOO individually resumable. A session
    that dies part-way through a comparison then resumes at the fit it died on instead of
    restarting the whole group, and Part 6's Pareto-k plots read these arrays rather than
    recomputing LOO on a posterior-only InferenceData that has no log_likelihood to use.
    """
    pw_name, sc_name = _pointwise_names(name)
    if have(pw_name) and have(sc_name):
        frame = load_df(pw_name)
        stream = dict(load_json(sc_name))
        stream["elpd_i"] = frame["elpd_i"].to_numpy(float)
        stream["pareto_k"] = frame["pareto_k"].to_numpy(float)
        return stream
    post = idata_post if idata_post is not None else load_posterior_only(name)
    stream = stream_loo(name, post)
    save_df(pd.DataFrame({"elpd_i": stream["elpd_i"], "pareto_k": stream["pareto_k"]}), pw_name)
    save_json({k: v for k, v in stream.items() if k not in ("elpd_i", "pareto_k")}, sc_name)
    return stream


# -------------------------------------------------------------------------------------
#  Comparison, on the streamed results
# -------------------------------------------------------------------------------------
def _elpd_data(stream):
    """Wrap a streamed result in the ELPDData object az.compare expects.

    Constructed rather than returned by az.loo because log_likelihood is never resident. Both
    ArviZ major versions are handled: 0.x takes a data/index pair, 1.x a positional record.
    With this in place az.compare does the comparison itself, so elpd_diff, dse, se, weight
    and rank are ArviZ's own numbers - verified identical to az.compare on the same fits.
    """
    n_obs = stream["n_obs"]
    elpd_i = xr.DataArray(stream["elpd_i"], dims=["__obs__"])
    pareto_k = xr.DataArray(stream["pareto_k"], dims=["__obs__"])
    try:
        from arviz_stats.utils import ELPDData as _ELPD1
        return _ELPD1("loo", stream["elpd_loo"], stream["se"], stream["p_loo"],
                      stream["n_samples"], n_obs, "log", stream["warning"], stream["good_k"],
                      elpd_i, pareto_k, approx_posterior=False, log_weights=None)
    except Exception:
        pass
    from arviz.stats import ELPDData as _ELPD0
    return _ELPD0(
        data=[stream["elpd_loo"], stream["se"], stream["p_loo"], stream["n_samples"], n_obs,
              stream["warning"], elpd_i.rename("loo_i"), pareto_k, "log", stream["good_k"]],
        index=["elpd_loo", "se", "p_loo", "n_samples", "n_data_points", "warning",
               "loo_i", "pareto_k", "scale", "good_k"])


def _compare_manual(streams):
    """Fallback comparison, on ArviZ's own definitions.

    Used only if az.compare cannot consume a constructed ELPDData on this ArviZ version.
    elpd_diff and dse - the two columns every decision below reads - are computed exactly as
    arviz.compare computes them: the best fit's per-observation elpd minus this fit's, summed,
    and sqrt(n * var(diff)) with var on ArviZ's default ddof=0. `weight` needs az.compare's
    stacking optimisation and is reported as NaN rather than guessed at.
    """
    names = sorted(streams, key=lambda n: -streams[n]["elpd_loo"])
    best = names[0]
    rows = []
    for rank, name in enumerate(names):
        s = streams[name]
        diff = streams[best]["elpd_i"] - s["elpd_i"]
        rows.append({"rank": rank, "elpd_loo": s["elpd_loo"], "p_loo": s["p_loo"],
                     "elpd_diff": float(diff.sum()), "weight": np.nan, "se": s["se"],
                     "dse": float(np.sqrt(len(diff) * np.var(diff))),
                     "warning": s["warning"], "scale": "log"})
    return pd.DataFrame(rows, index=pd.Index(names, name="model"))


def _compare_from_streams(streams):
    try:
        comparison = az.compare({k: _elpd_data(v) for k, v in streams.items()}, ic="loo")
        if "elpd_diff" in comparison and "dse" in comparison:
            return comparison
    except Exception as exc:
        print(f"    (az.compare unavailable on constructed ELPDData: {exc!r}; "
              f"using the equivalent table)")
    return _compare_manual(streams)


def _quality_from_streams(streams):
    rows = []
    for name, s in streams.items():
        max_k = float(np.nanmax(s["pareto_k"]))
        reliable = bool(np.isfinite(max_k) and not s["warning"] and max_k <= s["good_k"])
        rows.append({"fit": name, "max_pareto_k": max_k, "good_k": s["good_k"],
                     "loo_warning": s["warning"], "loo_reliable": reliable})
    return pd.DataFrame(rows).set_index("fit")


def _decide(comparison, quality):
    """bc.loo_compare's decision half, unchanged: 2*dse separation and Pareto-k reliability."""
    winner = str(comparison.index[0]); runner_up = str(comparison.index[1])
    margin = float(comparison.iloc[1]["elpd_diff"]); dse = float(comparison.iloc[1]["dse"])
    reliable = bool(quality["loo_reliable"].all())
    separated = bool(np.isfinite([margin, dse]).all() and margin >= 2.0 * dse)
    return {"winner": winner, "runner_up": runner_up, "elpd_margin": margin, "dse": dse,
            "loo_reliable": reliable, "separated_2dse": separated,
            "comparison_ok": reliable and separated}


def _have_loo_result(prefix):
    return all(have(f"{prefix}_{s}") for s in
               ("comparison.parquet", "quality.parquet", "decision.json"))


def _save_loo_result(prefix, comparison, quality, decision):
    save_df(comparison.rename_axis("model").reset_index(), f"{prefix}_comparison.parquet")
    save_df(quality.rename_axis("fit").reset_index(), f"{prefix}_quality.parquet")
    save_json(decision, f"{prefix}_decision.json")


def _load_loo_result(prefix):
    return (load_df(f"{prefix}_comparison.parquet").set_index("model"),
            load_df(f"{prefix}_quality.parquet").set_index("fit"),
            load_json(f"{prefix}_decision.json"))


def loo_group(prefix, members, subset=None, subset_prefix=None):
    """Stream LOO for every fit of one comparison, then compare and checkpoint.

    `members` maps a display label to its checkpoint name. Nothing but the current slice of
    one fit's log_likelihood is ever resident, and each fit's per-observation elpd_i (one
    float per observation, a few MB) is all that is carried between fits.
    """
    sub_name = subset_prefix or (f"{prefix}__subset" if subset is not None else None)
    if _have_loo_result(prefix) and (sub_name is None or _have_loo_result(sub_name)):
        comparison, quality, decision = _load_loo_result(prefix)
        print(f"  {prefix}: reusing checkpointed LOO comparison")
    else:
        streams = {}
        for label, name in members.items():
            streams[label] = stream_loo_cached(name)
            gc.collect()
        comparison = _compare_from_streams(streams)
        quality = _quality_from_streams(streams)
        decision = _decide(comparison, quality)
        _save_loo_result(prefix, comparison, quality, decision)
        if subset is not None:
            sub_streams = {k: streams[k] for k in subset}
            sub_cmp = _compare_from_streams(sub_streams)
            sub_q = _quality_from_streams(sub_streams)
            _save_loo_result(subset_prefix or f"{prefix}__subset",
                             sub_cmp, sub_q, _decide(sub_cmp, sub_q))
        del streams
        gc.collect()
    return comparison, quality, decision


def report(idata, var, transform=None, label=""):
    x = idata.posterior[var].values.ravel()
    if transform is not None:
        x = transform(x)
    lo, hi = np.quantile(x, [0.025, 0.975])
    print(f"  {label or var:<26s} median {np.median(x):+.4f}   95% CrI [{lo:+.4f}, {hi:+.4f}]")
    return dict(parameter=label or var, median=float(np.median(x)),
                lo=float(lo), hi=float(hi), sd=float(x.std()))


def prob(idata, var, condition):
    return float(np.mean(condition(idata.posterior[var].values.ravel())))


## 4.1, Model A: H1 at the behavioural level

**Estimand.** $\bar\beta$, the population log-ratio of forecast amplitude recovery at a phase-lock
against its matched controls. $e^{\bar\beta} < 1$ means the lock recovers *less*, which is what H1
predicts.

**Reported.** $e^{\bar\beta}$ with a 95% credible interval; $\Pr(\bar\beta<\log 0.8\mid D)$, the
probability of at least 20% attenuation; and $\Pr(|\bar\beta|<\log 1.1\mid D)$, the probability the
effect is practically nil. The per-configuration $\beta_c$ are shown as a forest plot, because H1
does not have to hold uniformly, and the overlap slope $\delta_O$ says whether it weakens as the
patches overlap more.

In [ ]:
if FIT_MODEL_A:
    idata_A = posterior_or_fit("04_A.nc", lambda: sample(model_A_contrast(arms), "A"))

    print("\nModel A - H1 behavioural and M1")
    rowsA = [report(idata_A, "gamma", label="gamma (log odds ratio)"),
             report(idata_A, "gamma", np.exp, label="odds ratio exp(gamma)"),
             report(idata_A, "delta_O", label="delta_O (overlap mitigation)"),
             report(idata_A, "delta_P", label="delta_P (log patch-size slope)")]
    pA_att = prob(idata_A, "gamma", lambda x: x < ATTENUATION_20)
    pA_rope = prob(idata_A, "gamma", lambda x: np.abs(x) < ROPE_LOG)
    pA_neg = prob(idata_A, "gamma", lambda x: x < 0)
    pA_mit = prob(idata_A, "delta_O", lambda x: x > 0)
    print(f"P(gamma < log 0.8 | D) = {pA_att:.3f}")
    print(f"P(|gamma| < log 1.1 | D) = {pA_rope:.3f}")
    print(f"P(gamma < 0 | D) = {pA_neg:.3f}")
    print(f"P(delta_O > 0 | D) = {pA_mit:.3f}  (positive is mitigation on d)")

    # The four configuration-level parameterisations of M1. Each fit's log_likelihood is
    # streamed off disk one observation-slice at a time and never held; only its per-
    # observation elpd_i survives to the comparison. The 4-way table and the exact
    # overlap-vs-patch-only subset are checkpointed under the names Part 5 expects.
    M1_MEMBERS = {"overlap + patch size": "04_A.nc",
                  "overlap only": "04_A_overlap.nc",
                  "patch size only": "04_A_patch.nc",
                  "neither": "04_A_none.nc"}
    # Every variant must exist before its LOO is streamed, so they are fitted (or loaded) first.
    # Posteriors are a few MB each and Part 5's convergence gate reads them.
    fitsM1 = {"overlap + patch size": idata_A}
    for level, label in (("overlap", "overlap only"), ("patch", "patch size only"),
                         ("none", "neither")):
        fitsM1[label] = posterior_or_fit(
            f"04_A_{level}.nc",
            lambda level=level, label=label: sample(
                model_A_contrast(arms, config_level=level), f"A:{label}"))

    cmpM1_all, looQ_M1_all, looM1_all = loo_group(
        "04_M1_all", M1_MEMBERS,
        subset=("overlap + patch size", "patch size only"), subset_prefix="04_M1_subset")
    cmpM1, looQ_M1, looM1 = _load_loo_result("04_M1_subset")

    M1_LOO_WIN = bc.required_loo_win(looM1, "overlap + patch size")
    display(cmpM1_all)
    display(looQ_M1_all)

    axes = plot_forest_compat(idata_A, "beta", prob=0.95, figsize=(8, 3.4))
    axes[0].axvline(0, color="k", lw=.9)
    axes[0].axvline(ATTENUATION_20, color="crimson", ls="--", lw=1)
    axes[0].set_title("Model A: configuration effects")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "P4_A_forest.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("FIT_MODEL_A is False - skipping Model A")


In [ ]:
# ---- Scope: is the estimand "all arms" or "arms where the tone is measurable"? -------------
# Both fits are produced and the choice is reported rather than assumed, the way Model B reports
# its blocking decision. Conditioning shifts the estimand to "among arms where the tone is
# measurable", which is the honest scope when the instrument's ceiling varies over the design.
#
# The comparison CANNOT go through az.compare. elpd is a sum over observations, and these two fits
# are not fitted to the same observations: the unconditioned one has every arm, the conditioned one
# has a subset. Ranking sums over different row sets compares the size of the data, not the models.
# What is comparable is the pointwise elpd on the rows they SHARE, so the difference is taken per
# observation on the measurable arms and its standard error from the paired differences, which is
# exactly how ArviZ forms dse for two fits on common data.
if FIT_MODEL_A:
    idata_A_cond = posterior_or_fit(
        "04_A_cond.nc", lambda: sample(model_A_contrast(arms_measurable), "A:conditioned"))

    _uncond = stream_loo_cached("04_A.nc")
    _cond = stream_loo_cached("04_A_cond.nc")
    shared_uncond = np.asarray(_uncond["elpd_i"], float)[MEASURABLE_IDX]
    shared_cond = np.asarray(_cond["elpd_i"], float)
    if shared_uncond.shape != shared_cond.shape:
        raise ValueError(f"scope comparison misaligned: {shared_uncond.shape} vs {shared_cond.shape}; "
                         f"MEASURABLE_IDX must index the unconditioned fit's observations")
    paired = shared_cond - shared_uncond
    elpd_diff = float(paired.sum())
    dse = float(np.sqrt(paired.size) * paired.std(ddof=1))
    separated = bool(np.isfinite([elpd_diff, dse]).all() and abs(elpd_diff) >= 2.0 * dse)

    scope_tbl = pd.DataFrame([{
        "rows_unconditioned": int(len(arms)), "rows_conditioned": int(len(arms_measurable)),
        "elpd_diff_cond_minus_uncond": round(elpd_diff, 2), "dse": round(dse, 2),
        "separated_2dse": separated,
        "better_on_shared_rows": ("conditioned" if elpd_diff > 0 else "unconditioned")
                                 if separated else "inconclusive",
        "gamma_uncond": round(float(np.median(idata_A.posterior["gamma"].values)), 4),
        "gamma_cond": round(float(np.median(idata_A_cond.posterior["gamma"].values)), 4),
    }])
    save_df(scope_tbl, "04_A_scope.parquet")
    display(scope_tbl)
    print("PRIMARY SCOPE:", A_PRIMARY_SCOPE, "- preregistered in Part 0.3b, not chosen here.")
    print("The other is reported beside it; a 2*dse-separated preference for the scope that is not"
          " primary is a finding to state in the report, not a licence to swap them after the fact.")
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for lbl, idata, col in (("unconditioned", idata_A, "#1565c0"),
                            ("conditioned", idata_A_cond, "#c62828")):
        g = np.asarray(idata.posterior["gamma"].values).ravel()
        ax.hist(g, bins=80, histtype="step", lw=1.7, density=True, color=col, label=lbl)
    ax.axvline(ATTENUATION_20, color="k", ls="--", lw=1, label=r"support threshold $\log 0.8$")
    ax.axvline(0.0, color="grey", lw=.8)
    ax.set_xlabel(r"$\gamma$ (log odds ratio)"); ax.set_ylabel("density")
    ax.set_title("H1 behavioural under both scopes"); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "P5_scope_comparison.png", dpi=140, bbox_inches="tight"); plt.show()
    del _uncond, _cond
    gc.collect()


## 4.2, Model B: H1 at the representational level

**Estimand.** $\theta_{lock}$, the log-ratio by which the frequency-local prequential codelength
expands at a locked frequency. $e^{\theta_{lock}} > 1$ means the representation needs *more* bits
to separate a locked frequency from its neighbours, information loss in the sense of Voita &
Titov, independent of whether the forecast happens to recover the tone.

This is the second half of the deliverable's original two-part framework, and it can disagree with
Model A. Model A asks whether the forecast rebuilds the tone; Model B asks whether the frequency is
still *readable* inside the network. A lock that is rebuilt but no longer distinguishable from its
neighbours would show up here and nowhere else.

The unadjusted form (Eq. (10) taken literally) is fitted alongside and compared by LOO, so the
blocking decision of Ãƒâ€šSection 3.2 is reported rather than assumed.

In [ ]:
if FIT_MODEL_B:
    idata_B = posterior_or_fit("04_B.nc",
        lambda: sample(model_B_codelength(mdl_cells, adjusted=True), "B"))
    idata_B0 = posterior_or_fit("04_B_unadjusted.nc",
        lambda: sample(model_B_codelength(mdl_cells, adjusted=False), "B0"))

    print("\nModel B - H1 representational")
    rowsB = [report(idata_B, "theta_lock", label="theta_lock (log ratio)"),
             report(idata_B, "theta_lock", np.exp, label="codelength ratio"),
             report(idata_B, "k", label="Gamma shape k")]
    pB_exp = prob(idata_B, "theta_lock", lambda x: x > 0)
    pB_20 = prob(idata_B, "theta_lock", lambda x: x > np.log(1.2))
    pB_rope = prob(idata_B, "theta_lock", lambda x: np.abs(x) < ROPE_LOG)
    print(f"P(theta_lock > log 1.2 | D) = {pB_20:.3f}")
    print(f"P(|theta_lock| < log 1.1 | D) = {pB_rope:.3f}")

    cmpB, looQ_B, looB = loo_group("04_B_loo", {
        "stage+geometry adjusted": "04_B.nc",
        "literal unadjusted": "04_B_unadjusted.nc"})
    display(cmpB)
    display(looQ_B)
else:
    print("FIT_MODEL_B is False - skipping Model B")


## 4.3, Model C: H2, is the deficit set by the geometry or by the phase?

**Estimand.** $\sigma_\phi$, the spread of the per-phase offsets. The phase circle is cut into
eight equal slices, each slice gets its own offset, and $\sigma_\phi$ says how far apart those
offsets are, that is, how much the lock deficit moves as the signal slides through its cycle.

H2 says the deficit is set by the stride/patch alignment and not by where the signal happens to
start, so it predicts $\sigma_\phi \approx 0$. Two readings are reported, answering different
questions:

* $\Pr(\sigma_\phi<\log 1.1\mid D)$, is the phase dependence *practically* negligible? (a
  magnitude question)
* LOO against the model with no phase term at all, does allowing phase to matter *predict better*?
  (an evidential question)

They can disagree, and the disagreement is informative: a small but consistently detected phase
dependence would show a low equivalence probability together with a LOO preference for the phase
model.

In [ ]:
if FIT_MODEL_C:
    idata_C = posterior_or_fit("04_C.nc",
        lambda: sample(model_C_phase(arms, with_phase=True), "C"))
    idata_C0 = posterior_or_fit("04_C_nophase.nc",
        lambda: sample(model_C_phase(arms, with_phase=False), "C0"))

    print("\nModel C - H2 phase invariance")
    rowsC = [report(idata_C, "sigma_phase", label="sigma_phase"),
             report(idata_C, "gamma", label="gamma (log odds ratio)")]
    pC_rope = prob(idata_C, "sigma_phase", lambda x: x < ROPE_LOG)
    print(f"P(sigma_phase < log 1.1 | D) = {pC_rope:.3f}")

    cmpC, looQ_C, looC = loo_group("04_C_loo", {
        "phase-dependent": "04_C.nc",
        "phase-free": "04_C_nophase.nc"})
    display(cmpC)
    display(looQ_C)

    u_phase = idata_C.posterior["u_phase"].values.reshape(-1, N_PHASE_BINS)
    low, median, high = np.quantile(u_phase, [0.025, 0.5, 0.975], axis=0)
    centres = (np.arange(N_PHASE_BINS) + 0.5) * (2 * np.pi / N_PHASE_BINS)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    ax[0].errorbar(centres, median, yerr=[median - low, high - median], fmt="o",
                   color="crimson", capsize=3)
    ax[0].axhline(0, color="k", lw=.8)
    ax[0].set_xlabel(r"phase $\phi$ [rad]")
    ax[0].set_title("per-phase offsets, 95% CrI")
    sigma_post = idata_C.posterior["sigma_phase"].values.ravel()
    ax[1].hist(sigma_post, bins=70, density=True, color="crimson", alpha=.8, label="posterior")
    ax[1].hist(sigma_phase_prior, bins=70, density=True, histtype="step", color="0.4",
               label="prior")
    ax[1].axvline(ROPE_LOG, color="k", ls="--")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "P4_C_phase.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("FIT_MODEL_C is False - skipping Model C")


## 4.4, Model D1: H3, which parameter generates the degradation sites?

**Estimand.** Not an effect size but a *location law*: is the token dispersion systematically lower
on one of the two predicted grids? Each frequency is labelled by the grid it falls on (within 1 Hz)
and three hypotheses are fitted to the same profiles, then compared by LOO:

* $M_S$, only the stride indicator $\mathbb{1}[f \in \{c f_s/S\}]$;
* $M_P$, only the patch indicator $\mathbb{1}[f \in \{k f_s/P\}]$;
* $M_0$, neither.

H3 predicts $\theta_S < 0$ under $M_S$. The comparison is only meaningful because the sweep grid is
the union of *every* geometry's predicted sites: each model is scored where its rival predicts a dip
as well as where it does. And it is only identified because the design contains configurations with
$S<P$, where the two grids genuinely differ, on the $P=S$ diagonal they coincide by construction
and no amount of data can separate them.

Fitted per signal mode. On a pure sinusoid the dispersion is exactly zero at a lock, so `z_norm` is
floored at `COMB_FLOOR` before the log; the floor bounds how large $|\theta_S|$ can get but affects
$M_S$, $M_P$ and $M_0$ identically, leaving the *comparison*, which is the inference, untouched.

In [ ]:
if FIT_MODEL_D1:
    D1 = {}
    cmpD1_rows = []
    cmpD1_decisions = {}
    looQ_D1 = {}
    D1_BOTH_LABEL = "M_SP: both grids (H3)"
    D1_GRIDS = (("both", D1_BOTH_LABEL), ("stride", "M_S: stride grid"),
                ("patch", "M_P: patch grid"), ("none", "M_0: no grid"))

    for mode in sorted(collapse["mode"].unique()):
        members = {label: f"04_D1_{mode}_{grid}.nc" for grid, label in D1_GRIDS}
        subset = collapse[collapse["mode"] == mode].reset_index(drop=True)

        # D1's posteriors are small (per-geometry level plus two grid coefficients), so all four
        # are kept for Part 5's convergence gate; only log_likelihood was ever the problem. They
        # are fitted or loaded before the comparison, which streams their log_likelihood.
        D1[mode] = {
            label: posterior_or_fit(
                name,
                lambda grid=grid, subset=subset, mode=mode: sample(
                    model_D1_sites(subset, grid), f"D1:{mode}:{grid}"))
            for (grid, label), name in zip(D1_GRIDS, members.values())}
        comparison, quality, decision = loo_group(f"04_D1_{mode}_loo", members)

        cmpD1_decisions[mode] = decision
        looQ_D1[mode] = quality
        comparison = comparison.copy()
        comparison.insert(0, "mode", mode)
        cmpD1_rows.append(comparison.reset_index().rename(columns={"index": "model"}))
        print(f"\n--- collapse mode: {mode} ---")
        display(comparison)
        display(quality)

    cmpD1 = pd.concat(cmpD1_rows, ignore_index=True)
    display(cmpD1.sort_values(["mode", "rank"]))
    print("Pure is an exact-degeneracy reference. Only TSMixup and KernelSynth enter the empirical H3 verdict.")
else:
    print("FIT_MODEL_D1 is False - skipping Model D1")


In [ ]:
if FIT_MODEL_D1:
    # the observed profiles with both candidate grids drawn on: this is where H3 is visible or not
    modes = sorted(collapse["mode"].unique())
    geoms = [pl.model_tag(P, S) for P, S in pl.MODELS]
    fig, axes = plt.subplots(len(geoms), len(modes), figsize=(5.0 * len(modes), 2.0 * len(geoms)),
                             sharex=True, squeeze=False)
    for r, gname in enumerate(geoms):
        for c, mode in enumerate(modes):
            ax = axes[r][c]
            sub = collapse[(collapse["model"] == gname) & (collapse["mode"] == mode)]
            if len(sub) == 0:
                ax.axis("off"); continue
            prof = sub.groupby("f", as_index=False)["z_norm"].mean().sort_values("f")
            P0, S0 = int(sub["P"].iloc[0]), int(sub["S"].iloc[0])
            ax.plot(prof["f"], prof["z_norm"], color="#c62828", lw=1.0)
            for f in pl.stride_locks(S0):
                ax.axvline(f, color="#6a1b9a", ls="--", lw=.9, alpha=.65)
            for f in pl.patch_nulls(P0):
                ax.axvline(f, color="#1565c0", ls=":", lw=.9, alpha=.5)
            ax.set_yticks([]); ax.margins(x=.01)
            if c == 0:
                ax.set_ylabel(gname, rotation=0, ha="right", va="center", fontsize=9)
            if r == 0:
                ax.set_title(mode, fontsize=10)
            if r == len(geoms) - 1:
                ax.set_xlabel("frequency [Hz]")
    fig.suptitle("H3: collapse profiles vs the two candidate grids "
                 "(purple dashed = $c f_s/S$, blue dotted = $k f_s/P$)", y=1.005, fontsize=11)
    fig.tight_layout(); fig.savefig(FIG_DIR / "P4_D1_combs.png", dpi=140, bbox_inches="tight"); plt.show()
else:
    print("FIT_MODEL_D1 is False - skipping D1 grid-profile plot")


## 4.5, Models D2: H3a and H3b, do the sites *move* with the geometry?

$\S 4.4$ says where the dips sit. It does not yet say they *move*: each geometry's grid is fixed,
so a model fitting that grid well is still consistent with dips that merely happen to coincide with
it. H3's actual claim is a law across geometries, and it is **two** laws, one per branch.

$\mathcal F_{lock}$ is a union, so the detected set is the union of two combs and has no single
spacing. Every detected dip is therefore first assigned to the branch that predicts it, sites
belonging to both are set aside, and the fundamental of each branch is compared with the spacing
that branch predicts:

$$\hat f^{\,F}_{1,g} \;\sim\; \mathcal N\!\left(\kappa_F\,\Delta^F_g,\;
\sqrt{\sigma_F^2+\Delta f^2}\right), \qquad
\Delta^S_g = f_s/S_g, \quad \Delta^P_g = f_s/P_g .$$

**H3a predicts $\kappa_S = 1$ and H3b predicts $\kappa_P = 1$**: each branch tracks its own
parameter one for one, so a site that stays put while that parameter changes refutes its branch.

$\kappa_S$ is identified by the series at fixed $P$, which the design supplies at $P=16$, $P=24$ and $P=32$; $\kappa_P$ by the pairs that share a stride and differ in patch size, at $S=8$, $12$ and
$16$.

The residual carries a floor $\Delta f$, the sweep step: a spacing read off a discrete grid is no
more precise than one step. Without it, sites landing exactly on the prediction drive $\sigma_F$ to
zero and the posterior develops a funnel NUTS cannot traverse. With it, a near-zero $\sigma_F$ is a
result rather than a sampler failure.

Both the fundamental and the median gap are fitted. With three or four sites in band a single
spurious detection inserts a short interval and flips the median, to which the fundamental is
immune, so a disagreement between the two is itself informative about detection stability.

In [ ]:
if FIT_MODEL_D2:
    # D2 primary response: the branch fundamental needs one detected site. The median-gap robustness
    # response needs at least two. Pure-tone rows remain a reference and do not enter primary evidence.
    D2_IDENTIFICATION = bc.identified_branches(
        sites, minimum_sites=CFG.min_d2_sites, modes=tuple(CFG.generators))
    sites_f1 = sites[(sites["rep"] >= 0) & (sites["n_sites"] >= 1)
                     & sites["mode"].isin(CFG.generators)].copy()
    sites_gap = sites[(sites["rep"] >= 0) & (sites["n_sites"] >= 2)
                      & sites["delta_hat"].notna() & sites["mode"].isin(CFG.generators)].copy()
    print("D2 identification (minimum 10 unambiguous sites across generated-background design):",
          D2_IDENTIFICATION)

    idata_D2, idata_D2d = {}, {}
    for branch in ("stride", "patch"):
        if not D2_IDENTIFICATION[branch]:
            print(f"{branch}: NOT IDENTIFIED; no D2 posterior will be read")
            continue
        branch_f1 = sites_f1[sites_f1["branch"] == branch]
        if branch_f1.empty:
            raise ValueError(f"{branch} passed the site-count bar but has no replicate-level f1 rows")
        idata_D2[branch] = posterior_or_fit(
            f"04_D2_{branch}.nc",
            lambda branch=branch: sample(
                model_D2_movement(sites_f1, branch, response="f1"), f"D2:{branch}:f1"))
        branch_gap = sites_gap[sites_gap["branch"] == branch]
        if not branch_gap.empty:
            idata_D2d[branch] = posterior_or_fit(
                f"04_D2_{branch}_deltahat.nc",
                lambda branch=branch: sample(
                    model_D2_movement(sites_gap, branch, response="delta_hat"),
                    f"D2:{branch}:delta_hat"))

    names = {"stride": "kappa_S", "patch": "kappa_P"}
    pD2, pD2_gap = {}, {}
    for branch, idata in idata_D2.items():
        parameter = names[branch]
        report(idata, parameter, label=f"{parameter} primary f1")
        pD2[branch] = prob(idata, parameter, lambda x: np.abs(x - 1.0) < ROPE_SLOPE)
        print(f"P(|{parameter}-1| < {ROPE_SLOPE} | D) = {pD2[branch]:.3f}")
        if branch in idata_D2d:
            pD2_gap[branch] = prob(idata_D2d[branch], parameter,
                                   lambda x: np.abs(x - 1.0) < ROPE_SLOPE)
            print(f"median-gap robustness probability = {pD2_gap[branch]:.3f}; "
                  f"absolute difference={abs(pD2[branch] - pD2_gap[branch]):.3f}")
        else:
            print("median-gap robustness: unavailable (fewer than two sites per contributing row)")
else:
    print("FIT_MODEL_D2 is False - skipping Model D2")


---
# Part 5, Checks

A posterior is a conditional statement: *given* that the sampler converged, that the model can
reproduce the data, and that the answer does not hinge on an arbitrary prior scale. This part tests
all three, and only then states the verdicts.

* **5.1 Convergence**, $\hat R$, bulk and tail ESS, divergences, against the thresholds the
  deliverable preregisters ($\hat R<1.01$, ESS $>1000$, zero divergences).
* **5.2 Posterior predictive**, can the fitted model generate data that look like the observations,
  stratified by frequency, phase, $P$, $S$ and generator? A model that fits the pooled histogram but
  fails a stratum is fitting the average and missing the structure.
* **5.3 Prior sensitivity**, the whole ladder $\{0.25, 0.5, 1.0\}$ plus a Gaussian likelihood. If
  the conclusion moves with the prior scale, it was the prior's conclusion.
* **5.4 Model comparison**, the LOO tables gathered in one place.
* **5.5 Verdicts**, one row per hypothesis, next to the frequentist PASS/FAIL that
  `chronos/testing/hypotheses.py` produces on the same phenomenon.

In [ ]:
if not RUN_PART_5:
    raise RuntimeError("RUN_PART_5 is False - stopping before Part 5.")
_missing = [name for name, flag in (
    ("FIT_MODEL_A", FIT_MODEL_A), ("FIT_MODEL_B", FIT_MODEL_B), ("FIT_MODEL_C", FIT_MODEL_C),
    ("FIT_MODEL_D1", FIT_MODEL_D1), ("FIT_MODEL_D2", FIT_MODEL_D2),
) if not flag]
if len(_missing) == 5:
    raise RuntimeError("Part 5 needs at least one model read in this session.")
if _missing:
    # Part 5 runs on the models this session selected. Every verdict in 5.5 already carries its
    # own diagnostics_ok/recovery_ok/ppc_ok terms, so a model that was never fitted simply never
    # reaches the table; demanding all five would make a single-model session impossible to check.
    print(f"Part 5 runs on the selected models only; not read this session: {_missing}")

banner("PART 5 - SCIENTIFIC VALIDITY CHECKS")

# Same set of fits, same gate (bc.fit_diagnostics: R-hat < 1.01, bulk/tail ESS > 1000, zero
# divergences). Every entry here is a posterior-only InferenceData - the gate reads posterior
# and sample_stats, never log_likelihood - so the whole table costs a few hundred MB.
FITS = {}
if FIT_MODEL_A:
    FITS.update({"A": idata_A,
                 "A:overlap-only": fitsM1["overlap only"],
                 "A:patch-only": fitsM1["patch size only"],
                 "A:neither": fitsM1["neither"]})
if FIT_MODEL_B:
    FITS.update({"B": idata_B, "B:unadjusted": idata_B0})
if FIT_MODEL_C:
    FITS.update({"C": idata_C, "C:phase-free": idata_C0})
if FIT_MODEL_D1:
    for mode, fits in D1.items():
        for label, idata in fits.items():
            FITS[f"D1:{mode}:{label}"] = idata
if FIT_MODEL_D2:
    for branch, idata in idata_D2.items():
        FITS[f"D2:{branch}:f1"] = idata
    for branch, idata in idata_D2d.items():
        FITS[f"D2:{branch}:delta_hat"] = idata

diagnostics = save_df(bc.diagnostics_table(FITS, az), "05_diagnostics.parquet")
display(diagnostics.round(3))
failed = diagnostics.loc[~diagnostics["diagnostics_ok"], "fit"].tolist()
print("diagnostic thresholds: R-hat < 1.01, bulk/tail ESS > 1000, zero divergences")
print("diagnostics gate:", "PASS" if not failed else f"FAIL for {failed}")

def diagnostics_ok(*names: str) -> bool:
    subset = diagnostics[diagnostics["fit"].isin(names)]
    return bool(len(subset) == len(set(names)) and subset["diagnostics_ok"].all())


## 5.2 - Posterior predictive checks, pooled and stratified

The primary A, B, C, D1 and D2 fits are checked at the level at which each model can fail. At least
90% of preregistered strata must place the observed mean inside the 95% replicated-mean interval.
This check is a model adequacy gate, not a hypothesis result.

In [ ]:
PPC_VERSION = "d3-bernoulli-v1"
PPC_FILE = "05_ppc_bernoulli_v1.parquet"
PPC_DRAW_CHUNK = 50          # posterior draws replicated at a time, per chain

def _ppc_rows(analysis: str, model, idata, observed_name: str, observed,
              frame: pd.DataFrame, columns: tuple[str, ...]) -> tuple[list[dict], set[str]]:
    """Stratified posterior-predictive check, replicated in draw-blocks.

    The statistic is a stratum mean per replicated dataset, and its 95% interval over
    replications. Materialising every replication first - draws x chains x observations, which
    for Model A is the same ~9 GB shape as log_likelihood - was the second thing that could
    exhaust this runtime. Only the per-draw stratum means are ever accumulated here (one float
    per draw per stratum), so the replicated array exists one draw-block at a time. Every draw
    still contributes exactly once, so the intervals are the ones the monolithic version
    computed, not an approximation of them.
    """
    frame = frame.reset_index(drop=True)
    observed = np.asarray(observed, float)
    if len(frame) != len(observed):
        raise ValueError(f"{analysis}: PPC frame/observation length mismatch")

    positions = {}
    for column in columns:
        for level, idx in frame.groupby(column, observed=True).indices.items():
            positions[f"{analysis}:{column}={level}"] = (column, level, np.asarray(idx, int))

    n_draw = int(idata.posterior.sizes["draw"])
    n_blocks = math.ceil(n_draw / PPC_DRAW_CHUNK)
    print(f"  PPC {analysis}: {len(frame):,} observations, {n_draw:,} draws per chain, "
          f"{n_blocks} blocks of {PPC_DRAW_CHUNK}")
    t0 = time.time()
    collected = {stratum: [] for stratum in positions}
    for start in range(0, n_draw, PPC_DRAW_CHUNK):
        stop = min(start + PPC_DRAW_CHUNK, n_draw)
        block = idata.isel(draw=slice(start, stop))
        with model:
            predictive = pm.sample_posterior_predictive(
                block, random_seed=SEED, progressbar=False)
        replicated = np.asarray(
            predictive.posterior_predictive[observed_name]).reshape(-1, len(frame))
        for stratum, (_column, _level, idx) in positions.items():
            collected[stratum].append(replicated[:, idx].mean(axis=1))
        del predictive, replicated, block
        gc.collect()
    print(f"     {time.time() - t0:.0f}s")

    rows, required = [], set()
    for stratum, (column, level, idx) in positions.items():
        observed_mean = float(observed[idx].mean())
        replicated_means = np.concatenate(collected[stratum])
        low, high = np.quantile(replicated_means, [0.025, 0.975])
        required.add(stratum)
        rows.append({
            "ppc_version": PPC_VERSION, "analysis": analysis, "stratum": stratum,
            "n": len(idx), "observed": observed_mean,
            "rep_low": float(low), "rep_high": float(high),
            "ppc_ok": bool(low <= observed_mean <= high),
        })
    return rows, required

def _ppc_part(key, build):
    """One analysis block of the PPC, checkpointed on its own.

    The completed table is checkpointed as a whole, but that is all-or-nothing: a session lost
    part-way through discards the blocks already finished, and A and C are the expensive ones
    (~138k observations replicated over every posterior draw). Each block is therefore written
    as it completes, on the same principle as the per-fit LOO in Part 4, so an interrupted run
    resumes at the block it stopped on. `build` is a callable so that a cached block never
    constructs its PyMC model, which for A and C is not free either.
    """
    part = f"05_ppc_part__{key}.parquet"
    if have(part):
        frame = load_df(part)
        return frame.to_dict("records"), set(frame["stratum"])
    rows, required = build()
    save_df(pd.DataFrame(rows), part)
    return rows, required


if have(PPC_FILE):
    ppc_table = load_df(PPC_FILE)
    required_ppc = set(ppc_table["stratum"])
else:
    ppc_rows, required_ppc = [], set()
    rows, required = _ppc_part("A", lambda: _ppc_rows(
        "A", model_A_contrast(arms), idata_A, "h", arms["h"].to_numpy(float),
        arms, ("model", "generator")))
    ppc_rows += rows; required_ppc |= required
    gc.collect()

    rows, required = _ppc_part("B", lambda: _ppc_rows(
        "B", model_B_codelength(mdl_cells, adjusted=True), idata_B, "y",
        np.clip(mdl_cells["L_bits"].to_numpy(float), 1e-3, None),
        mdl_cells, ("model", "stage")))
    ppc_rows += rows; required_ppc |= required
    gc.collect()

    rows, required = _ppc_part("C", lambda: _ppc_rows(
        "C", model_C_phase(arms, with_phase=True), idata_C, "h",
        arms["h"].to_numpy(float), arms, ("model", "generator")))
    ppc_rows += rows; required_ppc |= required
    gc.collect()

    for mode in CFG.generators:
        subset = collapse[collapse["mode"] == mode].reset_index(drop=True)
        rows, required = _ppc_part(f"D1_{mode}", lambda subset=subset, mode=mode: _ppc_rows(
            "D1", model_D1_sites(subset, "both"), D1[mode][D1_BOTH_LABEL], "logz",
            np.log(np.clip(subset["z_norm"].to_numpy(float), COMB_FLOOR, None)),
            subset, ("model",)))
        ppc_rows += rows; required_ppc |= required
        gc.collect()

    for branch, idata in idata_D2.items():
        subset = sites_f1[sites_f1["branch"] == branch].reset_index(drop=True)
        rows, required = _ppc_part(f"D2_{branch}", lambda branch=branch, subset=subset,
                                          idata=idata: _ppc_rows(
            f"D2-{branch}", model_D2_movement(sites_f1, branch, response="f1"),
            idata, "f1_hat", subset["f1"].to_numpy(float), subset, ("model", "mode")))
        ppc_rows += rows; required_ppc |= required
        gc.collect()

    ppc_table = save_df(pd.DataFrame(ppc_rows), PPC_FILE)

if set(ppc_table.get("ppc_version", [])) != {PPC_VERSION}:
    raise ValueError("stale/incompatible PPC checkpoint")
PPC_OK = bc.ppc_gate(ppc_table, required_ppc, minimum_coverage=0.90)
display(ppc_table)
display(ppc_table.groupby("analysis")["ppc_ok"].agg(["sum", "count", "mean"]))
print("posterior-predictive gate:", "PASS" if PPC_OK else "FAIL - claims become NOT REPORTABLE")

plot_table = ppc_table.sort_values(["analysis", "stratum"]).reset_index(drop=True)
figure, axis = plt.subplots(figsize=(10, max(4, .18 * len(plot_table))))
y = np.arange(len(plot_table))
axis.hlines(y, plot_table["rep_low"], plot_table["rep_high"], color="steelblue", lw=3, alpha=.6)
axis.scatter(plot_table["observed"], y, c=np.where(plot_table["ppc_ok"], "black", "crimson"), s=12)
axis.set_yticks(y)
axis.set_yticklabels(plot_table["stratum"], fontsize=6)
axis.set_title("Posterior predictive stratum means (red = outside 95% replicated interval)")
figure.tight_layout()
figure.savefig(FIG_DIR / "P5_ppc.png", dpi=140, bbox_inches="tight")
plt.show()


## 5.3 - Prior-scale and likelihood sensitivity

Model A is refitted at all three preregistered prior scales and under the Gaussian likelihood.
Both the H1 attenuation probability and M1 positive-overlap probability must vary by at most 0.10,
and every sensitivity fit must pass the full convergence gate.

In [ ]:

SENSITIVITY_VERSION = "d3-a-bernoulli-v1"
SENSITIVITY_FILE = "05_sensitivity_bernoulli_v1.parquet"

# The ladder re-fits Model A at three prior scales plus a probit-link check. A Bernoulli
# response has no Gaussian alternative, so the fourth variant varies the LINK instead of the
# likelihood; the question is the same, whether the verdict survives a change in the shape of
# the response curve at fixed data and priors. These are
# the only fits this notebook can still produce, and they are saved WITHOUT log_likelihood:
# nothing computes LOO on them, so the 8.4 GiB array PyMC would build after sampling - the
# same transient that has taken down a fresh Model C session - is never allocated, and each
# variant lands on Drive as a few MB instead of 8.4 GiB.
variants = [(f"StudentT scale {scale}", scale, "logit") for scale in PRIOR_LADDER]
variants.append(("probit link scale 0.5", 0.5, "probit"))

if have(SENSITIVITY_FILE):
    sens = load_df(SENSITIVITY_FILE)
else:
    rows, missing_variants = [], []
    wanted = (set(range(len(variants))) if SENSITIVITY_SUBSET is None
              else set(SENSITIVITY_SUBSET))
    for index, (label, scale, link) in enumerate(variants):
        checkpoint = f"05_sensitivity_{index}.nc"
        if not have(checkpoint):
            if not (ALLOW_SAMPLING and index in wanted):
                missing_variants.append(f"{checkpoint} ({label})")
                continue
            print(f"\nfitting {checkpoint} - {label}")
            save_idata(
                sample_without_log_likelihood(
                    model_A_contrast(arms, scale=scale, link=link),
                    f"sensitivity:{label}"),
                checkpoint)
            gc.collect()
        fit = load_posterior_only(checkpoint)
        beta = fit.posterior["gamma"].values.ravel()
        delta = fit.posterior["delta_O"].values.ravel()
        diagnostic = bc.fit_diagnostics(f"sensitivity:{label}", fit, az)
        rows.append({
            "sensitivity_version": SENSITIVITY_VERSION, "variant": label,
            "median_beta": float(np.median(beta)),
            "beta_low": float(np.quantile(beta, .025)), "beta_high": float(np.quantile(beta, .975)),
            "P_attenuation_20": float(np.mean(beta < ATTENUATION_20)),
            "P_beta_in_rope": float(np.mean(np.abs(beta) < ROPE_LOG)),
            "P_delta_O_positive": float(np.mean(delta > 0)),
            "diagnostics_ok": diagnostic["diagnostics_ok"],
            "max_rhat": diagnostic["max_rhat"],
            "min_ess_bulk": diagnostic["min_ess_bulk"],
            "min_ess_tail": diagnostic["min_ess_tail"],
            "divergences": diagnostic["divergences"],
        })
        del fit
        gc.collect()

    if missing_variants:
        # The ladder is a preregistered validation step, so an absent one is a failed gate
        # rather than an error to route around: bc.three_way_verdict is fail-closed, A and M1
        # come out NOT REPORTABLE, and every other claim is decided normally. Stopping here
        # would forfeit those other verdicts over a gate that concerns only two of them.
        print("!" * 78)
        print("Prior-sensitivity ladder incomplete - still absent from CKPT_DIR:")
        for item in missing_variants:
            print("   ", item)
        print("The sensitivity gate therefore FAILS, and H1 behavioural (Model A) and M1 are")
        print("reported NOT REPORTABLE. Their posteriors are still computed and shown above.")
        print("Fit the remaining variants (ALLOW_SAMPLING / SENSITIVITY_SUBSET in Part 0.3b)")
        print("and re-run Part 5; nothing else recomputes, every other checkpoint is reused.")
        print("!" * 78)

    sens = pd.DataFrame(rows)
    # Only a complete ladder is worth checkpointing: a partial one saved under this name would
    # be reloaded as finished by the next session and silently pass a gate it never cleared.
    if len(sens) == len(variants):
        sens = save_df(sens, SENSITIVITY_FILE)

if sens.empty:
    SENSITIVITY_OK = False
else:
    if set(sens.get("sensitivity_version", [])) != {SENSITIVITY_VERSION}:
        raise ValueError("stale/incompatible sensitivity checkpoint")
    SENSITIVITY_OK = (len(sens) == len(variants)
                      and bc.sensitivity_gate(
                          sens, ("P_attenuation_20", "P_delta_O_positive"), max_spread=0.10)
                      and bool(sens["diagnostics_ok"].all()))
    display(sens.round(4))
print(f"sensitivity ladder: {len(sens)}/{len(variants)} variants")
print("sensitivity gate:", "PASS" if SENSITIVITY_OK else "FAIL - A/M1 become NOT REPORTABLE")

if sens.empty:
    print("no sensitivity figure: the ladder has no fitted variants on disk")
else:
    figure, axis = plt.subplots(figsize=(8, 3.2))
    y = np.arange(len(sens))
    axis.hlines(y, sens["beta_low"], sens["beta_high"], color="steelblue", lw=4, alpha=.65)
    axis.scatter(sens["median_beta"], y, color="crimson")
    axis.axvline(ATTENUATION_20, color="crimson", ls="--")
    axis.axvline(0, color="k", lw=.8)
    axis.set_yticks(y)
    axis.set_yticklabels(sens["variant"])
    axis.set_title("Model A prior/likelihood sensitivity")
    figure.tight_layout()
    figure.savefig(FIG_DIR / "P5_sensitivity.png", dpi=140, bbox_inches="tight")
    plt.show()


## 5.4 - LOO comparisons with Pareto-k reliability

A comparison is decisive only when every PSIS-LOO object is reliable and the best model leads the
runner-up by at least twice the reported standard error. H2 reports its magnitude and LOO reading
separately because they answer different questions and may legitimately disagree.

In [ ]:
def loo_reading(decision: dict) -> str:
    if not decision["loo_reliable"]:
        return "INCONCLUSIVE: unreliable PSIS-LOO (inspect Pareto k)"
    if not decision["separated_2dse"]:
        return (f"INCONCLUSIVE: margin {decision['elpd_margin']:.2f} < "
                f"2*dse {2 * decision['dse']:.2f}")
    return f"{decision['winner']} preferred"

print("M1 exact comparison - full overlap+patch vs patch-only")
display(cmpM1); display(looQ_M1); print(loo_reading(looM1))
print("\nM1 four-way descriptive comparison")
display(cmpM1_all); display(looQ_M1_all); print(loo_reading(looM1_all))
print("\nModel B blocking comparison")
display(cmpB); display(looQ_B); print(loo_reading(looB))
print("\nH2 phase-dependent vs phase-free (separate from sigma_phase magnitude)")
display(cmpC); display(looQ_C); print(loo_reading(looC))
print("\nH3 D1 comparisons")
for mode in sorted(cmpD1_decisions):
    print(mode, "-", loo_reading(cmpD1_decisions[mode]))
    display(looQ_D1[mode])

## 5.5 - Fail-closed Deliverable 3 verdicts

Only empirical full-design fits can populate this table. Diagnostics, parameter recovery,
posterior-predictive adequacy and the applicable sensitivity/LOO checks are explicit gates. A
failed gate yields `NOT REPORTABLE`; a D2 branch below ten unambiguous sites yields
`NOT IDENTIFIED`, never a null result. No static result from another notebook or frequentist run is
copied into this table.

In [ ]:
def recovery_ok(model_name: str) -> bool:
    subset = recovery_tbl[recovery_tbl["model"] == model_name]
    return bool(not subset.empty and subset["covered"].all() and subset["diagnostics_ok"].all())

def ppc_ok(analysis: str) -> bool:
    subset = ppc_table[ppc_table["analysis"] == analysis]
    return bool(not subset.empty and subset["ppc_ok"].mean() >= 0.90)

verdict_rows = []

gate_A = diagnostics_ok("A") and recovery_ok("A") and ppc_ok("A") and SENSITIVITY_OK
verdict_rows.append({
    "hypothesis": "H1 behavioural", "model": "A",
    "decision_rule": "P(gamma < log 0.8)>=.95; refute by gamma ROPE>=.95",
    "probability": pA_att, "refute_probability": pA_rope,
    "gate_ok": gate_A,
    "verdict": bc.three_way_verdict(pA_att, pA_rope, gate_A),
    "comparison_or_robustness": "prior/likelihood probability spread <=0.10",
})

gate_B = diagnostics_ok("B") and recovery_ok("B") and ppc_ok("B")
verdict_rows.append({
    "hypothesis": "H1 representational", "model": "B",
    "decision_rule": "P(theta_lock > log 1.2)>=.95; refute by theta ROPE>=.95",
    "probability": pB_20, "refute_probability": pB_rope,
    "gate_ok": gate_B,
    "verdict": bc.three_way_verdict(pB_20, pB_rope, gate_B),
    "comparison_or_robustness": loo_reading(looB),
})

gate_C = diagnostics_ok("C") and recovery_ok("C") and ppc_ok("C")
verdict_rows.append({
    "hypothesis": "H2 phase invariance", "model": "C",
    "decision_rule": "P(sigma_phase < log 1.1)>=.95",
    "probability": pC_rope, "refute_probability": 1 - pC_rope,
    "gate_ok": gate_C,
    "verdict": bc.three_way_verdict(pC_rope, 1 - pC_rope, gate_C),
    "comparison_or_robustness": loo_reading(looC),
})

generated_modes = tuple(CFG.generators)
d1_fit_names = [f"D1:{mode}:{label}" for mode in generated_modes for label in D1[mode]]
d1_sign_probabilities = {}
d1_mode_support = {}
for mode in generated_modes:
    primary = D1[mode][D1_BOTH_LABEL]
    p_stride_negative = prob(primary, "theta_S", lambda x: x < 0)
    p_patch_negative = prob(primary, "theta_P", lambda x: x < 0)
    d1_sign_probabilities[mode] = min(p_stride_negative, p_patch_negative)
    d1_mode_support[mode] = (
        bc.required_loo_win(cmpD1_decisions[mode], D1_BOTH_LABEL)
        and p_stride_negative >= .95 and p_patch_negative >= .95)
gate_D1 = (diagnostics_ok(*d1_fit_names) and recovery_ok("D1") and ppc_ok("D1")
           and all(cmpD1_decisions[mode]["loo_reliable"] for mode in generated_modes))
d1_probability = min(d1_sign_probabilities.values())
if not gate_D1:
    d1_verdict = "NOT REPORTABLE"
elif all(d1_mode_support.values()):
    d1_verdict = "supported"
elif any(d1_probability <= .05 or
         (cmpD1_decisions[mode]["comparison_ok"] and
          cmpD1_decisions[mode]["winner"] != D1_BOTH_LABEL) for mode in generated_modes):
    d1_verdict = "refuted"
else:
    d1_verdict = "inconclusive"
verdict_rows.append({
    "hypothesis": "H3 site location", "model": "D1",
    "decision_rule": "both-grid fit wins LOO and theta_S, theta_P < 0 on both generators",
    "probability": d1_probability, "refute_probability": np.nan,
    "gate_ok": gate_D1, "verdict": d1_verdict,
    "comparison_or_robustness": "; ".join(
        f"{mode}: {loo_reading(cmpD1_decisions[mode])}" for mode in generated_modes),
})

gate_M1 = (diagnostics_ok("A", "A:patch-only") and recovery_ok("A") and ppc_ok("A")
           and SENSITIVITY_OK and looM1["loo_reliable"])
if not gate_M1:
    m1_verdict = "NOT REPORTABLE"
elif pA_mit >= .95 and M1_LOO_WIN:
    m1_verdict = "supported"
elif pA_mit <= .05:
    m1_verdict = "refuted"
else:
    m1_verdict = "inconclusive"
verdict_rows.append({
    "hypothesis": "M1 overlap mitigation", "model": "A'",
    "decision_rule": "P(delta_O > 0)>=.95 and full model beats patch-only by >=2*dse",
    "probability": pA_mit, "refute_probability": 1 - pA_mit,
    "gate_ok": gate_M1, "verdict": m1_verdict,
    "comparison_or_robustness": loo_reading(looM1),
})

for branch, hypothesis, parameter in (
    ("stride", "H3a stride movement", "kappa_S"),
    ("patch", "H3b patch movement", "kappa_P"),
):
    identified = D2_IDENTIFICATION[branch]
    if not identified:
        row = {"probability": np.nan, "refute_probability": np.nan, "gate_ok": True,
               "verdict": "NOT IDENTIFIED", "comparison_or_robustness": "<10 sites"}
    else:
        primary_name = f"D2:{branch}:f1"
        gap_names = ([f"D2:{branch}:delta_hat"] if branch in idata_D2d else [])
        gate = (diagnostics_ok(primary_name, *gap_names)
                and recovery_ok(f"D2-{branch}") and ppc_ok(f"D2-{branch}"))
        probability = pD2[branch]
        robustness = (f"gap probability={pD2_gap[branch]:.3f}, "
                      f"difference={abs(probability - pD2_gap[branch]):.3f}"
                      if branch in pD2_gap else "gap robustness unavailable")
        row = {
            "probability": probability, "refute_probability": 1 - probability,
            "gate_ok": gate,
            "verdict": bc.three_way_verdict(probability, 1 - probability, gate),
            "comparison_or_robustness": robustness,
        }
    verdict_rows.append({
        "hypothesis": hypothesis, "model": "D2",
        "decision_rule": f"P(|{parameter}-1| < {ROPE_SLOPE})>=.95", **row,
    })

verdicts = pd.DataFrame(verdict_rows)
if IS_SMOKE:
    verdicts["gate_ok"] = False
    verdicts["verdict"] = "NON-REPORTABLE SMOKE"
verdicts = save_df(verdicts, "05_verdicts.parquet")
display(verdicts)
print("\nPure-tone/smoke/recovery output is intentionally absent from empirical verdicts.")
print("checkpoints:", CKPT_DIR)

---
## Scope of the conclusions

Every reported configuration was retrained from scratch for the same 100,000-step budget. The
remaining design limitation is the single training seed: posterior uncertainty is conditional on
one checkpoint per geometry, while the configuration hierarchy can absorb but cannot separately
identify between-seed training variation. Failed diagnostics, recovery, PPC, sensitivity or LOO
quality do not become weak evidence; they become `NOT REPORTABLE`.

---
# Part 6, Supplementary diagnostic plots

Part 5 gates every fit on numeric thresholds (R-hat, ESS, divergences, Pareto-k, PPC coverage) but
never *looks* at the chains, the sampler geometry, or the LOO reliability per observation - a model
can pass every scalar gate while a plot would show something a table cannot: a chain that mixed
well on average but drifted early, a funnel the resolution floor (Model D2's docstring, Part 3.2)
didn't fully remove, or one observation dominating a LOO comparison. This part adds those pictures.

They are built directly on `idata.posterior` / `idata.sample_stats`, `az.loo`, and the already-saved
recovery/sensitivity tables - not on `az.plot_trace` / `az.plot_pair` / `az.plot_khat`. arviz's
plotting API changed incompatibly between the two major versions `uv.lock` can resolve to (the same
"PlotCollection" rework that broke `az.plot_forest`, worked around in Part 0.3 with
`plot_forest_compat`), so these are hand-built the same way, for identical output regardless of
which arviz version this session resolved to.

Nothing here recomputes a fit: everything reads variables already sitting in memory from Parts 4-5.
If you are resuming a completed run in a fresh kernel, just re-run the whole notebook top to bottom
first - `have()`/`cached_fit` reload every checkpoint from disk in seconds rather than refitting, so
by the time execution reaches this cell every `idata_*` / `recovery_tbl` / `sens` variable is
populated exactly as it would be after a fresh run.


## 6.1 - Trace plots, chain mixing beyond R-hat/ESS

For every primary estimand (A, B, C, the primary D1 fit per generator, both D2 branches): each
chain's value over draw index, side by side. R-hat and ESS already say whether the chains agree by
the end; this is what "agree" looks like, and would catch a chain that drifted early and only
happened to land in the same place as the others.


In [ ]:
banner("PART 6, SUPPLEMENTARY DIAGNOSTIC PLOTS")

def plot_trace_compat(idata, var_names, figsize=None):
    """Per-chain trace (draw index vs value) for one or more scalar (chain, draw)-only variables.

    Not built on az.plot_trace: arviz's plot_trace return type and keyword arguments changed
    incompatibly between arviz 0.x and >=1.0 (see Part 0.3), so this renders identically
    regardless of arviz version. R-hat/ESS already say WHETHER the chains agree; this shows HOW -
    a chain that drifts early and then agrees on average can still pass both thresholds.
    """
    names = [var_names] if isinstance(var_names, str) else list(var_names)
    fig, axes = plt.subplots(len(names), 1, figsize=figsize or (10, 1.8 * len(names)), squeeze=False)
    for row, name in enumerate(names):
        da = idata.posterior[name]
        extra = [d for d in da.dims if d not in ("chain", "draw")]
        if extra:
            raise ValueError(f"{name} has non-scalar dims {extra}; pass a scalar variable")
        ax = axes[row, 0]
        for c in range(da.sizes["chain"]):
            ax.plot(da.isel(chain=c).values, lw=.6, alpha=.8, label=f"chain {c}")
        ax.set_ylabel(name)
        if row == 0:
            ax.legend(fontsize=7, ncol=da.sizes["chain"], loc="upper right")
    axes[-1, 0].set_xlabel("draw")
    fig.tight_layout()
    return fig, axes

TRACE_TARGETS = [
    (idata_A, ["gamma", "beta_bar", "delta_O", "delta_P", "tau"], "A"),
    (idata_B, ["theta_lock", "k"], "B"),
    (idata_C, ["sigma_phase", "gamma"], "C"),
]
for mode in CFG.generators:
    TRACE_TARGETS.append((D1[mode][D1_BOTH_LABEL], ["theta_S", "theta_P"], f"D1_{mode}"))
for branch, idata in idata_D2.items():
    kappa_name = "kappa_S" if branch == "stride" else "kappa_P"
    TRACE_TARGETS.append((idata, [kappa_name, "sigma_extra"], f"D2_{branch}"))

for idata, names, label in TRACE_TARGETS:
    fig, axes = plot_trace_compat(idata, names, figsize=(10, 1.6 * len(names)))
    fig.suptitle(f"trace - {label}", y=1.02)
    fig.savefig(FIG_DIR / f"P6_trace_{label}.png", dpi=130, bbox_inches="tight")
    plt.show()


## 6.2 - Pair plots, funnel geometry on the parameters flagged at risk

Model D2's docstring (Part 3.2) explicitly names the failure mode: without the sweep-resolution
floor, sigma_extra is pushed toward zero and the posterior develops a funnel NUTS cannot traverse.
The zero-divergence gate (Part 5.1) says this didn't happen; this plot is where that claim becomes
visible, for both D2 branches' kappa/sigma_extra pair and, as a comparison point, Model A's
population effect against its hierarchical scale tau.


In [ ]:
def plot_pair_compat(idata, var_x, var_y, figsize=(6, 5)):
    """Scatter of two scalar parameters with divergent transitions marked - the standard way to
    see whether a near-zero-divergence gate corresponds to genuinely clean sampler geometry, or a
    funnel the resolution floor (Model D2's docstring, Part 3.2) only mostly fixed. Built directly
    on idata.posterior/idata.sample_stats for the same version-independence reason as
    plot_trace_compat above.
    """
    x = idata.posterior[var_x].values.ravel()
    y = idata.posterior[var_y].values.ravel()
    div = np.asarray(idata.sample_stats["diverging"]).ravel().astype(bool)
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(x[~div], y[~div], s=6, alpha=.3, color="steelblue", label="draws")
    if div.any():
        ax.scatter(x[div], y[div], s=22, color="crimson", marker="x",
                   label=f"divergences ({int(div.sum())})")
    ax.set_xlabel(var_x); ax.set_ylabel(var_y)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, ax

PAIR_TARGETS = [(idata_A, "gamma", "tau", "A_gamma_tau", "A: gamma vs tau")]
for branch, idata in idata_D2.items():
    kappa_name = "kappa_S" if branch == "stride" else "kappa_P"
    PAIR_TARGETS.append((idata, kappa_name, "sigma_extra", f"D2_{branch}_funnel",
                         f"D2:{branch} - {kappa_name} vs sigma_extra (the funnel-risk pair)"))

for idata, vx, vy, slug, title in PAIR_TARGETS:
    fig, ax = plot_pair_compat(idata, vx, vy)
    ax.set_title(title)
    fig.savefig(FIG_DIR / f"P6_pair_{slug}.png", dpi=130, bbox_inches="tight")
    plt.show()


## 6.3 - Pareto k per observation

`bc.loo_compare` (Parts 4-5) already gates every comparison on the *maximum* Pareto k; plotting
every point shows which specific observations are close to that threshold, rather than only the
worst one.


In [ ]:

def plot_khat_compat(stream, figsize=(9, 3.2), title=None):
    """Per-observation PSIS-LOO Pareto k, with the standard 0.5/0.7 reference lines.

    Reads the per-observation arrays Part 4 already streamed and checkpointed, rather than
    calling az.loo again: these InferenceData are posterior-only, and recomputing would mean
    materialising log_likelihood, which is the thing this notebook exists to avoid. The k
    values are the same ones the reliability gate in Part 4 was computed from.
    """
    k = np.asarray(stream["pareto_k"], dtype=float)
    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(np.arange(len(k)), k, s=10, color="steelblue")
    for thresh, color, label in ((0.5, "orange", "0.5 (ok)"), (0.7, "crimson", "0.7 (problematic)")):
        ax.axhline(thresh, color=color, ls="--", lw=1, label=label)
    ax.set_xlabel("observation index"); ax.set_ylabel("Pareto k")
    if title:
        ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, ax

# (posterior, checkpoint name, label) for every fit whose diagnostics are plotted below
KHAT_TARGETS = [(idata_A, "04_A.nc", "A"), (idata_B, "04_B.nc", "B"), (idata_C, "04_C.nc", "C")]
for mode in CFG.generators:
    KHAT_TARGETS.append((D1[mode][D1_BOTH_LABEL], f"04_D1_{mode}_both.nc", f"D1_{mode}"))
for branch, idata in idata_D2.items():
    KHAT_TARGETS.append((idata, f"04_D2_{branch}.nc", f"D2_{branch}"))

KHAT_STREAMS = {}
for idata, checkpoint, label in KHAT_TARGETS:
    stream = stream_loo_cached(checkpoint, idata)
    KHAT_STREAMS[label] = stream
    fig, ax = plot_khat_compat(stream, title=f"Pareto k - {label}")
    fig.savefig(FIG_DIR / f"P6_khat_{label}.png", dpi=130, bbox_inches="tight")
    plt.show()
    print(f"{label}: max k = {float(np.nanmax(stream['pareto_k'])):.3f}, "
          f"good_k = {float(stream['good_k']):.3f}")


## 6.4 - Parameter recovery, visually

`recovery_tbl` (Part 3.3) already gates PASS/FAIL on interval coverage; the plot is what makes a
systematic bias visible even when every interval technically covers its truth.


In [ ]:
def plot_recovery_compat(recovery_tbl, figsize=None):
    """Truth vs. posterior median+95% HDI for every synthetic parameter-recovery fit (Part 3.3).
    recovery_tbl already gates PASS/FAIL numerically (bc.recovery_gate); this is the picture that
    makes it legible for an appendix - and would surface a systematic bias a binary 'covered' flag
    can hide (e.g. every interval technically covering truth but all shifted the same direction).
    """
    tbl = recovery_tbl.reset_index(drop=True)
    figsize = figsize or (7, 0.5 * len(tbl) + 1.5)
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(tbl))
    colors = np.where(tbl["covered"], "steelblue", "crimson")
    ax.hlines(y, tbl["hdi_low"], tbl["hdi_high"], color=colors, lw=3, alpha=.6)
    ax.scatter(tbl["median"], y, color=colors, zorder=3, label="posterior median + 95% HDI")
    ax.scatter(tbl["truth"], y, color="k", marker="D", zorder=4, s=30, label="true value")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{m}: {p}" for m, p in zip(tbl["model"], tbl["parameter"])])
    ax.legend(fontsize=8, loc="best")
    ax.invert_yaxis()
    ax.set_title("Part 3.3 parameter recovery - SYNTHETIC, NON-REPORTABLE")
    fig.tight_layout()
    return fig, ax

fig, ax = plot_recovery_compat(recovery_tbl)
fig.savefig(FIG_DIR / "P6_recovery.png", dpi=130, bbox_inches="tight")
plt.show()


## 6.5 - Every primary estimand, one figure

A single forest-style summary of every hypothesis's headline estimand, gathered from across Part 4's
five separate sections.


In [ ]:
def plot_estimand_summary(entries, figsize=None):
    """One forest-style plot of every primary reported estimand, each on its own natural scale.
    Part 4 reports these as text plus one-off plots scattered across five sections; this puts them
    side by side as a single 'what did we find' figure."""
    figsize = figsize or (8, 0.45 * len(entries) + 1.5)
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(entries))
    labels = [e[0] for e in entries]
    med = [e[1] for e in entries]
    lo = [e[2] for e in entries]
    hi = [e[3] for e in entries]
    ax.hlines(y, lo, hi, color="steelblue", lw=3, alpha=.65)
    ax.scatter(med, y, color="steelblue", zorder=3)
    ax.axvline(0, color="k", lw=.8)
    ax.set_yticks(y); ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("posterior median and 95% CrI, natural scale per row")
    fig.tight_layout()
    return fig, ax

def _summary(idata, var, label):
    x = idata.posterior[var].values.ravel()
    lo, hi = np.quantile(x, [0.025, 0.975])
    return (label, float(np.median(x)), float(lo), float(hi))

ESTIMAND_ENTRIES = [
    _summary(idata_A, "gamma", "A: gamma (log odds ratio)"),
    _summary(idata_A, "delta_O", "A: delta_O (overlap slope)"),
    _summary(idata_A, "delta_P", "A: delta_P (log-patch slope)"),
    _summary(idata_B, "theta_lock", "B: theta_lock (log codelength ratio)"),
    _summary(idata_C, "sigma_phase", "C: sigma_phase (phase spread)"),
]
for mode in CFG.generators:
    idata = D1[mode][D1_BOTH_LABEL]
    ESTIMAND_ENTRIES.append(_summary(idata, "theta_S", f"D1:{mode} theta_S (stride dip)"))
    ESTIMAND_ENTRIES.append(_summary(idata, "theta_P", f"D1:{mode} theta_P (patch dip)"))
for branch, idata in idata_D2.items():
    param = "kappa_S" if branch == "stride" else "kappa_P"
    ESTIMAND_ENTRIES.append(_summary(idata, param, f"D2:{branch} {param}"))

fig, ax = plot_estimand_summary(ESTIMAND_ENTRIES)
ax.set_title("Part 4 - every primary estimand, one figure")
fig.savefig(FIG_DIR / "P6_all_estimands.png", dpi=130, bbox_inches="tight")
plt.show()


## 6.6 - Full per-parameter diagnostics and LOO objects

`bc.fit_diagnostics` (Parts 4-5) already reduces every fit to a pass/fail on the *worst* r-hat/ESS,
and `bc.loo_compare` reduces every LOO computation to a max-Pareto-k reliability flag - both correct
gates, but both throw away everything except one number. The full `az.summary()` table (every
parameter's mean/sd/interval/ESS/r-hat, not only the worst one) and the full LOO object
(`elpd_loo`, `se`, `p_loo`, the good_k threshold, the warning flag - 6.3 already plots the
per-observation Pareto-k this comes from, but never saved the numbers) are computed here and would
otherwise be garbage-collected the moment this cell finishes.

To be precise about what this cell buys: both tables stay cheap to recompute even after the kernel
is gone, since `az.summary`/`az.loo` need nothing but the already-checkpointed idata (posterior +
`sample_stats` + `log_likelihood`, all saved by `save_idata`/`cached_fit`). Saving them now is about
not having to reload eight checkpoints and rerun two functions just to look a number up later - not
about avoiding real recomputation cost, unlike 6.7 below.


In [ ]:

SUMMARY_FILE = "P6_param_summary.parquet"
LOO_FILE = "P6_loo_summary.parquet"

if have(SUMMARY_FILE) and have(LOO_FILE):
    param_summary = load_df(SUMMARY_FILE)
    loo_summary = load_df(LOO_FILE)
else:
    summary_rows, loo_rows = [], []
    for idata, checkpoint, label in KHAT_TARGETS:
        s = az.summary(idata).reset_index().rename(columns={"index": "parameter"})
        s.insert(0, "fit", label)
        summary_rows.append(s)

        # the streamed LOO for this fit, already checkpointed by Part 4 / the cell above
        stream = KHAT_STREAMS[label]
        k = np.asarray(stream["pareto_k"], dtype=float)
        loo_rows.append({
            "fit": label, "elpd_loo": float(stream["elpd_loo"]), "se": float(stream["se"]),
            "p_loo": float(stream["p_loo"]), "good_k": float(stream["good_k"]),
            "loo_warning": bool(stream["warning"]),
            "max_pareto_k": float(np.nanmax(k)),
            "frac_k_over_0.5": float(np.mean(k > 0.5)),
            "frac_k_over_0.7": float(np.mean(k > 0.7)),
            "n_obs": int(k.size),
        })
    param_summary = save_df(pd.concat(summary_rows, ignore_index=True), SUMMARY_FILE)
    loo_summary = save_df(pd.DataFrame(loo_rows), LOO_FILE)

display(loo_summary.round(3))
print(f"{len(param_summary)} parameter rows across {param_summary['fit'].nunique()} fits -> {SUMMARY_FILE}")


## 6.7 - Posterior-predictive draws, saved raw

Part 5's PPC gate (`ppc_table`, cell producing `05_ppc_d3_v1.parquet`) already rebuilds every model
and calls `pm.sample_posterior_predictive`, but keeps only the per-stratum coverage summary and lets
the draws themselves be garbage-collected. This is the one artifact in the whole notebook that is
*not* recoverable from a checkpoint alone: reproducing it later needs the idata (checkpointed), the
model rebuilt (cheap - the factories are pure PyTensor graph construction, not sampling), **and**
`sample_posterior_predictive` rerun (real, if modest, compute - not something a checkpoint reload
substitutes for). Right now none of that costs anything extra: `live`, `mdl_cells`, `collapse`,
`sites_f1` and every `idata_*` are already sitting in memory from Parts 2-5.

Saving the raw draws removes that dependency entirely - any future posterior-predictive density or
ECDF overlay (the one diagnostic Part 6 doesn't otherwise cover, since it needs draws Part 5 never
persisted) reads directly from these files. No model, no PyMC, no rerun, ever again. Resumable like
everything else: `have()` skips any target a previous run already saved.


In [ ]:
PPC_DRAW_SPECS = [
    ("A", lambda: model_A_contrast(arms), idata_A),
    ("B", lambda: model_B_codelength(mdl_cells, adjusted=True), idata_B),
    ("C", lambda: model_C_phase(arms, with_phase=True), idata_C),
]
for mode in CFG.generators:
    subset = collapse[collapse["mode"] == mode].reset_index(drop=True)
    PPC_DRAW_SPECS.append((f"D1_{mode}", lambda subset=subset: model_D1_sites(subset, "both"),
                           D1[mode][D1_BOTH_LABEL]))
for branch, idata in idata_D2.items():
    PPC_DRAW_SPECS.append((f"D2_{branch}",
                           lambda branch=branch: model_D2_movement(sites_f1, branch, response="f1"),
                           idata))

for label, build_model, idata in PPC_DRAW_SPECS:
    name = f"P6_ppcdraws_{label}.nc"
    if have(name):
        print(f"{label}: already saved -> {name}")
        continue
    with build_model():
        predictive = pm.sample_posterior_predictive(idata, random_seed=SEED, progressbar=False)
    save_idata(predictive, name)
    print(f"{label}: saved -> {name}")
